<a href="https://colab.research.google.com/github/jnramirezg/movilidad/blob/main/v7-20260930-explorador_matriz_OD_big_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visor de viajes — matrices de telefonía

En Google Colab, selecciona **Entorno de ejecución → Ejecutar todas**. Los cinco insumos se descargan automáticamente desde [la carpeta de Drive](https://drive.google.com/drive/folders/1qr8RvinqSzde3zjt7ujsOqrhw-t3XRJv). No se requiere subir archivos ni montar Drive.

La primera ejecución descarga aproximadamente **670 MB** y prepara las matrices. Espera a que aparezca el visor y usa **Aplicar filtros**.

- **Bogotá → localidad → UTAM/UPR → ZAT**. **Fuera de Bogotá → municipio → UTAM/UPR, cuando exista → ZAT**. Las asignaciones pendientes quedan en **Revisar asignación territorial**.
- Las letras **O** y **D** aparecen con **1 a 4 selecciones territoriales** por tipo; desde 5 se ocultan de forma independiente. Un municipio, localidad o UTAM completo cuenta como una selección. La delimitación gris de localidades se conserva.
- El selector **Fondo del mapa** permite usar **Sin fondo** si OpenStreetMap muestra un bloqueo; las capas y los cálculos permanecen disponibles.

Los archivos válidos se reutilizan durante la misma sesión. Para actualizar sus versiones, activa **ACTUALIZAR_INSUMOS** en la celda 3. Mantén los cinco archivos accesibles mediante enlace y conserva sus ID al actualizarlos en Drive. Si falta un insumo o falla una descarga, la celda muestra el problema y no abre un visor incompleto.


In [ ]:
#@title 1. Preparar el entorno
import sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError
paquetes = [
    ('requests','requests','requests>=2.32,<3',(2,32)),
    ('duckdb','duckdb','duckdb>=1.2,<2',(1,2)),
    ('shapefile','pyshp','pyshp>=2.3,<4',(2,3)),
    ('geopandas','geopandas','geopandas>=1.0,<2',(1,0)),
    ('shapely','shapely','shapely>=2.0,<3',(2,0)),
    ('pyproj','pyproj','pyproj>=3.6,<4',(3,6)),
    ('pyogrio','pyogrio','pyogrio>=0.9,<1',(0,9)),
    ('gdown','gdown','gdown==6.4.1',(6,4,1)),
]
faltan=[]
for modulo, distribucion, requisito, minima in paquetes:
    try:
        actual=tuple(int(p) for p in version(distribucion).split('.')[:len(minima)])
        if actual < minima or (distribucion == 'gdown' and actual != minima) or importlib.util.find_spec(modulo) is None:
            faltan.append(requisito)
    except (PackageNotFoundError, ValueError): faltan.append(requisito)
if faltan:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*faltan])
    if 'gdown==6.4.1' in faltan:
        # La celda puede repetirse después de haber importado otra versión.
        for nombre in [n for n in sys.modules if n == 'gdown' or n.startswith('gdown.')]:
            del sys.modules[nombre]
from IPython.display import display, HTML, JSON, clear_output
try:
    from google.colab import output, files
    EN_COLAB=True
except ImportError:
    EN_COLAB=False


In [ ]:
#@title 2. Preparar cálculos y mapas
from pathlib import Path, PurePosixPath
from urllib.parse import urljoin, unquote, urlparse
from html.parser import HTMLParser
from datetime import datetime
import csv, gzip, json, math, os, re, shutil, threading, time, zipfile
import duckdb
import requests

ANIOS = [2019, 2023, 2025]
FUENTE = 'https://observatorio.movilidadbogota.gov.co/publicaciones/informes/analisis-de-la-movilidad-con'
URLS = {
    2019: 'https://observatorio.movilidadbogota.gov.co/sites/default/files/2026-04/1_matrices_2019-20260421T214315Z-3-001_0.zip',
    2023: 'https://observatorio.movilidadbogota.gov.co/sites/default/files/2026-04/2_matrices_2023-20260421T214317Z-3-001.zip',
    2025: 'https://observatorio.movilidadbogota.gov.co/sites/default/files/2026-04/3_matrices_2025-20260421T214320Z-3-001.zip',
}
PERIODOS = {
    'laboral': {2019: 'laboral', 2023: '20230516', 2025: '20250513'},
    'sabado': {2019: 'sabado', 2023: '20230520', 2025: '20250517'},
    'domingo': {2019: 'domingo', 2023: '20230521', 2025: '20250525'},
}
CAMPOS = ['hora_inicio', 'origen', 'destino', 'proposito_origen',
          'proposito_destino', 'distancia', 'modo', 'residencia', 'nse',
          'genero', 'profesional', 'viajes']
NULO = '__SIN_DATO__'
NO_ZAT = '__SIN_ZAT__'
MODOS = {
    'no_motorizado': ['no_motorizado_caminata', 'no_motorizado_bicicleta'],
    'transporte_privado': ['transporte_privado_coche', 'transporte_privado_moto'],
    'transporte_publico': ['transporte_publico'],
}
NOMBRES_MODO = {
    'no_motorizado': 'No motorizado', 'transporte_privado': 'Transporte privado',
    'transporte_publico': 'Transporte público', 'no_motorizado_caminata': 'Caminata',
    'no_motorizado_bicicleta': 'Bicicleta', 'transporte_privado_coche': 'Coche',
    'transporte_privado_moto': 'Moto', NULO: 'Sin dato',
}
DISTANCIAS = ['0 a <500 m', '500 a <1,000 m', '1 a <2 km', '2 a <5 km',
              '5 a <10 km', '10 a <20 km', '20 a <50 km', '50 km o más']
MOTIVOS = {'H': 'Casa (H)', 'NF': 'Otras actividades no frecuentes (NF)',
           'O': 'Otras actividades frecuentes (O)', 'W': 'Trabajo / estudio (W)'}
POR_DEFECTO = {'dia': 'laboral', 'hora': [0, 23], 'modo': None,
               'motivo': None, 'residencia': None, 'origen': None, 'destino': None}

def orden_codigo(v):
    return (0, int(v), v) if str(v).isdigit() else (1, 0, str(v))

def literal_sql(v):
    return "'" + str(v).replace("'", "''") + "'"

class Enlaces(HTMLParser):
    def __init__(self):
        super().__init__(); self.urls = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            href = dict(attrs).get('href', '')
            if unquote(urlparse(href).path).lower().endswith('.zip'):
                self.urls.append(urljoin(FUENTE, href))

class VisorTelefonia:
    # La clase extendida incorpora las asignaciones de la zonificación final.
    # Se excluyen los registros identificados como conductores profesionales.
    def __init__(self, carpeta, rutas_locales=None, avisar=None):
        self.carpeta = Path(carpeta); self.carpeta.mkdir(parents=True, exist_ok=True)
        self.rutas_locales = dict(rutas_locales or {})
        self.avisar = avisar or (lambda texto: None)
        self.sesion = requests.Session()
        self.sesion.headers.update({'User-Agent': 'Mozilla/5.0', 'Accept-Encoding': 'identity'})
        self.urls = dict(URLS); self.enlaces_revisados = False
        self.db = duckdb.connect(str(self.carpeta / 'visor_v1.duckdb'))
        self.db.execute("SET memory_limit='2GB'")
        self.db.execute('SET threads=2')
        self.db.execute('SET preserve_insertion_order=false')
        self.db.execute('SET temp_directory=' + literal_sql(self.carpeta / 'temporal'))
        self.db.execute('CREATE TABLE IF NOT EXISTS catalogo (tabla VARCHAR PRIMARY KEY, firma VARCHAR)')
        self.estados = {}; self.dominios = {}; self.zonas_archivo = {}
        self.lock = threading.RLock()

    def registrar_zip(self, ruta):
        ruta = Path(ruta)
        if not zipfile.is_zipfile(ruta):
            raise ValueError('El archivo no es un ZIP válido.')
        hallados = set()
        with zipfile.ZipFile(ruta) as z:
            for n in z.namelist():
                if not n.lower().endswith(('matriz_od.csv.gz', 'matriz_od.csv')):
                    continue
                for anio in ANIOS:
                    if str(anio) in n or str(anio) in ruta.name:
                        hallados.add(anio)
        if not hallados:
            raise ValueError('No se pudo identificar el año del ZIP. Conserve el nombre original.')
        for anio in hallados:
            self.rutas_locales[anio] = str(ruta)
            self.estados = {k: v for k, v in self.estados.items() if k[0] != anio}
            self.dominios = {k: v for k, v in self.dominios.items() if k[0] != anio}
            self.zonas_archivo.pop(anio, None)
        return sorted(hallados)

    def descargar(self, anio):
        if anio in self.rutas_locales:
            p = Path(self.rutas_locales[anio])
            if not zipfile.is_zipfile(p):
                raise ValueError(f'ZIP local de {anio} no válido.')
            return p
        p = self.carpeta / f'matrices_{anio}.zip'
        if p.exists() and zipfile.is_zipfile(p):
            return p
        if not self.enlaces_revisados:
            self.enlaces_revisados = True
            try:
                r = self.sesion.get(FUENTE, timeout=(10, 25)); r.raise_for_status()
                parser = Enlaces(); parser.feed(r.text)
                for a in ANIOS:
                    candidatos = {u for u in parser.urls if 'matrices' in u and str(a) in u}
                    if len(candidatos) == 1:
                        self.urls[a] = candidatos.pop()
            except requests.RequestException:
                pass  # Los enlaces oficiales registrados siguen disponibles.
        parcial = p.with_suffix('.zip.part')
        self.avisar(f'Descargando {anio}…')
        try:
            with self.sesion.get(self.urls[anio], stream=True, timeout=(20, 120)) as r:
                r.raise_for_status()
                total = int(r.headers.get('Content-Length', 0))
                if total and shutil.disk_usage(self.carpeta).free < total + 512*1024**2:
                    raise OSError('No hay espacio para descargar el ZIP.')
                recibidos = 0; ultimo = time.monotonic()
                with parcial.open('wb') as f:
                    for b in r.iter_content(2**20):
                        if b:
                            f.write(b); recibidos += len(b)
                        if time.monotonic() - ultimo > 2:
                            self.avisar(f'Descargando {anio}: {recibidos/2**20:.0f} MB…')
                            ultimo = time.monotonic()
            if total and total != recibidos:
                raise IOError('Descarga incompleta; vuelva a intentarlo.')
            if not zipfile.is_zipfile(parcial):
                raise ValueError('El servidor no devolvió un ZIP válido.')
            parcial.replace(p)
            return p
        finally:
            parcial.unlink(missing_ok=True)

    def miembro(self, ruta, anio, dia):
        clave = PERIODOS[dia][anio]
        candidatos = []
        with zipfile.ZipFile(ruta) as z:
            for info in z.infolist():
                partes = PurePosixPath(info.filename.replace('\\', '/')).parts
                bajos = [s.lower() for s in partes]
                if '..' in partes or '__macosx' in bajos or len(partes) < 3:
                    continue
                if bajos[-1] not in {'matriz_od.csv.gz', 'matriz_od.csv'}:
                    continue
                if bajos[-2] != clave or bajos[-3] != 'matrices':
                    continue
                if any(str(a) in '/'.join(partes[:-2]) for a in ANIOS if a != anio):
                    continue
                candidatos.append(info)
        candidatos = [i for i in candidatos if i.filename.endswith('.gz')] or candidatos
        if len(candidatos) != 1:
            raise ValueError(f'No se identificó una matriz única para {anio}, {clave}.')
        return candidatos[0]

    def leer_ids_zonificacion(self, ruta, anio):
        # Solo se lee el campo ID del DBF de la entrega; NO geometrías ni joins.
        if anio in self.zonas_archivo:
            return self.zonas_archivo[anio]
        ids = set()
        try:
            import io, shapefile
            with zipfile.ZipFile(ruta) as z:
                nombres = [n for n in z.namelist() if n.lower().endswith('/zonificacion.dbf')
                           and not any(str(a) in n for a in ANIOS if a != anio)]
                if len(nombres) == 1:
                    r = shapefile.Reader(dbf=io.BytesIO(z.read(nombres[0])), encoding='utf-8', encodingErrors='replace')
                    nombres_campos = [f[0] for f in r.fields[1:]]
                    if 'ID' in nombres_campos:
                        i = nombres_campos.index('ID')
                        for fila in r.iterRecords():
                            valor = fila[i]
                            if isinstance(valor, float) and valor.is_integer(): valor = int(valor)
                            if valor is not None and str(valor).strip(): ids.add(str(valor).strip())
        except (ImportError, ValueError, OSError, zipfile.BadZipFile):
            pass
        self.zonas_archivo[anio] = ids
        return ids

    def importar(self, ruta, tabla, firma):
        # DuckDB puede leer un GZIP sin detectar un tráiler incompleto.
        # gzip valida EOF, tamaño y CRC antes de aceptar cualquier suma.
        if str(ruta).endswith('.gz'):
            with gzip.open(ruta, 'rb') as f:
                while f.read(4*1024**2):
                    pass
        tipos = {c: 'VARCHAR' for c in CAMPOS}; tipos['viajes'] = 'DOUBLE'
        cab = gzip.open if str(ruta).endswith('.gz') else open
        with cab(ruta, 'rt', encoding='utf-8-sig', newline='') as f:
            campos = next(csv.reader(f, delimiter='|'), [])
        if campos != CAMPOS:
            raise ValueError('La estructura de columnas no coincide con las matrices publicadas.')
        tipos_sql = '{' + ', '.join(literal_sql(c)+':'+literal_sql(t) for c,t in tipos.items()) + '}'
        clean = lambda c: f"coalesce(nullif(trim({c}), ''), '{NULO}')"
        campos_sql = ', '.join(f'{clean(c)} AS {c}' for c in ['origen','destino','residencia'])
        # La hora se valida con patrón completo: P00 a P23.
        hora = "CASE WHEN regexp_full_match(trim(hora_inicio), 'P([01][0-9]|2[0-3])') THEN substr(trim(hora_inicio),2)::INTEGER ELSE NULL END"
        sql = f'''CREATE TABLE {tabla}_nueva AS SELECT {hora} AS hora,
              {campos_sql}, {clean('proposito_destino')} AS motivo,
              {clean('modo')} AS modo, {clean('distancia')} AS distancia,
              CASE WHEN regexp_full_match(trim(nse), '(EE_|E_|E)[1-6]')
                   THEN 'E' || right(trim(nse),1) ELSE {clean('nse')} END AS nse,
              lower({clean('profesional')}) AS profesional, viajes
              FROM read_csv({literal_sql(ruta)}, delim='|', header=true,
                   columns={tipos_sql}, auto_detect=false, strict_mode=true,
                   ignore_errors=false, parallel=false, null_padding=false)'''
        self.db.execute('BEGIN')
        try:
            self.db.execute(f'DROP TABLE IF EXISTS {tabla}_nueva')
            self.db.execute(sql)
            n, invalidos = self.db.execute(f'''SELECT count(*), count(*) FILTER(
                 WHERE hora IS NULL OR viajes IS NULL OR NOT isfinite(viajes) OR viajes<0)
                 FROM {tabla}_nueva''').fetchone()
            if not n or invalidos:
                raise ValueError('Matriz vacía o con horas/pesos inválidos; no se mostrarán sumas parciales.')
            desconocidos = self.db.execute(f"SELECT count(*) FROM {tabla}_nueva WHERE profesional NOT IN ('falso','verdadero','false','true')").fetchone()[0]
            if desconocidos:
                raise ValueError('Hay valores no reconocidos en profesional; no se mostrarán sumas parciales.')
            # Validar la fuente completa antes de excluir profesionales.
            # Una matriz válida que solo contiene profesionales queda en cero.
            self.db.execute(f"DELETE FROM {tabla}_nueva WHERE profesional IN ('verdadero','true')")
            self.db.execute(f'ALTER TABLE {tabla}_nueva DROP COLUMN profesional')
            self.db.execute(f'DROP TABLE IF EXISTS {tabla}')
            self.db.execute(f'ALTER TABLE {tabla}_nueva RENAME TO {tabla}')
            self.db.execute('INSERT OR REPLACE INTO catalogo VALUES (?,?)', [tabla, firma])
            self.db.execute('COMMIT')
        except Exception:
            self.db.execute('ROLLBACK'); raise

    def preparar(self, anio, dia):
        key = (anio, dia)
        if key in self.estados:
            return self.estados[key]
        ruta = self.descargar(anio); info = self.miembro(ruta, anio, dia)
        tabla = f'v_{anio}_{dia}'
        firma = f'v2_sin_profesionales:{info.filename}:{info.CRC}:{info.file_size}'
        encontrada = self.db.execute('SELECT firma FROM catalogo WHERE tabla=?', [tabla]).fetchone()
        if encontrada != (firma,):
            archivo = self.carpeta / (tabla + ('.csv.gz' if info.filename.endswith('.gz') else '.csv'))
            self.avisar(f'Preparando {anio}…')
            try:
                with zipfile.ZipFile(ruta) as z, z.open(info.filename) as entrada, archivo.open('wb') as salida:
                    shutil.copyfileobj(entrada, salida, 2**20)
                self.importar(archivo, tabla, firma)
            finally:
                archivo.unlink(missing_ok=True)
        self.estados[key] = tabla
        dom = {}
        for campo in ['origen','destino','residencia','modo','motivo','distancia','nse']:
            dom[campo] = {r[0] for r in self.db.execute(f'SELECT DISTINCT {campo} FROM {tabla}').fetchall()}
        ids = self.leer_ids_zonificacion(ruta, anio)
        # Cuando no se puede leer ID, los códigos observados en OD son la referencia
        # disponible; los demás códigos de residencia NO se etiquetan como ZAT.
        dom['ids_zona'] = ids or ((dom['origen'] | dom['destino']) - {NULO})
        dom['hogar_no_zat'] = dom['residencia'] - dom['ids_zona'] - {NULO}
        self.dominios[key] = dom
        return tabla

    def validar_filtros(self, solicitud):
        f = dict(POR_DEFECTO); f.update(solicitud or {})
        if f['dia'] not in PERIODOS: raise ValueError('Día no válido.')
        h = f['hora']
        if not isinstance(h, (list,tuple)) or len(h)!=2 or any(type(x)!=int for x in h) or not 0<=h[0]<=h[1]<=23:
            raise ValueError('La hora debe estar entre 0 y 23, en orden creciente.')
        for c in ['modo','motivo','residencia','origen','destino']:
            if f[c] is not None and (not isinstance(f[c],list) or any(not isinstance(x,str) for x in f[c])):
                raise ValueError('Selección no válida.')
            if f[c] is not None: f[c] = sorted(set(f[c]))
        return {k:f[k] for k in POR_DEFECTO}

    def opciones(self, dia):
        doms = [self.dominios[(a,dia)] for a in ANIOS if (a,dia) in self.dominios]
        union = lambda campo: set().union(*(d[campo] for d in doms)) if doms else set()
        grupos = []
        vistos = union('modo')
        for grupo, hijos in MODOS.items():
            if grupo in vistos or vistos.intersection(hijos):
                grupos.append({'label':NOMBRES_MODO[grupo], 'children':[
                    {'value':c,'label':NOMBRES_MODO[c]} for c in hijos]})
        otros = vistos - set(MODOS) - {c for hs in MODOS.values() for c in hs}
        if otros: grupos.append({'label':'Otros','children':[{'value':v,'label':NOMBRES_MODO.get(v,v)} for v in sorted(otros)]})
        hogares = set().union(*(d['residencia'] & d['ids_zona'] for d in doms)) if doms else set()
        if NULO in union('residencia'): hogares.add(NULO)
        if any(d['hogar_no_zat'] for d in doms): hogares.add(NO_ZAT)
        etiqueta_zona = lambda v: 'Sin dato' if v==NULO else ('Sin ZAT identificada' if v==NO_ZAT else v)
        return {
            'modo':grupos,
            'motivo':[{'value':v,'label':MOTIVOS.get(v,'Sin dato' if v==NULO else v)} for v in list(MOTIVOS)+sorted(union('motivo')-set(MOTIVOS))],
            **{campo:[{'value':v,'label':etiqueta_zona(v)} for v in sorted(valores,key=orden_codigo)]
               for campo,valores in [('residencia',hogares),('origen',union('origen')),('destino',union('destino'))]},
        }

    def condicion(self, anio, f):
        dom = self.dominios[(anio,f['dia'])]
        clausulas = ['hora BETWEEN ? AND ?']; params = list(f['hora'])
        for campo in ['modo','motivo','residencia','origen','destino']:
            valores = f[campo]
            if valores is None: continue
            seleccion = set(valores)
            if campo=='modo':
                for grupo,hijos in MODOS.items():
                    if grupo in dom['modo'] and grupo not in hijos:
                        n = len(seleccion.intersection(hijos))
                        if 0<n<len(hijos):
                            return None, [], f'{anio}: sin desagregación para el modo seleccionado.'
                        if n==len(hijos): seleccion.add(grupo)
            if campo=='residencia' and NO_ZAT in seleccion:
                seleccion.remove(NO_ZAT); seleccion.update(dom['hogar_no_zat'])
            if not seleccion:
                clausulas.append('FALSE')
            else:
                clausulas.append(campo+' IN ('+','.join('?' for _ in seleccion)+')')
                params.extend(sorted(seleccion))
        return ' AND '.join(clausulas), params, None

    def consultar(self, solicitud=None):
        with self.lock:
            f = self.validar_filtros(solicitud)
            series = []; errores = []
            for anio in ANIOS:
                try:
                    tabla = self.preparar(anio,f['dia'])
                    condicion, params, aviso = self.condicion(anio,f)
                    if aviso:
                        series.append({'year':anio,'available':False,'message':aviso}); continue
                    modo_sql = "CASE WHEN modo LIKE 'no_motorizado%' THEN 'no_motorizado' WHEN modo LIKE 'transporte_privado%' THEN 'transporte_privado' ELSE modo END"
                    sql = f'''WITH base AS (SELECT hora, distancia, {modo_sql} AS grupo,
                           motivo, nse, viajes FROM {tabla} WHERE {condicion})
                           SELECT grouping_id(hora,distancia,grupo,motivo,nse) AS tipo,
                           hora,distancia,grupo,motivo,nse,coalesce(sum(viajes),0) AS total
                           FROM base GROUP BY GROUPING SETS ((),(hora),(distancia),(grupo),(motivo),(nse))'''
                    filas = self.db.execute(sql,params).fetchall()
                    resultado = {'year':anio,'available':True,'total':0.0,
                                 'hora':{str(h):0.0 for h in range(24)}, 'distancia':{},
                                 'modo':{},'motivo':{},'nse':{}}
                    indices = {15:('hora',1),23:('distancia',2),27:('modo',3),29:('motivo',4),30:('nse',5)}
                    for fila in filas:
                        tipo, total = fila[0], float(fila[-1])
                        if not math.isfinite(total): raise ValueError('La suma de viajes no es finita.')
                        if tipo==31: resultado['total']=total
                        else:
                            campo,i = indices[tipo]
                            valor = str(fila[i])
                            if campo=='distancia' and re.match(r'^D0[0-7]_',valor): valor=valor[:3]
                            resultado[campo][valor]=resultado[campo].get(valor,0.0)+total
                    series.append(resultado)
                except Exception as e:
                    if isinstance(e,requests.HTTPError):
                        msg=f'{anio}: descarga HTTP {e.response.status_code}. Vuelva a ejecutar la celda 3 para cargar los insumos de Drive.'
                    else:
                        msg=f'{anio}: {str(e)[:240]}'
                    series.append({'year':anio,'available':False,'message':msg}); errores.append(msg)
            return {'filters':f,'series':series,'options':self.opciones(f['dia']),
                    'errors':errores,'periods':{str(a):PERIODOS[f['dia']][a] for a in ANIOS},
                    'distanceLabels':dict(zip([f'D{i:02}' for i in range(8)],DISTANCIAS)),
                    'modeLabels':NOMBRES_MODO,'purposeLabels':MOTIVOS}


from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import json
import math
import re
import shutil
import stat
import unicodedata

import geopandas as gpd
import pandas as pd
from pyproj import Transformer
from shapely import make_valid
from shapely.geometry import MultiPolygon, Polygon, mapping
from shapely.ops import transform as transformar_geometria, unary_union


class GeografiaVisor:
    """Lee las cuatro capas y prepara geometrías de presentación sin repetir el join.

    ``rutas`` contiene ``zonificacion``, ``localidades``, ``municipios`` y ``final``;
    cada valor es la ruta de un ZIP con un shapefile o de un SHP con sus compañeros.
    ``zonas`` conserva una fila por ID de la capa final. Los códigos de ID, ZAT,
    municipio y localidad son cadenas; los valores ausentes son cadenas vacías.
    La simplificación y reparación geométrica afectan solo a ``geodata``.
    """

    def __init__(self, rutas):
        self.diagnostico = []
        requeridas = ("zonificacion", "localidades", "municipios", "final")
        faltantes = [k for k in requeridas if not rutas.get(k)]
        if faltantes:
            raise ValueError("Faltan las capas: " + ", ".join(faltantes))
        capas = {k: self._leer_capa(rutas[k], k) for k in requeridas}
        self._preparar_zonas(capas["final"])
        self._comprobar_original(capas["zonificacion"])
        zonas_metricas = self._geometrias_metricas(capas["final"], "zonificación final")
        self._crear_geodata(zonas_metricas, capas["localidades"], capas["municipios"])

    @staticmethod
    def _texto(valor):
        if valor is None or pd.isna(valor):
            return ""
        return str(valor).strip()

    @classmethod
    def _codigo(cls, valor, ancho=None):
        texto = cls._texto(valor)
        if re.fullmatch(r"[+-]?\d+\.0+", texto):
            texto = texto.split(".")[0]
        if ancho and texto.isdigit():
            texto = texto.zfill(ancho)
        return texto

    @staticmethod
    def _normalizar(texto):
        texto = unicodedata.normalize("NFKD", str(texto))
        return "".join(c for c in texto.upper() if c.isalnum() and not unicodedata.combining(c))

    @classmethod
    def _campo(cls, capa, opciones, nombre, obligatorio=True):
        campos = {cls._normalizar(c): c for c in capa.columns}
        for candidato in opciones:
            if cls._normalizar(candidato) in campos:
                return campos[cls._normalizar(candidato)]
        if obligatorio:
            raise ValueError("No se encontró " + nombre + ". Campos disponibles: " + ", ".join(capa.columns))
        return None

    def _leer_capa(self, ruta, nombre):
        ruta = Path(ruta).expanduser().resolve()
        if not ruta.is_file():
            raise FileNotFoundError("No se encontró la capa " + nombre + ": " + str(ruta))
        with TemporaryDirectory(prefix="visor_capa_") as temporal:
            temporal = Path(temporal)
            if ruta.suffix.lower() == ".zip":
                with ZipFile(ruta) as archivo:
                    entradas = archivo.infolist()
                    if sum(x.file_size for x in entradas) > 2_000_000_000:
                        raise ValueError("El ZIP de " + nombre + " supera el tamaño permitido al descomprimir.")
                    for entrada in entradas:
                        destino = (temporal / entrada.filename.replace("\\", "/")).resolve()
                        if not destino.is_relative_to(temporal) or stat.S_ISLNK(entrada.external_attr >> 16):
                            raise ValueError("El ZIP de " + nombre + " contiene una ruta no permitida.")
                        if entrada.is_dir():
                            destino.mkdir(parents=True, exist_ok=True)
                            continue
                        destino.parent.mkdir(parents=True, exist_ok=True)
                        with archivo.open(entrada) as origen, destino.open("wb") as salida:
                            shutil.copyfileobj(origen, salida)
                shapes = [p for p in temporal.rglob("*") if p.suffix.lower() == ".shp" and "__MACOSX" not in p.parts]
                if len(shapes) != 1:
                    raise ValueError("El ZIP de " + nombre + " debe contener exactamente un shapefile; contiene " + str(len(shapes)) + ".")
                shp = shapes[0]
            elif ruta.suffix.lower() == ".shp":
                shp = ruta
            else:
                raise ValueError("La capa " + nombre + " debe ser ZIP o SHP.")
            compañeros = {p.suffix.lower(): p for p in shp.parent.iterdir() if p.stem.lower() == shp.stem.lower()}
            ausentes = [ext for ext in (".shp", ".shx", ".dbf", ".prj") if ext not in compañeros]
            if ausentes:
                raise ValueError("En " + nombre + " faltan archivos compañeros: " + ", ".join(ausentes))
            capa = gpd.read_file(shp).reset_index(drop=True)
        if capa.empty:
            raise ValueError("La capa " + nombre + " está vacía.")
        if capa.crs is None:
            raise ValueError("La capa " + nombre + " no tiene un sistema de coordenadas reconocido.")
        if not capa.geometry.geom_type.dropna().isin(["Polygon", "MultiPolygon"]).all():
            raise ValueError("La capa " + nombre + " debe contener polígonos.")
        return capa

    def _preparar_zonas(self, capa):
        campos = {
            "id": (["ID"], True),
            "zat": (["ZAT"], True),
            "utam": (["UTAM", "UTAM_UPR", "UTAMUPR"], True),
            "mun_code": (["MUN_COD"], True),
            "mun_name": (["MUN_FIN"], True),
            "loc_code": (["LOC_COD"], True),
            "loc_name": (["LOC_NOM"], True),
            "rev_loc": (["REV_LOC"], False),
            "rev_mun": (["REV_MUN"], False),
        }
        datos = {}
        for salida, (opciones, obligatorio) in campos.items():
            campo = self._campo(capa, opciones, salida + " en zonificación final", obligatorio)
            datos[salida] = capa[campo].map(self._texto) if campo else pd.Series("", index=capa.index)
        self.zonas = pd.DataFrame(datos)
        for campo, ancho in (("id", None), ("zat", None), ("mun_code", 5), ("loc_code", 2)):
            self.zonas[campo] = self.zonas[campo].map(lambda v: self._codigo(v, ancho))
        if self.zonas.id.eq("").any() or self.zonas.id.duplicated().any():
            raise ValueError("El campo ID de la zonificación final debe ser único y estar completo.")
        self.zonas["label"] = self.zonas.apply(
            lambda r: ("Zona ID " + r.id + " (sin ZAT)") if not r.zat
            else ("ZAT " + r.zat if r.zat == r.id else "ZAT " + r.zat + " · ID " + r.id), axis=1)
        self.zonas = self.zonas[["id", "label", "zat", "utam", "mun_code", "mun_name", "loc_code", "loc_name", "rev_loc", "rev_mun"]]
        self.tabla_relacion = capa.drop(columns=capa.geometry.name).copy()
        self.tabla_relacion["id_visor"] = self.zonas.id
        sin_zat = int(self.zonas.zat.eq("").sum())
        distintos = int((self.zonas.zat.ne("") & self.zonas.zat.ne(self.zonas.id)).sum())
        if sin_zat or distintos:
            self.diagnostico.append(f"La capa final contiene {sin_zat} zonas sin ZAT y {distintos} con ID diferente de ZAT. La correspondencia con las matrices debe comprobarse por separado.")
        multi = self.zonas.mun_name.map(self._normalizar).eq("MULTIMUNICIPAL")
        if multi.any():
            self.diagnostico.append(f"Se conservan {int(multi.sum())} zonas multimunicipales sin asignarlas a un municipio predominante.")
        pendiente = self.zonas.rev_loc.eq("REVISAR")
        if pendiente.any():
            self.diagnostico.append(f"Se conservan las asignaciones de localidad del archivo final; {int(pendiente.sum())} zonas tienen REV_LOC=REVISAR.")

    def _comprobar_original(self, original):
        campo = self._campo(original, ["ID"], "ID en zonificación original")
        ids = original[campo].map(self._codigo)
        if ids.eq("").any() or ids.duplicated().any():
            raise ValueError("El campo ID de la zonificación original debe ser único y estar completo.")
        orig, final = set(ids), set(self.zonas.id)
        if orig != final:
            self.diagnostico.append(f"Las zonificaciones original y final difieren: {len(orig-final)} ID solo en la original y {len(final-orig)} ID solo en la final. Se utilizan las filas y asignaciones de la capa final.")
        else:
            self.diagnostico.append(f"Los {len(final)} ID de la zonificación final coinciden con los de la original.")
        originales = original.assign(_id=ids).set_index("_id")
        finales = self.zonas.set_index("id")
        comunes = sorted(orig & final)
        c_mun = self._campo(original, ["MUNCod", "MUN_COD"], "municipio original", False)
        if c_mun and comunes:
            antes = originales.loc[comunes, c_mun].map(lambda v: self._codigo(v, 5))
            despues = finales.loc[comunes, "mun_code"]
            cambios = antes.ne("") & despues.ne("") & antes.ne(despues)
            if cambios.any():
                ejemplo = ", ".join(cambios[cambios].index[:8])
                self.diagnostico.append(f"Hay {int(cambios.sum())} ID cuyo municipio definitivo difiere del original (ejemplos: {ejemplo}). Se conserva la asignación final.")

    @staticmethod
    def _poligonos(geometria):
        if geometria is None or geometria.is_empty:
            return None
        if isinstance(geometria, (Polygon, MultiPolygon)):
            return geometria
        piezas = []
        for parte in getattr(geometria, "geoms", []):
            poligono = GeografiaVisor._poligonos(parte)
            if poligono is not None:
                piezas.append(poligono)
        return unary_union(piezas) if piezas else None

    def _geometrias_metricas(self, capa, nombre):
        reproyeccion = Transformer.from_crs(capa.crs, "EPSG:9377", always_xy=True, allow_ballpark=False, only_best=True)
        reparadas = 0
        vacias = 0
        geometrias = []
        for geometria in capa.geometry:
            if geometria is not None and not geometria.is_empty:
                if not geometria.is_valid:
                    reparadas += 1
                    geometria = self._poligonos(make_valid(geometria))
                if geometria is not None:
                    geometria = transformar_geometria(reproyeccion.transform, geometria)
                    if not all(math.isfinite(v) for v in geometria.bounds):
                        raise ValueError("No se pudo transformar el CRS de " + nombre + ".")
                    if not geometria.is_valid:
                        geometria = self._poligonos(make_valid(geometria))
            if geometria is None or geometria.is_empty:
                vacias += 1
                geometria = None
            geometrias.append(geometria)
        if reparadas:
            self.diagnostico.append(f"En {nombre} se repararon {reparadas} geometrías solo para dibujar; las asignaciones se conservan.")
        if vacias:
            self.diagnostico.append(f"En {nombre} hay {vacias} geometrías vacías o no representables; sus registros no se eliminan.")
        return gpd.GeoDataFrame(capa.drop(columns=capa.geometry.name).copy(), geometry=geometrias, crs="EPSG:9377")

    @staticmethod
    def _union(geometrias):
        partes = [g for g in geometrias if g is not None and not g.is_empty]
        return unary_union(partes) if partes else None

    @staticmethod
    def _coordenadas_cortas(coordenadas):
        if coordenadas and isinstance(coordenadas[0], (int, float)):
            return [round(float(v), 6) for v in coordenadas]
        return [GeografiaVisor._coordenadas_cortas(c) for c in coordenadas]

    def _feature(self, geometria, propiedades, tolerancia=5):
        if geometria is None or geometria.is_empty:
            return {"type": "Feature", "properties": dict(propiedades, center=None), "geometry": None}
        simplificada = geometria.simplify(tolerancia, preserve_topology=True)
        punto = simplificada.representative_point()
        centro = self._a_wgs84.transform(punto.x, punto.y)
        geometria_wgs84 = transformar_geometria(self._a_wgs84.transform, simplificada)
        geometria_json = mapping(geometria_wgs84)
        geometria_json["coordinates"] = self._coordenadas_cortas(geometria_json["coordinates"])
        return {"type": "Feature", "properties": dict(propiedades, center=[round(centro[1], 6), round(centro[0], 6)]), "geometry": geometria_json}

    @staticmethod
    def _coleccion(features):
        return {"type": "FeatureCollection", "features": features}

    def _crear_geodata(self, zonas_metricas, localidades, municipios):
        self._a_wgs84 = Transformer.from_crs("EPSG:9377", "EPSG:4326", always_xy=True, allow_ballpark=False, only_best=True)
        zonas = self.zonas
        features_zonas = []
        for i, fila in zonas.iterrows():
            props = {"key": fila.id, "label": fila.label, "zone_ids": [fila.id], "loc_code": fila.loc_code, "mun_code": fila.mun_code, "utam": fila.utam}
            features_zonas.append(self._feature(zonas_metricas.geometry.iloc[i], props))

        loc_code = self._campo(localidades, ["LOCCODIGO", "LOC_COD", "COD_LOC", "CODIGO_LOC", "CODIGO", "COD_LOCALI"], "código de localidad")
        loc_name = self._campo(localidades, ["LOCNOMBRE", "LOC_NOM", "NOM_LOC", "NOMBRE", "LOCALIDAD", "NOMBRE_LOC"], "nombre de localidad")
        localidades = localidades.copy()
        localidades["_key"] = localidades[loc_code].map(lambda v: self._codigo(v, 2))
        if localidades._key.eq("").any():
            raise ValueError("Hay localidades sin código en la capa de referencia.")
        localidades = self._geometrias_metricas(localidades, "localidades")
        features_loc = []
        for codigo, grupo in localidades.groupby("_key", sort=True):
            ids = zonas.loc[(zonas.mun_code == "11001") & (zonas.loc_code == codigo), "id"].tolist()
            features_loc.append(self._feature(self._union(grupo.geometry), {"key": codigo, "label": self._texto(grupo.iloc[0][loc_name]), "zone_ids": ids}))
        loc_presentes = set(localidades._key)
        loc_faltantes = set(zonas.loc[zonas.mun_code.eq("11001") & zonas.loc_code.ne(""), "loc_code"]) - loc_presentes
        if loc_faltantes:
            self.diagnostico.append("No hay geometría de referencia para estas localidades de la capa final: " + ", ".join(sorted(loc_faltantes)) + ".")

        mun_code = self._campo(municipios, ["MpCodigo", "MUN_COD", "COD_MPIO", "MPIO_CCDGO", "DPTO_MPIO", "COD_DANE"], "código de municipio")
        mun_name = self._campo(municipios, ["MpNombre", "MUN_FIN", "MPIO_CNMBR", "NOM_MPIO", "MUNICIPIO", "NOMBRE"], "nombre de municipio")
        categoria = self._campo(municipios, ["MpCategor"], "categoría territorial", False)
        municipios = municipios.copy()
        if categoria:
            categorias = pd.to_numeric(municipios[categoria], errors="coerce")
            municipios = municipios.loc[categorias.isin([1, 2])].copy()
        municipios["_key"] = municipios[mun_code].map(lambda v: self._codigo(v, 5))
        validos = municipios._key.str.fullmatch(r"\d{5}") & municipios._key.ne("00000")
        municipios = municipios.loc[validos].copy()
        presentes = set(zonas.loc[zonas.mun_code.ne(""), "mun_code"])
        municipios = municipios.loc[municipios._key.str.startswith("25") | municipios._key.eq("11001") | municipios._key.isin(presentes)].copy()
        if municipios.empty:
            raise ValueError("La capa de municipios no contiene los municipios requeridos por la zonificación.")
        municipios = self._geometrias_metricas(municipios, "municipios")
        features_mun = []
        for codigo, grupo in municipios.groupby("_key", sort=True):
            ids = zonas.loc[zonas.mun_code.eq(codigo), "id"].tolist()
            features_mun.append(self._feature(self._union(grupo.geometry), {"key": codigo, "label": self._texto(grupo.iloc[0][mun_name]), "zone_ids": ids}, 30))
        mun_faltantes = presentes - set(municipios._key)
        if mun_faltantes:
            self.diagnostico.append("No hay geometría de referencia para estos municipios de la capa final: " + ", ".join(sorted(mun_faltantes)) + ".")

        features_utam = []
        grupos = zonas[zonas.utam.ne("")].groupby(["mun_code", "loc_code", "utam"], sort=True, dropna=False)
        for (mun, loc, utam), grupo in grupos:
            key = mun + "|" + loc + "|" + utam
            props = {"key": key, "label": self._etiqueta_utam(utam), "zone_ids": grupo.id.tolist(), "loc_code": loc, "mun_code": mun, "utam": utam}
            features_utam.append(self._feature(self._union(zonas_metricas.loc[grupo.index].geometry), props))
        self.geodata = {
            "zonas": self._coleccion(features_zonas),
            "localidades": self._coleccion(features_loc),
            "municipios": self._coleccion(features_mun),
            "utams": self._coleccion(features_utam),
        }

    @staticmethod
    def _etiqueta_utam(utam):
        if not utam or str(utam).strip().upper() in {"N/A", "NA", "NONE", "NAN", "NULL"}:
            return "Sin UTAM / UPR"
        return re.sub(r"^(UTAM|UPR)\s*(\d+)", r"\1 \2", str(utam), flags=re.IGNORECASE)

    @classmethod
    def _orden(cls, valor):
        return tuple((0, int(x)) if x.isdigit() else (1, cls._normalizar(x)) for x in re.split(r"(\d+)", str(valor)) if x)

    def _rama_utam(self, grupo):
        """Agrupa por UTAM/UPR cuando existe; deja las demás zonas como hojas."""
        ramas = []
        utams = grupo.utam.map(self._texto)
        sin_utam = utams.map(self._etiqueta_utam).eq("Sin UTAM / UPR")
        for utam in sorted(utams.loc[~sin_utam].unique(), key=self._orden):
            filas = grupo.loc[utams.eq(utam)]
            hojas = [{"value": r.id, "label": r.label} for r in sorted(filas.itertuples(), key=lambda r: self._orden(r.id))]
            ramas.append({"label": self._etiqueta_utam(utam), "children": hojas})
        ramas.extend({"value": r.id, "label": r.label} for r in sorted(grupo.loc[sin_utam].itertuples(), key=lambda r: self._orden(r.id)))
        return ramas

    def jerarquia(self, ids=None):
        """Devuelve nodos label/children y hojas value/label; sin duplicar zonas.

        Las dos ramas principales son Bogotá y Fuera de Bogotá. La primera
        contiene localidades; la segunda, municipios con código definitivo.
        Las asignaciones no resueltas y los ID sin correspondencia se muestran
        por separado, sin inferir su pertenencia territorial ni repetir el join.
        """
        if ids is None:
            zonas = self.zonas
            desconocidos = set()
        else:
            seleccion = {self._codigo(v) for v in ids if self._codigo(v)}
            zonas = self.zonas.loc[self.zonas.id.isin(seleccion)]
            desconocidos = seleccion - set(self.zonas.id)
        arbol = []
        nombres_mun = zonas.mun_name.map(self._normalizar)
        multimunicipales = nombres_mun.eq("MULTIMUNICIPAL")
        municipio_definitivo = (
            zonas.mun_code.str.fullmatch(r"\d{5}", na=False)
            & zonas.mun_code.ne("00000")
            & ~nombres_mun.isin({"MULTIMUNICIPAL", "SINDETERMINAR", "SINMUNICIPIOASIGNADO"})
        )
        bogota = zonas.loc[municipio_definitivo & zonas.mun_code.eq("11001")]
        localidades = []
        for loc in sorted(bogota.loc_code.unique(), key=lambda c: (not bool(c), c)):
            grupo = bogota.loc[bogota.loc_code.eq(loc)]
            nombres = grupo.loc_name[grupo.loc_name.ne("")]
            nombre = (nombres.iloc[0] if len(nombres) else "Localidad " + loc) if loc else "Sin localidad asignada"
            localidades.append({"label": nombre, "children": self._rama_utam(grupo)})
        if localidades:
            arbol.append({"label": "Bogotá", "children": localidades})
        externos = zonas.loc[municipio_definitivo & zonas.mun_code.ne("11001")]
        municipios = []
        for codigo, grupo in externos.groupby("mun_code", sort=True):
            nombres = grupo.mun_name[grupo.mun_name.ne("")]
            nombre = nombres.iloc[0] if len(nombres) else "Municipio " + codigo
            municipios.append({"label": nombre, "children": self._rama_utam(grupo)})
        if municipios:
            arbol.append({"label": "Fuera de Bogotá", "children": sorted(municipios, key=lambda m: self._orden(m["label"]))})
        revisar = []
        for mascara, nombre in (
            (~municipio_definitivo & multimunicipales, "Multimunicipal"),
            (~municipio_definitivo & ~multimunicipales, "Sin municipio definitivo"),
        ):
            grupo = zonas.loc[mascara]
            if not grupo.empty:
                revisar.append({"label": nombre, "children": self._rama_utam(grupo)})
        if desconocidos:
            revisar.append({"label": "Sin correspondencia cartográfica", "children": [{"value": z, "label": "Zona " + z + " (sin geometría)"} for z in sorted(desconocidos, key=self._orden)]})
        if revisar:
            arbol.append({"label": "Revisar asignación territorial", "children": revisar})
        return arbol

# Se incorpora después de VisorTelefonia. No cambia la importación ni la
# exclusión de registros profesionales definida en la clase base.
class VisorConMapas(VisorTelefonia):
    LIMITE_PARES = 10000

    def __init__(self, carpeta, geo, rutas_locales=None, avisar=None):
        super().__init__(carpeta, rutas_locales=rutas_locales, avisar=avisar)
        self.geo = geo
        columnas = ['id', 'label', 'zat', 'utam', 'mun_code', 'mun_name',
                    'loc_code', 'loc_name', 'rev_loc', 'rev_mun']
        faltantes = set(columnas) - set(geo.zonas.columns)
        if faltantes:
            raise ValueError('Faltan campos normalizados de zonificación: ' + ', '.join(sorted(faltantes)))
        zonas = geo.zonas[columnas].copy().fillna('')
        for campo in columnas:
            zonas[campo] = zonas[campo].astype(str).str.strip()
        if zonas['id'].eq('').any() or zonas['id'].duplicated().any():
            raise ValueError('La zonificación final necesita un ID único y no vacío por zona.')
        self.ids_cartografia = set(zonas['id'])
        self.diagnosticos_clave = {}
        self._diferencias_id_zat = int(((zonas['zat'] != '') & (zonas['zat'] != zonas['id'])).sum())
        self.db.register('_od_zonas_dataframe', zonas)
        self.db.execute('CREATE OR REPLACE TEMP TABLE od_zonas AS SELECT * FROM _od_zonas_dataframe')
        self.db.unregister('_od_zonas_dataframe')

    def preparar(self, anio, dia):
        tabla = super().preparar(anio, dia)
        dom = self.dominios[(anio, dia)]
        # La relación es exclusivamente matriz.origen/destino = zonificación.ID.
        # No se generan alias usando ZAT: existen códigos ID/ZAT contradictorios.
        ids_entrega = self.zonas_archivo.get(anio, set())
        confirmados = self.ids_cartografia & ids_entrega if ids_entrega else self.ids_cartografia
        dom['ids_zona'] = confirmados
        dom['hogar_no_zat'] = dom['residencia'] - confirmados - {NULO}
        observados = (dom['origen'] | dom['destino']) - {NULO}
        self.diagnosticos_clave[(anio, dia)] = {
            'field': 'ID', 'observed_count': len(observados),
            'matched_count': len(observados & confirmados),
            'unmatched_codes': sorted(observados - confirmados, key=orden_codigo),
            'source_dbf_available': bool(ids_entrega),
            'source_ids_not_in_final': sorted(ids_entrega - self.ids_cartografia, key=orden_codigo),
            'id_zat_differences': self._diferencias_id_zat,
        }
        return tabla

    def opciones(self, dia, anio=None):
        resultado = super().opciones(dia)
        doms = [d for (a, t), d in self.dominios.items() if t == dia and (anio is None or a == anio)]
        if not doms:
            # Permite preparar una consulta aun si el año/día todavía no se cargó.
            # Es el catálogo cartográfico, no una afirmación de viajes observados.
            for campo in ['origen', 'destino', 'residencia']:
                resultado[campo] = self.geo.jerarquia()
            return resultado
        union = lambda campo: set().union(*(d[campo] for d in doms)) if doms else set()
        for campo in ['origen', 'destino']:
            valores = union(campo)
            resultado[campo] = self.geo.jerarquia(valores - {NULO})
            if NULO in valores:
                resultado[campo].append({'value': NULO, 'label': 'Sin dato de zona'})
        hogares = set().union(*(d['residencia'] & d['ids_zona'] for d in doms)) if doms else set()
        resultado['residencia'] = self.geo.jerarquia(hogares)
        if any(d['hogar_no_zat'] for d in doms):
            resultado['residencia'].append({'value': NO_ZAT, 'label': 'Sin ZAT de hogar identificada'})
        if NULO in union('residencia'):
            resultado['residencia'].append({'value': NULO, 'label': 'Sin dato de hogar'})
        return resultado

    def consultar(self, solicitud=None):
        # Las pestañas sin mapas conservan las tres series comparables.
        resultado = super().consultar(solicitud)
        resultado['mapping'] = {
            str(a): self.diagnosticos_clave.get((a, resultado['filters']['dia']), {}) for a in ANIOS
        }
        return resultado

    @staticmethod
    def _numero_od(valor):
        valor = float(valor or 0)
        if not math.isfinite(valor) or valor < 0:
            raise ValueError('La agregación contiene un total de viajes inválido.')
        return valor

    def _avisos_clave(self, anio, dia):
        d = self.diagnosticos_clave.get((anio, dia), {})
        avisos = []
        if not d.get('source_dbf_available'):
            avisos.append('Cruce cartográfico por ID. El ZIP de matrices no permitió verificar su DBF de zonificación; revise la correspondencia de claves antes de interpretar los mapas.')
        if d.get('unmatched_codes'):
            avisos.append(f"Hay {len(d['unmatched_codes'])} códigos de las matrices sin correspondencia cartográfica; sus viajes se informan por separado.")
        if self._diferencias_id_zat:
            avisos.append(f'El cruce utiliza ID, sin sustituirlo por ZAT. En {self._diferencias_id_zat} zonas ambos campos tienen valores distintos.')
        return avisos

    def _estado_od(self, f, vista, anio):
        return {'year': anio, 'view': vista, 'filters': f, 'available': False,
                'origins': [], 'destinations': [], 'pairs': [], 'total': None,
                'base_total': None, 'origin_total': None, 'destination_total': None,
                'unmapped': {}, 'options': self.opciones(f['dia'], anio),
                'warnings': [], 'pair_count': 0, 'pairs_truncated': False}

    def _agrupar_lado_od(self, clave, etiqueta, criterio):
        filas = self.db.execute(f'''SELECT {clave} AS clave,
            any_value({etiqueta}) AS etiqueta, sum(value) AS viajes
            FROM od_cruce WHERE {criterio}
            GROUP BY 1 ORDER BY viajes DESC, clave''').fetchall()
        total = sum(self._numero_od(f[2]) for f in filas)
        return [{'key': str(k), 'label': str(l or k), 'value': self._numero_od(v),
                 'pct': 100 * self._numero_od(v) / total if total else 0.0}
                for k, l, v in filas], self._numero_od(total)

    def consultar_od(self, solicitud, vista, anio):
        if vista not in {'regional', 'localidades', 'zat'}:
            raise ValueError('Vista origen-destino no válida.')
        if type(anio) is not int or anio not in ANIOS:
            raise ValueError('Seleccione un único año: 2019, 2023 o 2025.')
        with self.lock:
            f = self.validar_filtros(solicitud)
            resultado = self._estado_od(f, vista, anio)
            if vista == 'zat' and (f['origen'] == [] or f['destino'] == [] or
                                  (f['origen'] is None and f['destino'] is None)):
                resultado.update(pending=True, message='Seleccione una o varias zonas en al menos un extremo y zonas o «Todos» en el otro.')
                return resultado
            try:
                tabla = self.preparar(anio, f['dia'])
                resultado['options'] = self.opciones(f['dia'], anio)
                resultado['mapping'] = self.diagnosticos_clave[(anio, f['dia'])]
                resultado['warnings'] = self._avisos_clave(anio, f['dia'])
                condicion, params, aviso = self.condicion(anio, f)
                if aviso:
                    resultado['message'] = aviso
                    return resultado
                # Primero filtrar y agregar la matriz grande; después unir las
                # geometrías/atributos, evitando un join por cada registro original.
                self.db.execute('DROP VIEW IF EXISTS od_cruce')
                self.db.execute(f'''CREATE OR REPLACE TEMP TABLE od_pares AS
                    SELECT origen, destino, sum(viajes) AS value FROM {tabla}
                    WHERE {condicion} GROUP BY origen, destino HAVING sum(viajes) > 0''', params)
                self.db.execute('''CREATE TEMP VIEW od_cruce AS
                    SELECT p.*, o.id AS o_id, d.id AS d_id,
                      o.label AS o_label, d.label AS d_label,
                      o.mun_code AS o_mun, d.mun_code AS d_mun,
                      o.mun_name AS o_mun_name, d.mun_name AS d_mun_name,
                      o.loc_code AS o_loc, d.loc_code AS d_loc,
                      o.loc_name AS o_loc_name, d.loc_name AS d_loc_name,
                      o.rev_loc AS o_rev_loc, d.rev_loc AS d_rev_loc
                    FROM od_pares p LEFT JOIN od_zonas o ON p.origen = o.id
                    LEFT JOIN od_zonas d ON p.destino = d.id''')
                base_total, pares = self.db.execute('SELECT coalesce(sum(value),0), count(*) FROM od_pares').fetchone()
                base_total = self._numero_od(base_total)
                resultado.update(available=True, base_total=base_total, total=base_total)
                resultado['unmapped'] = self._sin_correspondencia_od()
                if vista == 'regional':
                    # Izquierda: municipios que originan viajes hacia Bogotá.
                    # Derecha: municipios que reciben viajes desde Bogotá.
                    ext_o = "((o_mun <> '' AND o_mun <> '11001') OR lower(o_mun_name) = 'multimunicipal')"
                    ext_d = "((d_mun <> '' AND d_mun <> '11001') OR lower(d_mun_name) = 'multimunicipal')"
                    o_clave = "CASE WHEN lower(o_mun_name) = 'multimunicipal' THEN '__MULTIMUNICIPAL__' ELSE o_mun END"
                    d_clave = "CASE WHEN lower(d_mun_name) = 'multimunicipal' THEN '__MULTIMUNICIPAL__' ELSE d_mun END"
                    origenes, o_total = self._agrupar_lado_od(o_clave, 'o_mun_name', f"d_mun = '11001' AND {ext_o}")
                    destinos, d_total = self._agrupar_lado_od(d_clave, 'd_mun_name', f"o_mun = '11001' AND {ext_d}")
                    resultado['total'] = self._numero_od(o_total + d_total)
                    if any(r['key'] == '__MULTIMUNICIPAL__' for r in origenes + destinos):
                        resultado['warnings'].append('Multimunicipal se presenta como categoría independiente en las gráficas y tablas; sus viajes no se reparten entre municipios ni se pintan sobre un municipio concreto.')
                    if any(resultado['unmapped'][k] for k in ['municipal_origins', 'municipal_destinations']):
                        resultado['warnings'].append('Hay viajes sin municipio identificado en uno o ambos extremos. Se informan por separado y no se incluyen en los totales regionales clasificados.')
                elif vista == 'localidades':
                    origenes, o_total = self._agrupar_lado_od('o_loc', 'o_loc_name', "o_mun = '11001' AND o_loc <> ''")
                    destinos, d_total = self._agrupar_lado_od('d_loc', 'd_loc_name', "d_mun = '11001' AND d_loc <> ''")
                    if any(resultado['unmapped'][k] for k in ['locality_origins', 'locality_destinations']):
                        resultado['warnings'].append('Hay viajes de zonas de Bogotá sin localidad asignada; se informan por separado y no se asignan a otra localidad.')
                    dudosas = self.db.execute("SELECT count(*) FROM od_cruce WHERE (o_mun='11001' AND o_rev_loc IS NOT NULL AND o_rev_loc <> '' AND o_rev_loc <> 'OK') OR (d_mun='11001' AND d_rev_loc IS NOT NULL AND d_rev_loc <> '' AND d_rev_loc <> 'OK')").fetchone()[0]
                    if dudosas:
                        resultado['warnings'].append('Agregación según la localidad asignada en la zonificación final. Algunas asignaciones conservan observaciones pendientes de revisión.')
                else:
                    # Un código sin geometría permanece en las tablas y totales.
                    origenes, o_total = self._agrupar_lado_od('origen', "coalesce(o_label, 'Zona ' || origen || ' (sin geometría)')", 'TRUE')
                    destinos, d_total = self._agrupar_lado_od('destino', "coalesce(d_label, 'Zona ' || destino || ' (sin geometría)')", 'TRUE')
                    resultado['pair_count'] = int(pares)
                    filas = self.db.execute('''SELECT origen, destino,
                        coalesce(o_label, 'Zona ' || origen || ' (sin geometría)'),
                        coalesce(d_label, 'Zona ' || destino || ' (sin geometría)'), value
                        FROM od_cruce ORDER BY value DESC, origen, destino LIMIT ?''', [self.LIMITE_PARES]).fetchall()
                    resultado['pairs'] = [{'origin': o, 'destination': d, 'origin_label': ol,
                        'destination_label': dl, 'value': self._numero_od(v),
                        'pct': 100 * self._numero_od(v) / base_total if base_total else 0.0}
                        for o, d, ol, dl, v in filas]
                    resultado['pairs_truncated'] = pares > self.LIMITE_PARES
                    if resultado['pairs_truncated']:
                        resultado['warnings'].append(f'Se muestran las {self.LIMITE_PARES:,} relaciones con más viajes de {pares:,}. Los mapas, las gráficas y los totales incluyen todas. Descarga la tabla para obtener todas las relaciones.')
                resultado.update(origins=origenes, destinations=destinos,
                                 origin_total=o_total, destination_total=d_total)
                if not origenes and not destinos:
                    inciertos = (resultado['unmapped']['municipal_origins'] + resultado['unmapped']['municipal_destinations']
                                 if vista == 'regional' else resultado['unmapped']['either'] +
                                 resultado['unmapped']['locality_origins'] + resultado['unmapped']['locality_destinations'])
                    resultado['message'] = ('No hay viajes representables; consulte los viajes sin asignación territorial.' if inciertos else
                        'No hay viajes regionales para los filtros seleccionados.' if vista == 'regional' else
                        'No hay viajes representables para los filtros seleccionados.')
                return resultado
            except Exception as exc:
                # Un fallo o falta de desagregación nunca se convierte en cero.
                resultado.update(available=False, origins=[], destinations=[], pairs=[],
                                 total=None, base_total=None, origin_total=None, destination_total=None,
                                 message=f'{anio}: {str(exc)[:300]}')
                return resultado

    def _sin_correspondencia_od(self):
        condiciones = {
            'origins': 'o_id IS NULL', 'destinations': 'd_id IS NULL',
            'either': 'o_id IS NULL OR d_id IS NULL',
            'municipal_origins': "o_id IS NULL OR (coalesce(o_mun,'')='' AND lower(coalesce(o_mun_name,'')) <> 'multimunicipal')",
            'municipal_destinations': "d_id IS NULL OR (coalesce(d_mun,'')='' AND lower(coalesce(d_mun_name,'')) <> 'multimunicipal')",
            'locality_origins': "o_mun='11001' AND coalesce(o_loc,'')=''",
            'locality_destinations': "d_mun='11001' AND coalesce(d_loc,'')=''",
        }
        expresiones = [f'coalesce(sum(value) FILTER (WHERE {c}),0)' for c in condiciones.values()]
        fila = self.db.execute('SELECT ' + ','.join(expresiones) + ' FROM od_cruce').fetchone()
        salida = {c: self._numero_od(v) for c, v in zip(condiciones, fila)}
        salida['origin_codes'] = [r[0] for r in self.db.execute('SELECT DISTINCT origen FROM od_cruce WHERE o_id IS NULL ORDER BY origen').fetchall()]
        salida['destination_codes'] = [r[0] for r in self.db.execute('SELECT DISTINCT destino FROM od_cruce WHERE d_id IS NULL ORDER BY destino').fetchall()]
        return salida

    def exportar_pares(self, solicitud, anio, ruta):
        # La exportación no se limita a las filas enviadas a la interfaz.
        with self.lock:
            resultado = self.consultar_od(solicitud, 'zat', anio)
            if not resultado['available']:
                raise ValueError(resultado.get('message', 'No hay una consulta ZAT disponible.'))
            ruta = Path(ruta)
            ruta.parent.mkdir(parents=True, exist_ok=True)
            self.db.execute(f'''COPY (
                SELECT origen AS "Código origen", destino AS "Código destino",
                    coalesce(o_label, 'Zona ' || origen || ' (sin geometría)') AS "ZAT origen",
                    coalesce(d_label, 'Zona ' || destino || ' (sin geometría)') AS "ZAT destino",
                    printf('%.0f', value) AS "Viajes",
                    printf('%.1f', 100.0 * value / sum(value) OVER ()) AS "Porcentaje"
                FROM od_cruce ORDER BY value DESC, origen, destino
                ) TO {literal_sql(ruta)} (HEADER, DELIMITER ';')''')
            return str(ruta)

PLANTILLA = '<div id="visor-telefonia">\n<style>\n#visor-telefonia{font:15px Arial,sans-serif;color:#344055;background:#fff;padding:12px 8px 24px;box-sizing:border-box;min-height:820px}\n#visor-telefonia *{box-sizing:border-box}#visor-telefonia button,#visor-telefonia input{font:inherit;color:inherit}\n#visor-telefonia h1{font-size:25px;color:#19212e;line-height:1.15;margin:0 0 22px;font-weight:700}\n#visor-telefonia h1 span{display:block;font-size:18px;font-weight:400;margin-top:4px}\n#visor-telefonia .shell{display:grid;grid-template-columns:172px minmax(0,1fr);gap:20px;max-width:none;margin:auto}\n#visor-telefonia .side{border:1px solid #d8dadd;border-radius:17px;padding:20px 12px;height:max-content;box-shadow:0 3px 12px #00000008}\n#visor-telefonia .side strong{display:block;font-size:18px;color:#222;margin:0 9px 18px}\n#visor-telefonia .nav{display:block;border:0;background:none;text-align:left;width:100%;padding:11px 10px;border-radius:6px;cursor:pointer;font-size:17px;color:#1f2937}\n#visor-telefonia .nav.active{background:#f0f3e6;color:#263619;font-weight:bold;box-shadow:inset 3px 0 #a5cb00}\n#visor-telefonia .nav:disabled{color:#a2a8af;cursor:default}#visor-telefonia .nav:focus-visible{outline:2px solid #445cc4}\n#visor-telefonia .main{min-width:0}#visor-telefonia .filters{position:relative;z-index:20;border:1px solid #d7d9dc;border-radius:14px;padding:22px 22px 18px;background:#fff;box-shadow:0 3px 12px #00000008}\n#visor-telefonia .top{display:grid;grid-template-columns:1fr 1fr 1fr 1.08fr;gap:24px}\n#visor-telefonia .bottom{display:grid;grid-template-columns:1fr 1fr 1fr;gap:28px;margin-top:27px}\n#visor-telefonia .field{position:relative;min-width:0}#visor-telefonia .field.open{z-index:50}\n#visor-telefonia .label{display:block;font-size:16px;margin-bottom:8px;white-space:nowrap}\n#visor-telefonia .trigger{width:100%;height:39px;border:1px solid #cfd4dc;background:#fff;border-radius:2px;text-align:left;display:flex;align-items:center;justify-content:space-between;padding:0 11px;cursor:pointer;gap:8px}\n#visor-telefonia .trigger-text{white-space:nowrap;overflow:hidden;text-overflow:ellipsis}#visor-telefonia .chevron{width:9px;height:9px;border-bottom:1.5px solid;border-right:1.5px solid;transform:rotate(45deg);flex-shrink:0;margin-right:3px;margin-top:-4px}\n#visor-telefonia .open>.trigger .chevron{transform:rotate(225deg);margin-top:4px}\n#visor-telefonia .hint{font-size:12px;margin-top:6px;color:#697386}\n#visor-telefonia .popup{display:none;position:absolute;top:65px;left:0;width:100%;min-width:220px;border:1px solid #cbd1d9;background:#fff;box-shadow:0 5px 16px #1b263426;border-radius:0 0 5px 5px;overflow:hidden;z-index:60}\n#visor-telefonia .field.open>.popup{display:block}#visor-telefonia .field[data-key="motivo"] .popup{width:385px;max-width:70vw}\n#visor-telefonia .search{width:100%;height:37px;border:0;border-bottom:1px solid #ddd;padding:7px 11px 7px 33px;background:#fff}\n#visor-telefonia .searchbox{position:relative}#visor-telefonia .searchbox:before{content:\'⌕\';position:absolute;left:10px;top:3px;font-size:26px;color:#718096;pointer-events:none}\n#visor-telefonia .choices{overflow-y:scroll;max-height:290px;min-height:40px;scrollbar-color:#9198a3 #edf0f3;scrollbar-width:auto}\n#visor-telefonia .choices::-webkit-scrollbar{width:12px}#visor-telefonia .choices::-webkit-scrollbar-track{background:#edf0f3}#visor-telefonia .choices::-webkit-scrollbar-thumb{background:#969da7;border:2px solid #edf0f3;border-radius:6px}\n#visor-telefonia .choice{display:flex;align-items:center;gap:9px;min-height:32px;padding:5px 10px;cursor:pointer;font-size:14px;line-height:1.2}\n#visor-telefonia .choice:hover{background:#f5f7f9}#visor-telefonia .choice input{accent-color:#495467;width:16px;height:16px;margin:0;flex-shrink:0}\n#visor-telefonia .choice span{overflow-wrap:anywhere}#visor-telefonia .parent{font-weight:500}#visor-telefonia .child{padding-left:35px}\n#visor-telefonia .expand{border:0;background:none;width:14px;padding:0;cursor:pointer;flex-shrink:0;font-size:15px}\n#visor-telefonia .popup-foot{padding:8px 11px;font-size:11px;border-top:1px solid #ddd;color:#6b7280}\n#visor-telefonia .hours{display:flex;gap:10px}#visor-telefonia .hours input{width:50%;min-width:0;height:37px;border:1px solid #cfd4dc;padding:6px 9px}\n#visor-telefonia .ranges{position:relative;height:27px;margin-top:4px}\n#visor-telefonia .rail,#visor-telefonia .rail-active{position:absolute;top:11px;left:0;right:0;height:4px;background:#e4e7dd}\n#visor-telefonia .rail-active{background:#afd000}\n#visor-telefonia .ranges input{pointer-events:none;position:absolute;left:0;top:0;width:100%;height:26px;margin:0;background:none;-webkit-appearance:none;appearance:none}\n#visor-telefonia .ranges input::-webkit-slider-thumb{pointer-events:auto;cursor:pointer;-webkit-appearance:none;width:22px;height:22px;border:3px solid #afcd00;background:white;border-radius:50%}\n#visor-telefonia .ranges input::-moz-range-thumb{pointer-events:auto;cursor:pointer;width:17px;height:17px;border:3px solid #afcd00;background:white;border-radius:50%}\n#visor-telefonia .actions{display:flex;gap:10px;align-items:center;margin:16px 0 14px;flex-wrap:wrap}\n#visor-telefonia .btn{border:1px solid #cfd4dc;background:#fff;border-radius:6px;padding:8px 13px;cursor:pointer}\n#visor-telefonia .btn.primary{background:#394856;color:#fff;border-color:#394856}#visor-telefonia .btn:disabled{opacity:.5;cursor:wait}\n#visor-telefonia .status{font-size:12px;color:#697386;flex:1;min-width:160px}#visor-telefonia .status.dirty{color:#92630b}\n#visor-telefonia .note{font-size:12px;line-height:1.5;color:#697386;margin:4px 0 10px}#visor-telefonia .errors{font-size:13px;color:#99521c;line-height:1.6;margin:8px 0}\n#visor-telefonia .results{border:1px solid #e0e3e6;border-radius:12px;background:#fff;padding:19px 22px;position:relative;z-index:1}\n#visor-telefonia .result-head{display:flex;justify-content:space-between;gap:12px;align-items:center;margin-bottom:8px}\n#visor-telefonia h2{font-size:18px;margin:0;color:#1e2835}#visor-telefonia .legend{display:flex;gap:22px;justify-content:center;margin:4px 0 12px}\n#visor-telefonia .legend span{display:flex;align-items:center;gap:6px;font-size:13px}#visor-telefonia .swatch{width:10px;height:10px;border-radius:50%;display:inline-block}\n#visor-telefonia .chart{width:100%;min-height:355px}#visor-telefonia .chart svg{width:100%;height:auto;display:block;overflow:visible}\n#visor-telefonia .table-wrap{overflow-x:auto;width:100%;max-width:640px;margin:20px auto 0}#visor-telefonia .table-wrap.totals-only{max-width:340px}#visor-telefonia table{border-collapse:collapse;width:100%;font-size:13px}\n#visor-telefonia th,#visor-telefonia td{padding:10px 12px;border-bottom:1px solid #e5e8ec;text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}\n#visor-telefonia th:first-child,#visor-telefonia td:first-child{text-align:left;white-space:normal;max-width:300px}\n#visor-telefonia th{background:#f5f7f9;font-weight:600;color:#445064}#visor-telefonia tr.total td{font-weight:bold;border-top:2px solid #d8dde3}\n#visor-telefonia .empty{padding:100px 20px;text-align:center;color:#7b8491}#visor-telefonia .foot{margin-top:15px;font-size:11px;color:#7b8491}\n@media(max-width:1000px){#visor-telefonia .shell{grid-template-columns:132px minmax(0,1fr);gap:12px}#visor-telefonia .top{gap:12px}#visor-telefonia .label{font-size:14px}#visor-telefonia .filters{padding:18px 15px}#visor-telefonia .side{padding:18px 7px}#visor-telefonia .nav{font-size:15px}}\n@media(max-width:720px){#visor-telefonia .shell{grid-template-columns:1fr}#visor-telefonia .side{display:flex;overflow-x:auto;padding:8px}#visor-telefonia .side strong{display:none}#visor-telefonia .nav{width:auto;white-space:nowrap}#visor-telefonia .top{grid-template-columns:1fr 1fr;row-gap:18px}#visor-telefonia .bottom{gap:10px}#visor-telefonia .label{white-space:normal;min-height:32px}#visor-telefonia .popup{top:79px}#visor-telefonia .results{padding:15px 10px}#visor-telefonia .chart{min-height:260px}}\n\n#visor-telefonia [hidden]{display:none!important}\n#visor-telefonia .choice-label{display:flex;align-items:center;gap:9px;flex:1;min-width:0;cursor:pointer}\n#visor-telefonia .field[data-key="residencia"] .popup,#visor-telefonia .field[data-key="origen"] .popup,#visor-telefonia .field[data-key="destino"] .popup{min-width:315px;max-width:80vw}\n#visor-telefonia .year-selector{display:flex;align-items:center;gap:17px;flex-wrap:wrap;border:0;padding:0;margin:0 0 18px}\n#visor-telefonia .year-selector legend{float:left;font-weight:bold;margin-right:18px}\n#visor-telefonia .year-selector label{display:flex;align-items:center;gap:6px;cursor:pointer}\n#visor-telefonia .year-selector input{accent-color:#185e9d}\n#visor-telefonia .od-results{padding:18px 14px}\n#visor-telefonia .od-results .result-head{margin-bottom:20px}\n#visor-telefonia #od-pairs{margin:28px auto 0;max-width:760px}\n#visor-telefonia #pair-table{max-height:410px;overflow:auto;margin:16px auto;width:fit-content;max-width:100%}\n#visor-telefonia #pair-table th{position:sticky;top:0}\n#visor-telefonia .normal-results .table-wrap{max-width:700px}\n#visor-telefonia .normal-results .table-wrap.totals-only{max-width:340px}\n#visor-telefonia .normal-results th,#visor-telefonia .normal-results td{font-size:12px;padding:9px 8px}\n#visor-telefonia .errors:not(:empty){padding:10px 12px;border:1px solid #eadabf;border-radius:6px;background:#fffcf6}\n#visor-telefonia .busy #od-maps{pointer-events:none;opacity:.8}\n#visor-telefonia .data-details{font-size:12px;color:#687386;margin:10px 0}\n#visor-telefonia .data-details summary{cursor:pointer}\n\n/* Mapas OD: reglas limitadas a este componente para preservar el panel. */\n#visor-telefonia .od-maps { width: 100%; min-width: 0; color: #27354a; }\n#visor-telefonia .od-maps .od-map-grid { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); gap: 18px; align-items: start; }\n#visor-telefonia .od-maps .od-map-column { min-width: 0; margin: 0; padding: 0; }\n#visor-telefonia .od-maps .od-map-heading { padding: 0 2px 9px; min-height: 83px; }\n#visor-telefonia .od-maps .od-map-title { margin: 0 0 5px; font-size: 19px; line-height: 1.3; font-weight: 700; color: #253550; }\n#visor-telefonia .od-maps .od-map-subtitle { margin: 0 0 5px; font-size: 12px; color: #6b7586; line-height: 1.35; }\n#visor-telefonia .od-maps .od-map-count { font-size: 20px; font-weight: 700; color: #17477f; }\n#visor-telefonia .od-maps .od-map-tools { display: flex; justify-content: space-between; align-items: center; gap: 8px; margin: 0 0 7px; }\n#visor-telefonia .od-maps .od-basemap-tools { display: flex; align-items: center; flex-wrap: wrap; gap: 8px; margin: 0 0 12px; color: #596574; font-size: 12px; }\n#visor-telefonia .od-maps .od-basemap-tools select { padding: 5px 8px; border: 1px solid #d8dfe8; border-radius: 6px; color: #26354a; background: #fff; }\n#visor-telefonia .od-maps .od-map-year { font-size: 13px; font-weight: 650; }\n#visor-telefonia .od-maps .od-reset-extent { cursor: pointer; border: 1px solid #cfd7e1; border-radius: 5px; padding: 5px 9px; color: #40546c; background: white; font: inherit; font-size: 11px; line-height: 1.4; }\n#visor-telefonia .od-maps .od-reset-extent:hover { background: #f0f5fa; }\n#visor-telefonia .od-maps .od-reset-extent:focus-visible, #visor-telefonia .od-maps .od-bar-row:focus-visible { outline: 3px solid #7aa8e1; outline-offset: 2px; }\n#visor-telefonia .od-maps .od-map-frame { position: relative; overflow: hidden; border: 1px solid #dce2e8; border-radius: 7px; background: #f3f5f7; isolation: isolate; }\n#visor-telefonia .od-maps .od-map-canvas { width: 100%; height: clamp(520px, 66vh, 720px); min-height: 520px; background: #eef1f4; font-family: inherit; }\n#visor-telefonia .od-maps .od-map-empty { position: absolute; top: 14px; left: 55px; right: 14px; z-index: 450; padding: 11px 14px; border-radius: 6px; background: rgba(255,255,255,.95); border: 1px solid #d4dce6; box-shadow: 0 2px 9px #25354a15; color: #526078; font-size: 13px; line-height: 1.45; pointer-events: none; }\n#visor-telefonia .od-maps [hidden] { display: none !important; }\n#visor-telefonia .od-maps .od-map-legend { display: flex; align-items: center; gap: 8px; padding: 9px 3px 3px; font-size: 11px; color: #5c6a7d; }\n#visor-telefonia .od-maps .od-legend-unit { font-weight: 600; }\n#visor-telefonia .od-maps .od-legend-gradient { flex: 1; max-width: 200px; min-width: 50px; height: 9px; background: linear-gradient(to right, #eff6ff, #bfdbfe, #60a5fa, #2563eb, #08306b); border: 1px solid #d2dce8; border-radius: 2px; }\n#visor-telefonia .od-maps .od-legend-min, #visor-telefonia .od-maps .od-legend-max { white-space: nowrap; }\n#visor-telefonia .od-maps .od-selection-key { margin: 7px 2px 0; color: #798394; font-size: 10px; line-height: 1.5; }\n#visor-telefonia .od-maps .od-selection-letter { display: flex; align-items: center; justify-content: center; border: 0; background: transparent; }\n#visor-telefonia .od-maps .od-selection-letter span { display: inline-flex; align-items: center; justify-content: center; min-width: 21px; height: 22px; padding: 0 3px; color: #b71f2b; font: 800 13px/1 Arial, sans-serif; background: rgba(255,255,255,.94); border: 1px solid #c73a44; border-radius: 4px; box-shadow: 0 1px 4px #26344535; }\n#visor-telefonia .od-maps .od-map-tooltip { font-family: inherit; font-size: 12px; padding: 8px 10px; line-height: 1.5; }\n#visor-telefonia .od-maps .od-tooltip-hint { color: #718096; font-size: 10px; }\n#visor-telefonia .od-maps .od-section-title { margin: 19px 2px 10px; font-size: 14px; line-height: 1.3; font-weight: 650; color: #34465e; }\n#visor-telefonia .od-maps .od-bar-chart { max-height: 405px; overflow-y: auto; padding: 1px 6px 1px 0; scrollbar-width: thin; scrollbar-color: #aab7c7 #edf1f5; }\n#visor-telefonia .od-maps .od-bar-row { display: grid; grid-template-columns: minmax(92px, 32%) minmax(45px, 1fr) minmax(62px, auto); align-items: center; gap: 8px; width: 100%; min-height: 30px; margin: 0; border: 0; border-radius: 3px; padding: 5px 4px; background: transparent; color: #46566c; font: inherit; font-size: 11px; line-height: 1.3; text-align: left; }\n#visor-telefonia .od-maps button.od-bar-row { cursor: pointer; }\n#visor-telefonia .od-maps button.od-bar-row:hover { background: #f1f5fa; }\n#visor-telefonia .od-maps .od-bar-label { min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }\n#visor-telefonia .od-maps .od-bar-track { height: 15px; display: block; background: #f0f3f7; border-radius: 2px; overflow: hidden; }\n#visor-telefonia .od-maps .od-bar-fill { display: block; height: 100%; border-radius: 2px; }\n#visor-telefonia .od-maps .od-bar-number { text-align: right; white-space: nowrap; font-variant-numeric: tabular-nums; }\n#visor-telefonia .od-maps .od-empty-chart { padding: 20px 12px; text-align: center; font-size: 13px; color: #778496; background: #f8fafc; border-radius: 5px; }\n#visor-telefonia .od-maps .od-compact-table { width: fit-content; max-width: 100%; max-height: 350px; margin: 18px auto 4px; overflow: auto; border: 1px solid #e1e6ed; border-radius: 5px; scrollbar-width: thin; scrollbar-color: #aab7c7 #edf1f5; }\n#visor-telefonia .od-maps table.od-values { width: auto; min-width: 275px; margin: 0; border-collapse: separate; border-spacing: 0; font-size: 11px; line-height: 1.4; color: #43546a; }\n#visor-telefonia .od-maps table.od-values th, #visor-telefonia .od-maps table.od-values td { padding: 7px 11px; border: 0; border-bottom: 1px solid #edf0f4; text-align: right; white-space: nowrap; font-variant-numeric: tabular-nums; }\n#visor-telefonia .od-maps table.od-values th:first-child, #visor-telefonia .od-maps table.od-values td:first-child { text-align: left; white-space: normal; max-width: 230px; }\n#visor-telefonia .od-maps table.od-values th { position: sticky; top: 0; z-index: 1; background: #f0f4f9; color: #3f5470; font-weight: 650; }\n#visor-telefonia .od-maps table.od-values tfoot td { position: sticky; bottom: 0; background: #edf3fa; font-weight: 700; border-top: 1px solid #d8e2ee; border-bottom: 0; }\n#visor-telefonia .od-maps .leaflet-control-attribution { font-size: 9px; line-height: 1.5; }\n#visor-telefonia .od-maps .leaflet-control-attribution a { color: #315d8f; }\n#visor-telefonia .od-maps .leaflet-container img.leaflet-tile { max-width: none !important; max-height: none !important; }\n#visor-telefonia .od-maps .leaflet-pane > svg { display: block; max-width: none !important; overflow: visible; }\n@media (max-width: 850px) {\n  #visor-telefonia .od-maps .od-map-grid { grid-template-columns: minmax(0, 1fr); gap: 28px; }\n  #visor-telefonia .od-maps .od-map-canvas { height: 520px; }\n  #visor-telefonia .od-maps .od-map-heading { min-height: 0; }\n}\n\n</style>\n<style>/* required styles */\n\n.leaflet-pane,\n.leaflet-tile,\n.leaflet-marker-icon,\n.leaflet-marker-shadow,\n.leaflet-tile-container,\n.leaflet-pane > svg,\n.leaflet-pane > canvas,\n.leaflet-zoom-box,\n.leaflet-image-layer,\n.leaflet-layer {\n\tposition: absolute;\n\tleft: 0;\n\ttop: 0;\n\t}\n.leaflet-container {\n\toverflow: hidden;\n\t}\n.leaflet-tile,\n.leaflet-marker-icon,\n.leaflet-marker-shadow {\n\t-webkit-user-select: none;\n\t   -moz-user-select: none;\n\t        user-select: none;\n\t  -webkit-user-drag: none;\n\t}\n/* Prevents IE11 from highlighting tiles in blue */\n.leaflet-tile::selection {\n\tbackground: transparent;\n}\n/* Safari renders non-retina tile on retina better with this, but Chrome is worse */\n.leaflet-safari .leaflet-tile {\n\timage-rendering: -webkit-optimize-contrast;\n\t}\n/* hack that prevents hw layers "stretching" when loading new tiles */\n.leaflet-safari .leaflet-tile-container {\n\twidth: 1600px;\n\theight: 1600px;\n\t-webkit-transform-origin: 0 0;\n\t}\n.leaflet-marker-icon,\n.leaflet-marker-shadow {\n\tdisplay: block;\n\t}\n/* .leaflet-container svg: reset svg max-width decleration shipped in Joomla! (joomla.org) 3.x */\n/* .leaflet-container img: map is broken in FF if you have max-width: 100% on tiles */\n.leaflet-container .leaflet-overlay-pane svg {\n\tmax-width: none !important;\n\tmax-height: none !important;\n\t}\n.leaflet-container .leaflet-marker-pane img,\n.leaflet-container .leaflet-shadow-pane img,\n.leaflet-container .leaflet-tile-pane img,\n.leaflet-container img.leaflet-image-layer,\n.leaflet-container .leaflet-tile {\n\tmax-width: none !important;\n\tmax-height: none !important;\n\twidth: auto;\n\tpadding: 0;\n\t}\n\n.leaflet-container img.leaflet-tile {\n\t/* See: https://bugs.chromium.org/p/chromium/issues/detail?id=600120 */\n\tmix-blend-mode: plus-lighter;\n}\n\n.leaflet-container.leaflet-touch-zoom {\n\t-ms-touch-action: pan-x pan-y;\n\ttouch-action: pan-x pan-y;\n\t}\n.leaflet-container.leaflet-touch-drag {\n\t-ms-touch-action: pinch-zoom;\n\t/* Fallback for FF which doesn\'t support pinch-zoom */\n\ttouch-action: none;\n\ttouch-action: pinch-zoom;\n}\n.leaflet-container.leaflet-touch-drag.leaflet-touch-zoom {\n\t-ms-touch-action: none;\n\ttouch-action: none;\n}\n.leaflet-container {\n\t-webkit-tap-highlight-color: transparent;\n}\n.leaflet-container a {\n\t-webkit-tap-highlight-color: rgba(51, 181, 229, 0.4);\n}\n.leaflet-tile {\n\tfilter: inherit;\n\tvisibility: hidden;\n\t}\n.leaflet-tile-loaded {\n\tvisibility: inherit;\n\t}\n.leaflet-zoom-box {\n\twidth: 0;\n\theight: 0;\n\t-moz-box-sizing: border-box;\n\t     box-sizing: border-box;\n\tz-index: 800;\n\t}\n/* workaround for https://bugzilla.mozilla.org/show_bug.cgi?id=888319 */\n.leaflet-overlay-pane svg {\n\t-moz-user-select: none;\n\t}\n\n.leaflet-pane         { z-index: 400; }\n\n.leaflet-tile-pane    { z-index: 200; }\n.leaflet-overlay-pane { z-index: 400; }\n.leaflet-shadow-pane  { z-index: 500; }\n.leaflet-marker-pane  { z-index: 600; }\n.leaflet-tooltip-pane   { z-index: 650; }\n.leaflet-popup-pane   { z-index: 700; }\n\n.leaflet-map-pane canvas { z-index: 100; }\n.leaflet-map-pane svg    { z-index: 200; }\n\n.leaflet-vml-shape {\n\twidth: 1px;\n\theight: 1px;\n\t}\n.lvml {\n\tbehavior: url(#default#VML);\n\tdisplay: inline-block;\n\tposition: absolute;\n\t}\n\n\n/* control positioning */\n\n.leaflet-control {\n\tposition: relative;\n\tz-index: 800;\n\tpointer-events: visiblePainted; /* IE 9-10 doesn\'t have auto */\n\tpointer-events: auto;\n\t}\n.leaflet-top,\n.leaflet-bottom {\n\tposition: absolute;\n\tz-index: 1000;\n\tpointer-events: none;\n\t}\n.leaflet-top {\n\ttop: 0;\n\t}\n.leaflet-right {\n\tright: 0;\n\t}\n.leaflet-bottom {\n\tbottom: 0;\n\t}\n.leaflet-left {\n\tleft: 0;\n\t}\n.leaflet-control {\n\tfloat: left;\n\tclear: both;\n\t}\n.leaflet-right .leaflet-control {\n\tfloat: right;\n\t}\n.leaflet-top .leaflet-control {\n\tmargin-top: 10px;\n\t}\n.leaflet-bottom .leaflet-control {\n\tmargin-bottom: 10px;\n\t}\n.leaflet-left .leaflet-control {\n\tmargin-left: 10px;\n\t}\n.leaflet-right .leaflet-control {\n\tmargin-right: 10px;\n\t}\n\n\n/* zoom and fade animations */\n\n.leaflet-fade-anim .leaflet-popup {\n\topacity: 0;\n\t-webkit-transition: opacity 0.2s linear;\n\t   -moz-transition: opacity 0.2s linear;\n\t        transition: opacity 0.2s linear;\n\t}\n.leaflet-fade-anim .leaflet-map-pane .leaflet-popup {\n\topacity: 1;\n\t}\n.leaflet-zoom-animated {\n\t-webkit-transform-origin: 0 0;\n\t    -ms-transform-origin: 0 0;\n\t        transform-origin: 0 0;\n\t}\nsvg.leaflet-zoom-animated {\n\twill-change: transform;\n}\n\n.leaflet-zoom-anim .leaflet-zoom-animated {\n\t-webkit-transition: -webkit-transform 0.25s cubic-bezier(0,0,0.25,1);\n\t   -moz-transition:    -moz-transform 0.25s cubic-bezier(0,0,0.25,1);\n\t        transition:         transform 0.25s cubic-bezier(0,0,0.25,1);\n\t}\n.leaflet-zoom-anim .leaflet-tile,\n.leaflet-pan-anim .leaflet-tile {\n\t-webkit-transition: none;\n\t   -moz-transition: none;\n\t        transition: none;\n\t}\n\n.leaflet-zoom-anim .leaflet-zoom-hide {\n\tvisibility: hidden;\n\t}\n\n\n/* cursors */\n\n.leaflet-interactive {\n\tcursor: pointer;\n\t}\n.leaflet-grab {\n\tcursor: -webkit-grab;\n\tcursor:    -moz-grab;\n\tcursor:         grab;\n\t}\n.leaflet-crosshair,\n.leaflet-crosshair .leaflet-interactive {\n\tcursor: crosshair;\n\t}\n.leaflet-popup-pane,\n.leaflet-control {\n\tcursor: auto;\n\t}\n.leaflet-dragging .leaflet-grab,\n.leaflet-dragging .leaflet-grab .leaflet-interactive,\n.leaflet-dragging .leaflet-marker-draggable {\n\tcursor: move;\n\tcursor: -webkit-grabbing;\n\tcursor:    -moz-grabbing;\n\tcursor:         grabbing;\n\t}\n\n/* marker & overlays interactivity */\n.leaflet-marker-icon,\n.leaflet-marker-shadow,\n.leaflet-image-layer,\n.leaflet-pane > svg path,\n.leaflet-tile-container {\n\tpointer-events: none;\n\t}\n\n.leaflet-marker-icon.leaflet-interactive,\n.leaflet-image-layer.leaflet-interactive,\n.leaflet-pane > svg path.leaflet-interactive,\nsvg.leaflet-image-layer.leaflet-interactive path {\n\tpointer-events: visiblePainted; /* IE 9-10 doesn\'t have auto */\n\tpointer-events: auto;\n\t}\n\n/* visual tweaks */\n\n.leaflet-container {\n\tbackground: #ddd;\n\toutline-offset: 1px;\n\t}\n.leaflet-container a {\n\tcolor: #0078A8;\n\t}\n.leaflet-zoom-box {\n\tborder: 2px dotted #38f;\n\tbackground: rgba(255,255,255,0.5);\n\t}\n\n\n/* general typography */\n.leaflet-container {\n\tfont-family: "Helvetica Neue", Arial, Helvetica, sans-serif;\n\tfont-size: 12px;\n\tfont-size: 0.75rem;\n\tline-height: 1.5;\n\t}\n\n\n/* general toolbar styles */\n\n.leaflet-bar {\n\tbox-shadow: 0 1px 5px rgba(0,0,0,0.65);\n\tborder-radius: 4px;\n\t}\n.leaflet-bar a {\n\tbackground-color: #fff;\n\tborder-bottom: 1px solid #ccc;\n\twidth: 26px;\n\theight: 26px;\n\tline-height: 26px;\n\tdisplay: block;\n\ttext-align: center;\n\ttext-decoration: none;\n\tcolor: black;\n\t}\n.leaflet-bar a,\n.leaflet-control-layers-toggle {\n\tbackground-position: 50% 50%;\n\tbackground-repeat: no-repeat;\n\tdisplay: block;\n\t}\n.leaflet-bar a:hover,\n.leaflet-bar a:focus {\n\tbackground-color: #f4f4f4;\n\t}\n.leaflet-bar a:first-child {\n\tborder-top-left-radius: 4px;\n\tborder-top-right-radius: 4px;\n\t}\n.leaflet-bar a:last-child {\n\tborder-bottom-left-radius: 4px;\n\tborder-bottom-right-radius: 4px;\n\tborder-bottom: none;\n\t}\n.leaflet-bar a.leaflet-disabled {\n\tcursor: default;\n\tbackground-color: #f4f4f4;\n\tcolor: #bbb;\n\t}\n\n.leaflet-touch .leaflet-bar a {\n\twidth: 30px;\n\theight: 30px;\n\tline-height: 30px;\n\t}\n.leaflet-touch .leaflet-bar a:first-child {\n\tborder-top-left-radius: 2px;\n\tborder-top-right-radius: 2px;\n\t}\n.leaflet-touch .leaflet-bar a:last-child {\n\tborder-bottom-left-radius: 2px;\n\tborder-bottom-right-radius: 2px;\n\t}\n\n/* zoom control */\n\n.leaflet-control-zoom-in,\n.leaflet-control-zoom-out {\n\tfont: bold 18px \'Lucida Console\', Monaco, monospace;\n\ttext-indent: 1px;\n\t}\n\n.leaflet-touch .leaflet-control-zoom-in, .leaflet-touch .leaflet-control-zoom-out  {\n\tfont-size: 22px;\n\t}\n\n\n/* layers control */\n\n.leaflet-control-layers {\n\tbox-shadow: 0 1px 5px rgba(0,0,0,0.4);\n\tbackground: #fff;\n\tborder-radius: 5px;\n\t}\n.leaflet-control-layers-toggle {\n\tbackground-image: url(images/layers.png);\n\twidth: 36px;\n\theight: 36px;\n\t}\n.leaflet-retina .leaflet-control-layers-toggle {\n\tbackground-image: url(images/layers-2x.png);\n\tbackground-size: 26px 26px;\n\t}\n.leaflet-touch .leaflet-control-layers-toggle {\n\twidth: 44px;\n\theight: 44px;\n\t}\n.leaflet-control-layers .leaflet-control-layers-list,\n.leaflet-control-layers-expanded .leaflet-control-layers-toggle {\n\tdisplay: none;\n\t}\n.leaflet-control-layers-expanded .leaflet-control-layers-list {\n\tdisplay: block;\n\tposition: relative;\n\t}\n.leaflet-control-layers-expanded {\n\tpadding: 6px 10px 6px 6px;\n\tcolor: #333;\n\tbackground: #fff;\n\t}\n.leaflet-control-layers-scrollbar {\n\toverflow-y: scroll;\n\toverflow-x: hidden;\n\tpadding-right: 5px;\n\t}\n.leaflet-control-layers-selector {\n\tmargin-top: 2px;\n\tposition: relative;\n\ttop: 1px;\n\t}\n.leaflet-control-layers label {\n\tdisplay: block;\n\tfont-size: 13px;\n\tfont-size: 1.08333em;\n\t}\n.leaflet-control-layers-separator {\n\theight: 0;\n\tborder-top: 1px solid #ddd;\n\tmargin: 5px -10px 5px -6px;\n\t}\n\n/* Default icon URLs */\n.leaflet-default-icon-path { /* used only in path-guessing heuristic, see L.Icon.Default */\n\tbackground-image: url(images/marker-icon.png);\n\t}\n\n\n/* attribution and scale controls */\n\n.leaflet-container .leaflet-control-attribution {\n\tbackground: #fff;\n\tbackground: rgba(255, 255, 255, 0.8);\n\tmargin: 0;\n\t}\n.leaflet-control-attribution,\n.leaflet-control-scale-line {\n\tpadding: 0 5px;\n\tcolor: #333;\n\tline-height: 1.4;\n\t}\n.leaflet-control-attribution a {\n\ttext-decoration: none;\n\t}\n.leaflet-control-attribution a:hover,\n.leaflet-control-attribution a:focus {\n\ttext-decoration: underline;\n\t}\n.leaflet-attribution-flag {\n\tdisplay: inline !important;\n\tvertical-align: baseline !important;\n\twidth: 1em;\n\theight: 0.6669em;\n\t}\n.leaflet-left .leaflet-control-scale {\n\tmargin-left: 5px;\n\t}\n.leaflet-bottom .leaflet-control-scale {\n\tmargin-bottom: 5px;\n\t}\n.leaflet-control-scale-line {\n\tborder: 2px solid #777;\n\tborder-top: none;\n\tline-height: 1.1;\n\tpadding: 2px 5px 1px;\n\twhite-space: nowrap;\n\t-moz-box-sizing: border-box;\n\t     box-sizing: border-box;\n\tbackground: rgba(255, 255, 255, 0.8);\n\ttext-shadow: 1px 1px #fff;\n\t}\n.leaflet-control-scale-line:not(:first-child) {\n\tborder-top: 2px solid #777;\n\tborder-bottom: none;\n\tmargin-top: -2px;\n\t}\n.leaflet-control-scale-line:not(:first-child):not(:last-child) {\n\tborder-bottom: 2px solid #777;\n\t}\n\n.leaflet-touch .leaflet-control-attribution,\n.leaflet-touch .leaflet-control-layers,\n.leaflet-touch .leaflet-bar {\n\tbox-shadow: none;\n\t}\n.leaflet-touch .leaflet-control-layers,\n.leaflet-touch .leaflet-bar {\n\tborder: 2px solid rgba(0,0,0,0.2);\n\tbackground-clip: padding-box;\n\t}\n\n\n/* popup */\n\n.leaflet-popup {\n\tposition: absolute;\n\ttext-align: center;\n\tmargin-bottom: 20px;\n\t}\n.leaflet-popup-content-wrapper {\n\tpadding: 1px;\n\ttext-align: left;\n\tborder-radius: 12px;\n\t}\n.leaflet-popup-content {\n\tmargin: 13px 24px 13px 20px;\n\tline-height: 1.3;\n\tfont-size: 13px;\n\tfont-size: 1.08333em;\n\tmin-height: 1px;\n\t}\n.leaflet-popup-content p {\n\tmargin: 17px 0;\n\tmargin: 1.3em 0;\n\t}\n.leaflet-popup-tip-container {\n\twidth: 40px;\n\theight: 20px;\n\tposition: absolute;\n\tleft: 50%;\n\tmargin-top: -1px;\n\tmargin-left: -20px;\n\toverflow: hidden;\n\tpointer-events: none;\n\t}\n.leaflet-popup-tip {\n\twidth: 17px;\n\theight: 17px;\n\tpadding: 1px;\n\n\tmargin: -10px auto 0;\n\tpointer-events: auto;\n\n\t-webkit-transform: rotate(45deg);\n\t   -moz-transform: rotate(45deg);\n\t    -ms-transform: rotate(45deg);\n\t        transform: rotate(45deg);\n\t}\n.leaflet-popup-content-wrapper,\n.leaflet-popup-tip {\n\tbackground: white;\n\tcolor: #333;\n\tbox-shadow: 0 3px 14px rgba(0,0,0,0.4);\n\t}\n.leaflet-container a.leaflet-popup-close-button {\n\tposition: absolute;\n\ttop: 0;\n\tright: 0;\n\tborder: none;\n\ttext-align: center;\n\twidth: 24px;\n\theight: 24px;\n\tfont: 16px/24px Tahoma, Verdana, sans-serif;\n\tcolor: #757575;\n\ttext-decoration: none;\n\tbackground: transparent;\n\t}\n.leaflet-container a.leaflet-popup-close-button:hover,\n.leaflet-container a.leaflet-popup-close-button:focus {\n\tcolor: #585858;\n\t}\n.leaflet-popup-scrolled {\n\toverflow: auto;\n\t}\n\n.leaflet-oldie .leaflet-popup-content-wrapper {\n\t-ms-zoom: 1;\n\t}\n.leaflet-oldie .leaflet-popup-tip {\n\twidth: 24px;\n\tmargin: 0 auto;\n\n\t-ms-filter: "progid:DXImageTransform.Microsoft.Matrix(M11=0.70710678, M12=0.70710678, M21=-0.70710678, M22=0.70710678)";\n\tfilter: progid:DXImageTransform.Microsoft.Matrix(M11=0.70710678, M12=0.70710678, M21=-0.70710678, M22=0.70710678);\n\t}\n\n.leaflet-oldie .leaflet-control-zoom,\n.leaflet-oldie .leaflet-control-layers,\n.leaflet-oldie .leaflet-popup-content-wrapper,\n.leaflet-oldie .leaflet-popup-tip {\n\tborder: 1px solid #999;\n\t}\n\n\n/* div icon */\n\n.leaflet-div-icon {\n\tbackground: #fff;\n\tborder: 1px solid #666;\n\t}\n\n\n/* Tooltip */\n/* Base styles for the element that has a tooltip */\n.leaflet-tooltip {\n\tposition: absolute;\n\tpadding: 6px;\n\tbackground-color: #fff;\n\tborder: 1px solid #fff;\n\tborder-radius: 3px;\n\tcolor: #222;\n\twhite-space: nowrap;\n\t-webkit-user-select: none;\n\t-moz-user-select: none;\n\t-ms-user-select: none;\n\tuser-select: none;\n\tpointer-events: none;\n\tbox-shadow: 0 1px 3px rgba(0,0,0,0.4);\n\t}\n.leaflet-tooltip.leaflet-interactive {\n\tcursor: pointer;\n\tpointer-events: auto;\n\t}\n.leaflet-tooltip-top:before,\n.leaflet-tooltip-bottom:before,\n.leaflet-tooltip-left:before,\n.leaflet-tooltip-right:before {\n\tposition: absolute;\n\tpointer-events: none;\n\tborder: 6px solid transparent;\n\tbackground: transparent;\n\tcontent: "";\n\t}\n\n/* Directions */\n\n.leaflet-tooltip-bottom {\n\tmargin-top: 6px;\n}\n.leaflet-tooltip-top {\n\tmargin-top: -6px;\n}\n.leaflet-tooltip-bottom:before,\n.leaflet-tooltip-top:before {\n\tleft: 50%;\n\tmargin-left: -6px;\n\t}\n.leaflet-tooltip-top:before {\n\tbottom: 0;\n\tmargin-bottom: -12px;\n\tborder-top-color: #fff;\n\t}\n.leaflet-tooltip-bottom:before {\n\ttop: 0;\n\tmargin-top: -12px;\n\tmargin-left: -6px;\n\tborder-bottom-color: #fff;\n\t}\n.leaflet-tooltip-left {\n\tmargin-left: -6px;\n}\n.leaflet-tooltip-right {\n\tmargin-left: 6px;\n}\n.leaflet-tooltip-left:before,\n.leaflet-tooltip-right:before {\n\ttop: 50%;\n\tmargin-top: -6px;\n\t}\n.leaflet-tooltip-left:before {\n\tright: 0;\n\tmargin-right: -12px;\n\tborder-left-color: #fff;\n\t}\n.leaflet-tooltip-right:before {\n\tleft: 0;\n\tmargin-left: -12px;\n\tborder-right-color: #fff;\n\t}\n\n/* Printing */\n\n@media print {\n\t/* Prevent printers from removing background-images of controls. */\n\t.leaflet-control {\n\t\t-webkit-print-color-adjust: exact;\n\t\tprint-color-adjust: exact;\n\t\t}\n\t}\n</style>\n<h1>Visor de viajes matrices telefonía<span>(big data)</span></h1>\n<div class="shell">\n  <nav class="side" aria-label="Vistas de resultados"><strong>Viajes diarios</strong>\n    <button class="nav active" data-view="viajes">Viajes</button><button class="nav" data-view="distancia">Distancia</button>\n    <button class="nav" data-view="hora">Hora</button><button class="nav" data-view="modo">Modo</button>\n    <button class="nav" data-view="motivo">Motivo</button><button class="nav" data-view="nse">Estrato</button>\n    <button class="nav" data-view="regional">OD regional</button><button class="nav" data-view="localidades">OD localidades</button><button class="nav" data-view="zat">OD ZAT</button>\n  </nav>\n  <main class="main">\n    <div class="filters"><div class="top">\n      <div class="field" data-key="dia"><label class="label">Día</label><button class="trigger" aria-expanded="false"><span class="trigger-text">Laboral</span><i class="chevron"></i></button><div class="popup"><div class="searchbox"><input class="search" placeholder="Buscar" aria-label="Buscar día"></div><div class="choices"></div><div class="popup-foot">Años: 2019, 2023 y 2025</div></div></div>\n      <div class="field" data-key="modo"><label class="label">Modo</label><button class="trigger" aria-expanded="false"><span class="trigger-text">Todos</span><i class="chevron"></i></button><div class="popup"><div class="searchbox"><input class="search" placeholder="Buscar" aria-label="Buscar modo"></div><div class="choices"></div></div><div class="hint">Grupo › Modo</div></div>\n      <div class="field" data-key="motivo"><label class="label">Motivo del viaje</label><button class="trigger" aria-expanded="false"><span class="trigger-text">Todos</span><i class="chevron"></i></button><div class="popup"><div class="searchbox"><input class="search" placeholder="Buscar" aria-label="Buscar motivo"></div><div class="choices"></div></div></div>\n      <div class="field"><label class="label">Hora de inicio de viaje</label><div class="hours"><input type="number" min="0" max="23" value="0" data-hour="0" aria-label="Hora mínima"><input type="number" min="0" max="23" value="23" data-hour="1" aria-label="Hora máxima"></div><div class="ranges"><div class="rail"></div><div class="rail-active"></div><input type="range" min="0" max="23" value="0" data-hour="0" aria-label="Deslizar hora mínima"><input type="range" min="0" max="23" value="23" data-hour="1" aria-label="Deslizar hora máxima"></div></div>\n    </div><div class="bottom">\n      <div class="field" data-key="residencia"><label class="label">Municipio/Localidad hogar</label><button class="trigger" aria-expanded="false"><span class="trigger-text">Todas</span><i class="chevron"></i></button><div class="popup"><div class="searchbox"><input class="search" placeholder="Buscar código" aria-label="Buscar ZAT hogar"></div><div class="choices"></div></div></div>\n      <div class="field" data-key="origen"><label class="label">ZAT origen</label><button class="trigger" aria-expanded="false"><span class="trigger-text">Todas</span><i class="chevron"></i></button><div class="popup"><div class="searchbox"><input class="search" placeholder="Buscar código" aria-label="Buscar ZAT origen"></div><div class="choices"></div></div></div>\n      <div class="field" data-key="destino"><label class="label">ZAT destino</label><button class="trigger" aria-expanded="false"><span class="trigger-text">Todas</span><i class="chevron"></i></button><div class="popup"><div class="searchbox"><input class="search" placeholder="Buscar código" aria-label="Buscar ZAT destino"></div><div class="choices"></div></div></div>\n    </div></div>\n    <div class="actions"><button class="btn primary" id="v-aplicar">Aplicar filtros</button><button class="btn" id="v-limpiar">Restablecer</button><span class="status" role="status"></span></div>\n    <div class="errors" role="alert"></div>\n    <section class="results normal-results"><div class="result-head"><h2></h2><button class="btn" id="v-csv">Descargar tabla</button></div><div class="note view-note"></div><div class="chart"></div><div class="legend"></div><div class="table-wrap"></div><div class="foot">Los resultados excluyen los viajes identificados como profesionales.</div></section>\n\n    <section class="results od-results" hidden>\n      <div class="result-head"><h2 id="od-title"></h2><button class="btn" id="od-csv">Descargar tablas</button></div>\n      <fieldset id="year-selector" class="year-selector" hidden><legend>Año</legend><label><input type="radio" name="od-year" value="2019">2019</label><label><input type="radio" name="od-year" value="2023">2023</label><label><input type="radio" name="od-year" value="2025" checked>2025</label></fieldset>\n      <div id="od-maps"></div>\n      <section id="od-pairs" hidden><div class="result-head"><h3>Viajes entre las ZAT seleccionadas</h3><button class="btn" id="pairs-csv">Descargar relaciones</button></div><strong id="pair-total"></strong><div id="pair-table"></div><p id="pair-note" class="note"></p></section>\n      <details class="data-details"><summary>Observaciones de los datos</summary><div id="od-diagnostics"></div></details>\n      <div class="foot">Los resultados excluyen los viajes identificados como profesionales.</div>\n    </section>\n  </main>\n</div>\n<script>\n/*\nBSD 2-Clause License\n\nCopyright (c) 2010-2023, Volodymyr Agafonkin\nCopyright (c) 2010-2011, CloudMade\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n\n*/\n/* @preserve\n * Leaflet 1.9.4, a JS library for interactive maps. https://leafletjs.com\n * (c) 2010-2023 Vladimir Agafonkin, (c) 2010-2011 CloudMade\n */\n!function(t,e){"object"==typeof exports&&"undefined"!=typeof module?e(exports):"function"==typeof define&&define.amd?define(["exports"],e):e((t="undefined"!=typeof globalThis?globalThis:t||self).leaflet={})}(this,function(t){"use strict";function l(t){for(var e,i,n=1,o=arguments.length;n<o;n++)for(e in i=arguments[n])t[e]=i[e];return t}var R=Object.create||function(t){return N.prototype=t,new N};function N(){}function a(t,e){var i,n=Array.prototype.slice;return t.bind?t.bind.apply(t,n.call(arguments,1)):(i=n.call(arguments,2),function(){return t.apply(e,i.length?i.concat(n.call(arguments)):arguments)})}var D=0;function h(t){return"_leaflet_id"in t||(t._leaflet_id=++D),t._leaflet_id}function j(t,e,i){var n,o,s=function(){n=!1,o&&(r.apply(i,o),o=!1)},r=function(){n?o=arguments:(t.apply(i,arguments),setTimeout(s,e),n=!0)};return r}function H(t,e,i){var n=e[1],e=e[0],o=n-e;return t===n&&i?t:((t-e)%o+o)%o+e}function u(){return!1}function i(t,e){return!1===e?t:(e=Math.pow(10,void 0===e?6:e),Math.round(t*e)/e)}function W(t){return t.trim?t.trim():t.replace(/^\\s+|\\s+$/g,"")}function F(t){return W(t).split(/\\s+/)}function c(t,e){for(var i in Object.prototype.hasOwnProperty.call(t,"options")||(t.options=t.options?R(t.options):{}),e)t.options[i]=e[i];return t.options}function U(t,e,i){var n,o=[];for(n in t)o.push(encodeURIComponent(i?n.toUpperCase():n)+"="+encodeURIComponent(t[n]));return(e&&-1!==e.indexOf("?")?"&":"?")+o.join("&")}var V=/\\{ *([\\w_ -]+) *\\}/g;function q(t,i){return t.replace(V,function(t,e){e=i[e];if(void 0===e)throw new Error("No value provided for variable "+t);return e="function"==typeof e?e(i):e})}var d=Array.isArray||function(t){return"[object Array]"===Object.prototype.toString.call(t)};function G(t,e){for(var i=0;i<t.length;i++)if(t[i]===e)return i;return-1}var K="data:image/gif;base64,R0lGODlhAQABAAD/ACwAAAAAAQABAAACADs=";function Y(t){return window["webkit"+t]||window["moz"+t]||window["ms"+t]}var X=0;function J(t){var e=+new Date,i=Math.max(0,16-(e-X));return X=e+i,window.setTimeout(t,i)}var $=window.requestAnimationFrame||Y("RequestAnimationFrame")||J,Q=window.cancelAnimationFrame||Y("CancelAnimationFrame")||Y("CancelRequestAnimationFrame")||function(t){window.clearTimeout(t)};function x(t,e,i){if(!i||$!==J)return $.call(window,a(t,e));t.call(e)}function r(t){t&&Q.call(window,t)}var tt={__proto__:null,extend:l,create:R,bind:a,get lastId(){return D},stamp:h,throttle:j,wrapNum:H,falseFn:u,formatNum:i,trim:W,splitWords:F,setOptions:c,getParamString:U,template:q,isArray:d,indexOf:G,emptyImageUrl:K,requestFn:$,cancelFn:Q,requestAnimFrame:x,cancelAnimFrame:r};function et(){}et.extend=function(t){function e(){c(this),this.initialize&&this.initialize.apply(this,arguments),this.callInitHooks()}var i,n=e.__super__=this.prototype,o=R(n);for(i in(o.constructor=e).prototype=o,this)Object.prototype.hasOwnProperty.call(this,i)&&"prototype"!==i&&"__super__"!==i&&(e[i]=this[i]);if(t.statics&&l(e,t.statics),t.includes){var s=t.includes;if("undefined"!=typeof L&&L&&L.Mixin){s=d(s)?s:[s];for(var r=0;r<s.length;r++)s[r]===L.Mixin.Events&&console.warn("Deprecated include of L.Mixin.Events: this property will be removed in future releases, please inherit from L.Evented instead.",(new Error).stack)}l.apply(null,[o].concat(t.includes))}return l(o,t),delete o.statics,delete o.includes,o.options&&(o.options=n.options?R(n.options):{},l(o.options,t.options)),o._initHooks=[],o.callInitHooks=function(){if(!this._initHooksCalled){n.callInitHooks&&n.callInitHooks.call(this),this._initHooksCalled=!0;for(var t=0,e=o._initHooks.length;t<e;t++)o._initHooks[t].call(this)}},e},et.include=function(t){var e=this.prototype.options;return l(this.prototype,t),t.options&&(this.prototype.options=e,this.mergeOptions(t.options)),this},et.mergeOptions=function(t){return l(this.prototype.options,t),this},et.addInitHook=function(t){var e=Array.prototype.slice.call(arguments,1),i="function"==typeof t?t:function(){this[t].apply(this,e)};return this.prototype._initHooks=this.prototype._initHooks||[],this.prototype._initHooks.push(i),this};var e={on:function(t,e,i){if("object"==typeof t)for(var n in t)this._on(n,t[n],e);else for(var o=0,s=(t=F(t)).length;o<s;o++)this._on(t[o],e,i);return this},off:function(t,e,i){if(arguments.length)if("object"==typeof t)for(var n in t)this._off(n,t[n],e);else{t=F(t);for(var o=1===arguments.length,s=0,r=t.length;s<r;s++)o?this._off(t[s]):this._off(t[s],e,i)}else delete this._events;return this},_on:function(t,e,i,n){"function"!=typeof e?console.warn("wrong listener type: "+typeof e):!1===this._listens(t,e,i)&&(e={fn:e,ctx:i=i===this?void 0:i},n&&(e.once=!0),this._events=this._events||{},this._events[t]=this._events[t]||[],this._events[t].push(e))},_off:function(t,e,i){var n,o,s;if(this._events&&(n=this._events[t]))if(1===arguments.length){if(this._firingCount)for(o=0,s=n.length;o<s;o++)n[o].fn=u;delete this._events[t]}else"function"!=typeof e?console.warn("wrong listener type: "+typeof e):!1!==(e=this._listens(t,e,i))&&(i=n[e],this._firingCount&&(i.fn=u,this._events[t]=n=n.slice()),n.splice(e,1))},fire:function(t,e,i){if(this.listens(t,i)){var n=l({},e,{type:t,target:this,sourceTarget:e&&e.sourceTarget||this});if(this._events){var o=this._events[t];if(o){this._firingCount=this._firingCount+1||1;for(var s=0,r=o.length;s<r;s++){var a=o[s],h=a.fn;a.once&&this.off(t,h,a.ctx),h.call(a.ctx||this,n)}this._firingCount--}}i&&this._propagateEvent(n)}return this},listens:function(t,e,i,n){"string"!=typeof t&&console.warn(\'"string" type argument expected\');var o=e,s=("function"!=typeof e&&(n=!!e,i=o=void 0),this._events&&this._events[t]);if(s&&s.length&&!1!==this._listens(t,o,i))return!0;if(n)for(var r in this._eventParents)if(this._eventParents[r].listens(t,e,i,n))return!0;return!1},_listens:function(t,e,i){if(this._events){var n=this._events[t]||[];if(!e)return!!n.length;i===this&&(i=void 0);for(var o=0,s=n.length;o<s;o++)if(n[o].fn===e&&n[o].ctx===i)return o}return!1},once:function(t,e,i){if("object"==typeof t)for(var n in t)this._on(n,t[n],e,!0);else for(var o=0,s=(t=F(t)).length;o<s;o++)this._on(t[o],e,i,!0);return this},addEventParent:function(t){return this._eventParents=this._eventParents||{},this._eventParents[h(t)]=t,this},removeEventParent:function(t){return this._eventParents&&delete this._eventParents[h(t)],this},_propagateEvent:function(t){for(var e in this._eventParents)this._eventParents[e].fire(t.type,l({layer:t.target,propagatedFrom:t.target},t),!0)}},it=(e.addEventListener=e.on,e.removeEventListener=e.clearAllEventListeners=e.off,e.addOneTimeEventListener=e.once,e.fireEvent=e.fire,e.hasEventListeners=e.listens,et.extend(e));function p(t,e,i){this.x=i?Math.round(t):t,this.y=i?Math.round(e):e}var nt=Math.trunc||function(t){return 0<t?Math.floor(t):Math.ceil(t)};function m(t,e,i){return t instanceof p?t:d(t)?new p(t[0],t[1]):null==t?t:"object"==typeof t&&"x"in t&&"y"in t?new p(t.x,t.y):new p(t,e,i)}function f(t,e){if(t)for(var i=e?[t,e]:t,n=0,o=i.length;n<o;n++)this.extend(i[n])}function _(t,e){return!t||t instanceof f?t:new f(t,e)}function s(t,e){if(t)for(var i=e?[t,e]:t,n=0,o=i.length;n<o;n++)this.extend(i[n])}function g(t,e){return t instanceof s?t:new s(t,e)}function v(t,e,i){if(isNaN(t)||isNaN(e))throw new Error("Invalid LatLng object: ("+t+", "+e+")");this.lat=+t,this.lng=+e,void 0!==i&&(this.alt=+i)}function w(t,e,i){return t instanceof v?t:d(t)&&"object"!=typeof t[0]?3===t.length?new v(t[0],t[1],t[2]):2===t.length?new v(t[0],t[1]):null:null==t?t:"object"==typeof t&&"lat"in t?new v(t.lat,"lng"in t?t.lng:t.lon,t.alt):void 0===e?null:new v(t,e,i)}p.prototype={clone:function(){return new p(this.x,this.y)},add:function(t){return this.clone()._add(m(t))},_add:function(t){return this.x+=t.x,this.y+=t.y,this},subtract:function(t){return this.clone()._subtract(m(t))},_subtract:function(t){return this.x-=t.x,this.y-=t.y,this},divideBy:function(t){return this.clone()._divideBy(t)},_divideBy:function(t){return this.x/=t,this.y/=t,this},multiplyBy:function(t){return this.clone()._multiplyBy(t)},_multiplyBy:function(t){return this.x*=t,this.y*=t,this},scaleBy:function(t){return new p(this.x*t.x,this.y*t.y)},unscaleBy:function(t){return new p(this.x/t.x,this.y/t.y)},round:function(){return this.clone()._round()},_round:function(){return this.x=Math.round(this.x),this.y=Math.round(this.y),this},floor:function(){return this.clone()._floor()},_floor:function(){return this.x=Math.floor(this.x),this.y=Math.floor(this.y),this},ceil:function(){return this.clone()._ceil()},_ceil:function(){return this.x=Math.ceil(this.x),this.y=Math.ceil(this.y),this},trunc:function(){return this.clone()._trunc()},_trunc:function(){return this.x=nt(this.x),this.y=nt(this.y),this},distanceTo:function(t){var e=(t=m(t)).x-this.x,t=t.y-this.y;return Math.sqrt(e*e+t*t)},equals:function(t){return(t=m(t)).x===this.x&&t.y===this.y},contains:function(t){return t=m(t),Math.abs(t.x)<=Math.abs(this.x)&&Math.abs(t.y)<=Math.abs(this.y)},toString:function(){return"Point("+i(this.x)+", "+i(this.y)+")"}},f.prototype={extend:function(t){var e,i;if(t){if(t instanceof p||"number"==typeof t[0]||"x"in t)e=i=m(t);else if(e=(t=_(t)).min,i=t.max,!e||!i)return this;this.min||this.max?(this.min.x=Math.min(e.x,this.min.x),this.max.x=Math.max(i.x,this.max.x),this.min.y=Math.min(e.y,this.min.y),this.max.y=Math.max(i.y,this.max.y)):(this.min=e.clone(),this.max=i.clone())}return this},getCenter:function(t){return m((this.min.x+this.max.x)/2,(this.min.y+this.max.y)/2,t)},getBottomLeft:function(){return m(this.min.x,this.max.y)},getTopRight:function(){return m(this.max.x,this.min.y)},getTopLeft:function(){return this.min},getBottomRight:function(){return this.max},getSize:function(){return this.max.subtract(this.min)},contains:function(t){var e,i;return(t=("number"==typeof t[0]||t instanceof p?m:_)(t))instanceof f?(e=t.min,i=t.max):e=i=t,e.x>=this.min.x&&i.x<=this.max.x&&e.y>=this.min.y&&i.y<=this.max.y},intersects:function(t){t=_(t);var e=this.min,i=this.max,n=t.min,t=t.max,o=t.x>=e.x&&n.x<=i.x,t=t.y>=e.y&&n.y<=i.y;return o&&t},overlaps:function(t){t=_(t);var e=this.min,i=this.max,n=t.min,t=t.max,o=t.x>e.x&&n.x<i.x,t=t.y>e.y&&n.y<i.y;return o&&t},isValid:function(){return!(!this.min||!this.max)},pad:function(t){var e=this.min,i=this.max,n=Math.abs(e.x-i.x)*t,t=Math.abs(e.y-i.y)*t;return _(m(e.x-n,e.y-t),m(i.x+n,i.y+t))},equals:function(t){return!!t&&(t=_(t),this.min.equals(t.getTopLeft())&&this.max.equals(t.getBottomRight()))}},s.prototype={extend:function(t){var e,i,n=this._southWest,o=this._northEast;if(t instanceof v)i=e=t;else{if(!(t instanceof s))return t?this.extend(w(t)||g(t)):this;if(e=t._southWest,i=t._northEast,!e||!i)return this}return n||o?(n.lat=Math.min(e.lat,n.lat),n.lng=Math.min(e.lng,n.lng),o.lat=Math.max(i.lat,o.lat),o.lng=Math.max(i.lng,o.lng)):(this._southWest=new v(e.lat,e.lng),this._northEast=new v(i.lat,i.lng)),this},pad:function(t){var e=this._southWest,i=this._northEast,n=Math.abs(e.lat-i.lat)*t,t=Math.abs(e.lng-i.lng)*t;return new s(new v(e.lat-n,e.lng-t),new v(i.lat+n,i.lng+t))},getCenter:function(){return new v((this._southWest.lat+this._northEast.lat)/2,(this._southWest.lng+this._northEast.lng)/2)},getSouthWest:function(){return this._southWest},getNorthEast:function(){return this._northEast},getNorthWest:function(){return new v(this.getNorth(),this.getWest())},getSouthEast:function(){return new v(this.getSouth(),this.getEast())},getWest:function(){return this._southWest.lng},getSouth:function(){return this._southWest.lat},getEast:function(){return this._northEast.lng},getNorth:function(){return this._northEast.lat},contains:function(t){t=("number"==typeof t[0]||t instanceof v||"lat"in t?w:g)(t);var e,i,n=this._southWest,o=this._northEast;return t instanceof s?(e=t.getSouthWest(),i=t.getNorthEast()):e=i=t,e.lat>=n.lat&&i.lat<=o.lat&&e.lng>=n.lng&&i.lng<=o.lng},intersects:function(t){t=g(t);var e=this._southWest,i=this._northEast,n=t.getSouthWest(),t=t.getNorthEast(),o=t.lat>=e.lat&&n.lat<=i.lat,t=t.lng>=e.lng&&n.lng<=i.lng;return o&&t},overlaps:function(t){t=g(t);var e=this._southWest,i=this._northEast,n=t.getSouthWest(),t=t.getNorthEast(),o=t.lat>e.lat&&n.lat<i.lat,t=t.lng>e.lng&&n.lng<i.lng;return o&&t},toBBoxString:function(){return[this.getWest(),this.getSouth(),this.getEast(),this.getNorth()].join(",")},equals:function(t,e){return!!t&&(t=g(t),this._southWest.equals(t.getSouthWest(),e)&&this._northEast.equals(t.getNorthEast(),e))},isValid:function(){return!(!this._southWest||!this._northEast)}};var ot={latLngToPoint:function(t,e){t=this.projection.project(t),e=this.scale(e);return this.transformation._transform(t,e)},pointToLatLng:function(t,e){e=this.scale(e),t=this.transformation.untransform(t,e);return this.projection.unproject(t)},project:function(t){return this.projection.project(t)},unproject:function(t){return this.projection.unproject(t)},scale:function(t){return 256*Math.pow(2,t)},zoom:function(t){return Math.log(t/256)/Math.LN2},getProjectedBounds:function(t){var e;return this.infinite?null:(e=this.projection.bounds,t=this.scale(t),new f(this.transformation.transform(e.min,t),this.transformation.transform(e.max,t)))},infinite:!(v.prototype={equals:function(t,e){return!!t&&(t=w(t),Math.max(Math.abs(this.lat-t.lat),Math.abs(this.lng-t.lng))<=(void 0===e?1e-9:e))},toString:function(t){return"LatLng("+i(this.lat,t)+", "+i(this.lng,t)+")"},distanceTo:function(t){return st.distance(this,w(t))},wrap:function(){return st.wrapLatLng(this)},toBounds:function(t){var t=180*t/40075017,e=t/Math.cos(Math.PI/180*this.lat);return g([this.lat-t,this.lng-e],[this.lat+t,this.lng+e])},clone:function(){return new v(this.lat,this.lng,this.alt)}}),wrapLatLng:function(t){var e=this.wrapLng?H(t.lng,this.wrapLng,!0):t.lng;return new v(this.wrapLat?H(t.lat,this.wrapLat,!0):t.lat,e,t.alt)},wrapLatLngBounds:function(t){var e=t.getCenter(),i=this.wrapLatLng(e),n=e.lat-i.lat,e=e.lng-i.lng;return 0==n&&0==e?t:(i=t.getSouthWest(),t=t.getNorthEast(),new s(new v(i.lat-n,i.lng-e),new v(t.lat-n,t.lng-e)))}},st=l({},ot,{wrapLng:[-180,180],R:6371e3,distance:function(t,e){var i=Math.PI/180,n=t.lat*i,o=e.lat*i,s=Math.sin((e.lat-t.lat)*i/2),e=Math.sin((e.lng-t.lng)*i/2),t=s*s+Math.cos(n)*Math.cos(o)*e*e,i=2*Math.atan2(Math.sqrt(t),Math.sqrt(1-t));return this.R*i}}),rt=6378137,rt={R:rt,MAX_LATITUDE:85.0511287798,project:function(t){var e=Math.PI/180,i=this.MAX_LATITUDE,i=Math.max(Math.min(i,t.lat),-i),i=Math.sin(i*e);return new p(this.R*t.lng*e,this.R*Math.log((1+i)/(1-i))/2)},unproject:function(t){var e=180/Math.PI;return new v((2*Math.atan(Math.exp(t.y/this.R))-Math.PI/2)*e,t.x*e/this.R)},bounds:new f([-(rt=rt*Math.PI),-rt],[rt,rt])};function at(t,e,i,n){d(t)?(this._a=t[0],this._b=t[1],this._c=t[2],this._d=t[3]):(this._a=t,this._b=e,this._c=i,this._d=n)}function ht(t,e,i,n){return new at(t,e,i,n)}at.prototype={transform:function(t,e){return this._transform(t.clone(),e)},_transform:function(t,e){return t.x=(e=e||1)*(this._a*t.x+this._b),t.y=e*(this._c*t.y+this._d),t},untransform:function(t,e){return new p((t.x/(e=e||1)-this._b)/this._a,(t.y/e-this._d)/this._c)}};var lt=l({},st,{code:"EPSG:3857",projection:rt,transformation:ht(lt=.5/(Math.PI*rt.R),.5,-lt,.5)}),ut=l({},lt,{code:"EPSG:900913"});function ct(t){return document.createElementNS("http://www.w3.org/2000/svg",t)}function dt(t,e){for(var i,n,o,s,r="",a=0,h=t.length;a<h;a++){for(i=0,n=(o=t[a]).length;i<n;i++)r+=(i?"L":"M")+(s=o[i]).x+" "+s.y;r+=e?b.svg?"z":"x":""}return r||"M0 0"}var _t=document.documentElement.style,pt="ActiveXObject"in window,mt=pt&&!document.addEventListener,n="msLaunchUri"in navigator&&!("documentMode"in document),ft=y("webkit"),gt=y("android"),vt=y("android 2")||y("android 3"),yt=parseInt(/WebKit\\/([0-9]+)|$/.exec(navigator.userAgent)[1],10),yt=gt&&y("Google")&&yt<537&&!("AudioNode"in window),xt=!!window.opera,wt=!n&&y("chrome"),bt=y("gecko")&&!ft&&!xt&&!pt,Pt=!wt&&y("safari"),Lt=y("phantom"),o="OTransition"in _t,Tt=0===navigator.platform.indexOf("Win"),Mt=pt&&"transition"in _t,zt="WebKitCSSMatrix"in window&&"m11"in new window.WebKitCSSMatrix&&!vt,_t="MozPerspective"in _t,Ct=!window.L_DISABLE_3D&&(Mt||zt||_t)&&!o&&!Lt,Zt="undefined"!=typeof orientation||y("mobile"),St=Zt&&ft,Et=Zt&&zt,kt=!window.PointerEvent&&window.MSPointerEvent,Ot=!(!window.PointerEvent&&!kt),At="ontouchstart"in window||!!window.TouchEvent,Bt=!window.L_NO_TOUCH&&(At||Ot),It=Zt&&xt,Rt=Zt&&bt,Nt=1<(window.devicePixelRatio||window.screen.deviceXDPI/window.screen.logicalXDPI),Dt=function(){var t=!1;try{var e=Object.defineProperty({},"passive",{get:function(){t=!0}});window.addEventListener("testPassiveEventSupport",u,e),window.removeEventListener("testPassiveEventSupport",u,e)}catch(t){}return t}(),jt=!!document.createElement("canvas").getContext,Ht=!(!document.createElementNS||!ct("svg").createSVGRect),Wt=!!Ht&&((Wt=document.createElement("div")).innerHTML="<svg/>","http://www.w3.org/2000/svg"===(Wt.firstChild&&Wt.firstChild.namespaceURI));function y(t){return 0<=navigator.userAgent.toLowerCase().indexOf(t)}var b={ie:pt,ielt9:mt,edge:n,webkit:ft,android:gt,android23:vt,androidStock:yt,opera:xt,chrome:wt,gecko:bt,safari:Pt,phantom:Lt,opera12:o,win:Tt,ie3d:Mt,webkit3d:zt,gecko3d:_t,any3d:Ct,mobile:Zt,mobileWebkit:St,mobileWebkit3d:Et,msPointer:kt,pointer:Ot,touch:Bt,touchNative:At,mobileOpera:It,mobileGecko:Rt,retina:Nt,passiveEvents:Dt,canvas:jt,svg:Ht,vml:!Ht&&function(){try{var t=document.createElement("div"),e=(t.innerHTML=\'<v:shape adj="1"/>\',t.firstChild);return e.style.behavior="url(#default#VML)",e&&"object"==typeof e.adj}catch(t){return!1}}(),inlineSvg:Wt,mac:0===navigator.platform.indexOf("Mac"),linux:0===navigator.platform.indexOf("Linux")},Ft=b.msPointer?"MSPointerDown":"pointerdown",Ut=b.msPointer?"MSPointerMove":"pointermove",Vt=b.msPointer?"MSPointerUp":"pointerup",qt=b.msPointer?"MSPointerCancel":"pointercancel",Gt={touchstart:Ft,touchmove:Ut,touchend:Vt,touchcancel:qt},Kt={touchstart:function(t,e){e.MSPOINTER_TYPE_TOUCH&&e.pointerType===e.MSPOINTER_TYPE_TOUCH&&O(e);ee(t,e)},touchmove:ee,touchend:ee,touchcancel:ee},Yt={},Xt=!1;function Jt(t,e,i){return"touchstart"!==e||Xt||(document.addEventListener(Ft,$t,!0),document.addEventListener(Ut,Qt,!0),document.addEventListener(Vt,te,!0),document.addEventListener(qt,te,!0),Xt=!0),Kt[e]?(i=Kt[e].bind(this,i),t.addEventListener(Gt[e],i,!1),i):(console.warn("wrong event specified:",e),u)}function $t(t){Yt[t.pointerId]=t}function Qt(t){Yt[t.pointerId]&&(Yt[t.pointerId]=t)}function te(t){delete Yt[t.pointerId]}function ee(t,e){if(e.pointerType!==(e.MSPOINTER_TYPE_MOUSE||"mouse")){for(var i in e.touches=[],Yt)e.touches.push(Yt[i]);e.changedTouches=[e],t(e)}}var ie=200;function ne(t,i){t.addEventListener("dblclick",i);var n,o=0;function e(t){var e;1!==t.detail?n=t.detail:"mouse"===t.pointerType||t.sourceCapabilities&&!t.sourceCapabilities.firesTouchEvents||((e=Ne(t)).some(function(t){return t instanceof HTMLLabelElement&&t.attributes.for})&&!e.some(function(t){return t instanceof HTMLInputElement||t instanceof HTMLSelectElement})||((e=Date.now())-o<=ie?2===++n&&i(function(t){var e,i,n={};for(i in t)e=t[i],n[i]=e&&e.bind?e.bind(t):e;return(t=n).type="dblclick",n.detail=2,n.isTrusted=!1,n._simulated=!0,n}(t)):n=1,o=e))}return t.addEventListener("click",e),{dblclick:i,simDblclick:e}}var oe,se,re,ae,he,le,ue=we(["transform","webkitTransform","OTransform","MozTransform","msTransform"]),ce=we(["webkitTransition","transition","OTransition","MozTransition","msTransition"]),de="webkitTransition"===ce||"OTransition"===ce?ce+"End":"transitionend";function _e(t){return"string"==typeof t?document.getElementById(t):t}function pe(t,e){var i=t.style[e]||t.currentStyle&&t.currentStyle[e];return"auto"===(i=i&&"auto"!==i||!document.defaultView?i:(t=document.defaultView.getComputedStyle(t,null))?t[e]:null)?null:i}function P(t,e,i){t=document.createElement(t);return t.className=e||"",i&&i.appendChild(t),t}function T(t){var e=t.parentNode;e&&e.removeChild(t)}function me(t){for(;t.firstChild;)t.removeChild(t.firstChild)}function fe(t){var e=t.parentNode;e&&e.lastChild!==t&&e.appendChild(t)}function ge(t){var e=t.parentNode;e&&e.firstChild!==t&&e.insertBefore(t,e.firstChild)}function ve(t,e){return void 0!==t.classList?t.classList.contains(e):0<(t=xe(t)).length&&new RegExp("(^|\\\\s)"+e+"(\\\\s|$)").test(t)}function M(t,e){var i;if(void 0!==t.classList)for(var n=F(e),o=0,s=n.length;o<s;o++)t.classList.add(n[o]);else ve(t,e)||ye(t,((i=xe(t))?i+" ":"")+e)}function z(t,e){void 0!==t.classList?t.classList.remove(e):ye(t,W((" "+xe(t)+" ").replace(" "+e+" "," ")))}function ye(t,e){void 0===t.className.baseVal?t.className=e:t.className.baseVal=e}function xe(t){return void 0===(t=t.correspondingElement?t.correspondingElement:t).className.baseVal?t.className:t.className.baseVal}function C(t,e){if("opacity"in t.style)t.style.opacity=e;else if("filter"in t.style){var i=!1,n="DXImageTransform.Microsoft.Alpha";try{i=t.filters.item(n)}catch(t){if(1===e)return}e=Math.round(100*e),i?(i.Enabled=100!==e,i.Opacity=e):t.style.filter+=" progid:"+n+"(opacity="+e+")"}}function we(t){for(var e=document.documentElement.style,i=0;i<t.length;i++)if(t[i]in e)return t[i];return!1}function be(t,e,i){e=e||new p(0,0);t.style[ue]=(b.ie3d?"translate("+e.x+"px,"+e.y+"px)":"translate3d("+e.x+"px,"+e.y+"px,0)")+(i?" scale("+i+")":"")}function Z(t,e){t._leaflet_pos=e,b.any3d?be(t,e):(t.style.left=e.x+"px",t.style.top=e.y+"px")}function Pe(t){return t._leaflet_pos||new p(0,0)}function Le(){S(window,"dragstart",O)}function Te(){k(window,"dragstart",O)}function Me(t){for(;-1===t.tabIndex;)t=t.parentNode;t.style&&(ze(),le=(he=t).style.outlineStyle,t.style.outlineStyle="none",S(window,"keydown",ze))}function ze(){he&&(he.style.outlineStyle=le,le=he=void 0,k(window,"keydown",ze))}function Ce(t){for(;!((t=t.parentNode).offsetWidth&&t.offsetHeight||t===document.body););return t}function Ze(t){var e=t.getBoundingClientRect();return{x:e.width/t.offsetWidth||1,y:e.height/t.offsetHeight||1,boundingClientRect:e}}ae="onselectstart"in document?(re=function(){S(window,"selectstart",O)},function(){k(window,"selectstart",O)}):(se=we(["userSelect","WebkitUserSelect","OUserSelect","MozUserSelect","msUserSelect"]),re=function(){var t;se&&(t=document.documentElement.style,oe=t[se],t[se]="none")},function(){se&&(document.documentElement.style[se]=oe,oe=void 0)});pt={__proto__:null,TRANSFORM:ue,TRANSITION:ce,TRANSITION_END:de,get:_e,getStyle:pe,create:P,remove:T,empty:me,toFront:fe,toBack:ge,hasClass:ve,addClass:M,removeClass:z,setClass:ye,getClass:xe,setOpacity:C,testProp:we,setTransform:be,setPosition:Z,getPosition:Pe,get disableTextSelection(){return re},get enableTextSelection(){return ae},disableImageDrag:Le,enableImageDrag:Te,preventOutline:Me,restoreOutline:ze,getSizedParentNode:Ce,getScale:Ze};function S(t,e,i,n){if(e&&"object"==typeof e)for(var o in e)ke(t,o,e[o],i);else for(var s=0,r=(e=F(e)).length;s<r;s++)ke(t,e[s],i,n);return this}var E="_leaflet_events";function k(t,e,i,n){if(1===arguments.length)Se(t),delete t[E];else if(e&&"object"==typeof e)for(var o in e)Oe(t,o,e[o],i);else if(e=F(e),2===arguments.length)Se(t,function(t){return-1!==G(e,t)});else for(var s=0,r=e.length;s<r;s++)Oe(t,e[s],i,n);return this}function Se(t,e){for(var i in t[E]){var n=i.split(/\\d/)[0];e&&!e(n)||Oe(t,n,null,null,i)}}var Ee={mouseenter:"mouseover",mouseleave:"mouseout",wheel:!("onwheel"in window)&&"mousewheel"};function ke(e,t,i,n){var o,s,r=t+h(i)+(n?"_"+h(n):"");e[E]&&e[E][r]||(s=o=function(t){return i.call(n||e,t||window.event)},!b.touchNative&&b.pointer&&0===t.indexOf("touch")?o=Jt(e,t,o):b.touch&&"dblclick"===t?o=ne(e,o):"addEventListener"in e?"touchstart"===t||"touchmove"===t||"wheel"===t||"mousewheel"===t?e.addEventListener(Ee[t]||t,o,!!b.passiveEvents&&{passive:!1}):"mouseenter"===t||"mouseleave"===t?e.addEventListener(Ee[t],o=function(t){t=t||window.event,We(e,t)&&s(t)},!1):e.addEventListener(t,s,!1):e.attachEvent("on"+t,o),e[E]=e[E]||{},e[E][r]=o)}function Oe(t,e,i,n,o){o=o||e+h(i)+(n?"_"+h(n):"");var s,r,i=t[E]&&t[E][o];i&&(!b.touchNative&&b.pointer&&0===e.indexOf("touch")?(n=t,r=i,Gt[s=e]?n.removeEventListener(Gt[s],r,!1):console.warn("wrong event specified:",s)):b.touch&&"dblclick"===e?(n=i,(r=t).removeEventListener("dblclick",n.dblclick),r.removeEventListener("click",n.simDblclick)):"removeEventListener"in t?t.removeEventListener(Ee[e]||e,i,!1):t.detachEvent("on"+e,i),t[E][o]=null)}function Ae(t){return t.stopPropagation?t.stopPropagation():t.originalEvent?t.originalEvent._stopped=!0:t.cancelBubble=!0,this}function Be(t){return ke(t,"wheel",Ae),this}function Ie(t){return S(t,"mousedown touchstart dblclick contextmenu",Ae),t._leaflet_disable_click=!0,this}function O(t){return t.preventDefault?t.preventDefault():t.returnValue=!1,this}function Re(t){return O(t),Ae(t),this}function Ne(t){if(t.composedPath)return t.composedPath();for(var e=[],i=t.target;i;)e.push(i),i=i.parentNode;return e}function De(t,e){var i,n;return e?(n=(i=Ze(e)).boundingClientRect,new p((t.clientX-n.left)/i.x-e.clientLeft,(t.clientY-n.top)/i.y-e.clientTop)):new p(t.clientX,t.clientY)}var je=b.linux&&b.chrome?window.devicePixelRatio:b.mac?3*window.devicePixelRatio:0<window.devicePixelRatio?2*window.devicePixelRatio:1;function He(t){return b.edge?t.wheelDeltaY/2:t.deltaY&&0===t.deltaMode?-t.deltaY/je:t.deltaY&&1===t.deltaMode?20*-t.deltaY:t.deltaY&&2===t.deltaMode?60*-t.deltaY:t.deltaX||t.deltaZ?0:t.wheelDelta?(t.wheelDeltaY||t.wheelDelta)/2:t.detail&&Math.abs(t.detail)<32765?20*-t.detail:t.detail?t.detail/-32765*60:0}function We(t,e){var i=e.relatedTarget;if(!i)return!0;try{for(;i&&i!==t;)i=i.parentNode}catch(t){return!1}return i!==t}var mt={__proto__:null,on:S,off:k,stopPropagation:Ae,disableScrollPropagation:Be,disableClickPropagation:Ie,preventDefault:O,stop:Re,getPropagationPath:Ne,getMousePosition:De,getWheelDelta:He,isExternalTarget:We,addListener:S,removeListener:k},Fe=it.extend({run:function(t,e,i,n){this.stop(),this._el=t,this._inProgress=!0,this._duration=i||.25,this._easeOutPower=1/Math.max(n||.5,.2),this._startPos=Pe(t),this._offset=e.subtract(this._startPos),this._startTime=+new Date,this.fire("start"),this._animate()},stop:function(){this._inProgress&&(this._step(!0),this._complete())},_animate:function(){this._animId=x(this._animate,this),this._step()},_step:function(t){var e=+new Date-this._startTime,i=1e3*this._duration;e<i?this._runFrame(this._easeOut(e/i),t):(this._runFrame(1),this._complete())},_runFrame:function(t,e){t=this._startPos.add(this._offset.multiplyBy(t));e&&t._round(),Z(this._el,t),this.fire("step")},_complete:function(){r(this._animId),this._inProgress=!1,this.fire("end")},_easeOut:function(t){return 1-Math.pow(1-t,this._easeOutPower)}}),A=it.extend({options:{crs:lt,center:void 0,zoom:void 0,minZoom:void 0,maxZoom:void 0,layers:[],maxBounds:void 0,renderer:void 0,zoomAnimation:!0,zoomAnimationThreshold:4,fadeAnimation:!0,markerZoomAnimation:!0,transform3DLimit:8388608,zoomSnap:1,zoomDelta:1,trackResize:!0},initialize:function(t,e){e=c(this,e),this._handlers=[],this._layers={},this._zoomBoundLayers={},this._sizeChanged=!0,this._initContainer(t),this._initLayout(),this._onResize=a(this._onResize,this),this._initEvents(),e.maxBounds&&this.setMaxBounds(e.maxBounds),void 0!==e.zoom&&(this._zoom=this._limitZoom(e.zoom)),e.center&&void 0!==e.zoom&&this.setView(w(e.center),e.zoom,{reset:!0}),this.callInitHooks(),this._zoomAnimated=ce&&b.any3d&&!b.mobileOpera&&this.options.zoomAnimation,this._zoomAnimated&&(this._createAnimProxy(),S(this._proxy,de,this._catchTransitionEnd,this)),this._addLayers(this.options.layers)},setView:function(t,e,i){if((e=void 0===e?this._zoom:this._limitZoom(e),t=this._limitCenter(w(t),e,this.options.maxBounds),i=i||{},this._stop(),this._loaded&&!i.reset&&!0!==i)&&(void 0!==i.animate&&(i.zoom=l({animate:i.animate},i.zoom),i.pan=l({animate:i.animate,duration:i.duration},i.pan)),this._zoom!==e?this._tryAnimatedZoom&&this._tryAnimatedZoom(t,e,i.zoom):this._tryAnimatedPan(t,i.pan)))return clearTimeout(this._sizeTimer),this;return this._resetView(t,e,i.pan&&i.pan.noMoveStart),this},setZoom:function(t,e){return this._loaded?this.setView(this.getCenter(),t,{zoom:e}):(this._zoom=t,this)},zoomIn:function(t,e){return t=t||(b.any3d?this.options.zoomDelta:1),this.setZoom(this._zoom+t,e)},zoomOut:function(t,e){return t=t||(b.any3d?this.options.zoomDelta:1),this.setZoom(this._zoom-t,e)},setZoomAround:function(t,e,i){var n=this.getZoomScale(e),o=this.getSize().divideBy(2),t=(t instanceof p?t:this.latLngToContainerPoint(t)).subtract(o).multiplyBy(1-1/n),n=this.containerPointToLatLng(o.add(t));return this.setView(n,e,{zoom:i})},_getBoundsCenterZoom:function(t,e){e=e||{},t=t.getBounds?t.getBounds():g(t);var i=m(e.paddingTopLeft||e.padding||[0,0]),n=m(e.paddingBottomRight||e.padding||[0,0]),o=this.getBoundsZoom(t,!1,i.add(n));return(o="number"==typeof e.maxZoom?Math.min(e.maxZoom,o):o)===1/0?{center:t.getCenter(),zoom:o}:(e=n.subtract(i).divideBy(2),n=this.project(t.getSouthWest(),o),i=this.project(t.getNorthEast(),o),{center:this.unproject(n.add(i).divideBy(2).add(e),o),zoom:o})},fitBounds:function(t,e){if((t=g(t)).isValid())return t=this._getBoundsCenterZoom(t,e),this.setView(t.center,t.zoom,e);throw new Error("Bounds are not valid.")},fitWorld:function(t){return this.fitBounds([[-90,-180],[90,180]],t)},panTo:function(t,e){return this.setView(t,this._zoom,{pan:e})},panBy:function(t,e){var i;return e=e||{},(t=m(t).round()).x||t.y?(!0===e.animate||this.getSize().contains(t)?(this._panAnim||(this._panAnim=new Fe,this._panAnim.on({step:this._onPanTransitionStep,end:this._onPanTransitionEnd},this)),e.noMoveStart||this.fire("movestart"),!1!==e.animate?(M(this._mapPane,"leaflet-pan-anim"),i=this._getMapPanePos().subtract(t).round(),this._panAnim.run(this._mapPane,i,e.duration||.25,e.easeLinearity)):(this._rawPanBy(t),this.fire("move").fire("moveend"))):this._resetView(this.unproject(this.project(this.getCenter()).add(t)),this.getZoom()),this):this.fire("moveend")},flyTo:function(n,o,t){if(!1===(t=t||{}).animate||!b.any3d)return this.setView(n,o,t);this._stop();var s=this.project(this.getCenter()),r=this.project(n),e=this.getSize(),a=this._zoom,h=(n=w(n),o=void 0===o?a:o,Math.max(e.x,e.y)),i=h*this.getZoomScale(a,o),l=r.distanceTo(s)||1,u=1.42,c=u*u;function d(t){t=(i*i-h*h+(t?-1:1)*c*c*l*l)/(2*(t?i:h)*c*l),t=Math.sqrt(t*t+1)-t;return t<1e-9?-18:Math.log(t)}function _(t){return(Math.exp(t)-Math.exp(-t))/2}function p(t){return(Math.exp(t)+Math.exp(-t))/2}var m=d(0);function f(t){return h*(p(m)*(_(t=m+u*t)/p(t))-_(m))/c}var g=Date.now(),v=(d(1)-m)/u,y=t.duration?1e3*t.duration:1e3*v*.8;return this._moveStart(!0,t.noMoveStart),function t(){var e=(Date.now()-g)/y,i=(1-Math.pow(1-e,1.5))*v;e<=1?(this._flyToFrame=x(t,this),this._move(this.unproject(s.add(r.subtract(s).multiplyBy(f(i)/l)),a),this.getScaleZoom(h/(e=i,h*(p(m)/p(m+u*e))),a),{flyTo:!0})):this._move(n,o)._moveEnd(!0)}.call(this),this},flyToBounds:function(t,e){t=this._getBoundsCenterZoom(t,e);return this.flyTo(t.center,t.zoom,e)},setMaxBounds:function(t){return t=g(t),this.listens("moveend",this._panInsideMaxBounds)&&this.off("moveend",this._panInsideMaxBounds),t.isValid()?(this.options.maxBounds=t,this._loaded&&this._panInsideMaxBounds(),this.on("moveend",this._panInsideMaxBounds)):(this.options.maxBounds=null,this)},setMinZoom:function(t){var e=this.options.minZoom;return this.options.minZoom=t,this._loaded&&e!==t&&(this.fire("zoomlevelschange"),this.getZoom()<this.options.minZoom)?this.setZoom(t):this},setMaxZoom:function(t){var e=this.options.maxZoom;return this.options.maxZoom=t,this._loaded&&e!==t&&(this.fire("zoomlevelschange"),this.getZoom()>this.options.maxZoom)?this.setZoom(t):this},panInsideBounds:function(t,e){this._enforcingBounds=!0;var i=this.getCenter(),t=this._limitCenter(i,this._zoom,g(t));return i.equals(t)||this.panTo(t,e),this._enforcingBounds=!1,this},panInside:function(t,e){var i=m((e=e||{}).paddingTopLeft||e.padding||[0,0]),n=m(e.paddingBottomRight||e.padding||[0,0]),o=this.project(this.getCenter()),t=this.project(t),s=this.getPixelBounds(),i=_([s.min.add(i),s.max.subtract(n)]),s=i.getSize();return i.contains(t)||(this._enforcingBounds=!0,n=t.subtract(i.getCenter()),i=i.extend(t).getSize().subtract(s),o.x+=n.x<0?-i.x:i.x,o.y+=n.y<0?-i.y:i.y,this.panTo(this.unproject(o),e),this._enforcingBounds=!1),this},invalidateSize:function(t){if(!this._loaded)return this;t=l({animate:!1,pan:!0},!0===t?{animate:!0}:t);var e=this.getSize(),i=(this._sizeChanged=!0,this._lastCenter=null,this.getSize()),n=e.divideBy(2).round(),o=i.divideBy(2).round(),n=n.subtract(o);return n.x||n.y?(t.animate&&t.pan?this.panBy(n):(t.pan&&this._rawPanBy(n),this.fire("move"),t.debounceMoveend?(clearTimeout(this._sizeTimer),this._sizeTimer=setTimeout(a(this.fire,this,"moveend"),200)):this.fire("moveend")),this.fire("resize",{oldSize:e,newSize:i})):this},stop:function(){return this.setZoom(this._limitZoom(this._zoom)),this.options.zoomSnap||this.fire("viewreset"),this._stop()},locate:function(t){var e,i;return t=this._locateOptions=l({timeout:1e4,watch:!1},t),"geolocation"in navigator?(e=a(this._handleGeolocationResponse,this),i=a(this._handleGeolocationError,this),t.watch?this._locationWatchId=navigator.geolocation.watchPosition(e,i,t):navigator.geolocation.getCurrentPosition(e,i,t)):this._handleGeolocationError({code:0,message:"Geolocation not supported."}),this},stopLocate:function(){return navigator.geolocation&&navigator.geolocation.clearWatch&&navigator.geolocation.clearWatch(this._locationWatchId),this._locateOptions&&(this._locateOptions.setView=!1),this},_handleGeolocationError:function(t){var e;this._container._leaflet_id&&(e=t.code,t=t.message||(1===e?"permission denied":2===e?"position unavailable":"timeout"),this._locateOptions.setView&&!this._loaded&&this.fitWorld(),this.fire("locationerror",{code:e,message:"Geolocation error: "+t+"."}))},_handleGeolocationResponse:function(t){if(this._container._leaflet_id){var e,i,n=new v(t.coords.latitude,t.coords.longitude),o=n.toBounds(2*t.coords.accuracy),s=this._locateOptions,r=(s.setView&&(e=this.getBoundsZoom(o),this.setView(n,s.maxZoom?Math.min(e,s.maxZoom):e)),{latlng:n,bounds:o,timestamp:t.timestamp});for(i in t.coords)"number"==typeof t.coords[i]&&(r[i]=t.coords[i]);this.fire("locationfound",r)}},addHandler:function(t,e){return e&&(e=this[t]=new e(this),this._handlers.push(e),this.options[t]&&e.enable()),this},remove:function(){if(this._initEvents(!0),this.options.maxBounds&&this.off("moveend",this._panInsideMaxBounds),this._containerId!==this._container._leaflet_id)throw new Error("Map container is being reused by another instance");try{delete this._container._leaflet_id,delete this._containerId}catch(t){this._container._leaflet_id=void 0,this._containerId=void 0}for(var t in void 0!==this._locationWatchId&&this.stopLocate(),this._stop(),T(this._mapPane),this._clearControlPos&&this._clearControlPos(),this._resizeRequest&&(r(this._resizeRequest),this._resizeRequest=null),this._clearHandlers(),this._loaded&&this.fire("unload"),this._layers)this._layers[t].remove();for(t in this._panes)T(this._panes[t]);return this._layers=[],this._panes=[],delete this._mapPane,delete this._renderer,this},createPane:function(t,e){e=P("div","leaflet-pane"+(t?" leaflet-"+t.replace("Pane","")+"-pane":""),e||this._mapPane);return t&&(this._panes[t]=e),e},getCenter:function(){return this._checkIfLoaded(),this._lastCenter&&!this._moved()?this._lastCenter.clone():this.layerPointToLatLng(this._getCenterLayerPoint())},getZoom:function(){return this._zoom},getBounds:function(){var t=this.getPixelBounds();return new s(this.unproject(t.getBottomLeft()),this.unproject(t.getTopRight()))},getMinZoom:function(){return void 0===this.options.minZoom?this._layersMinZoom||0:this.options.minZoom},getMaxZoom:function(){return void 0===this.options.maxZoom?void 0===this._layersMaxZoom?1/0:this._layersMaxZoom:this.options.maxZoom},getBoundsZoom:function(t,e,i){t=g(t),i=m(i||[0,0]);var n=this.getZoom()||0,o=this.getMinZoom(),s=this.getMaxZoom(),r=t.getNorthWest(),t=t.getSouthEast(),i=this.getSize().subtract(i),t=_(this.project(t,n),this.project(r,n)).getSize(),r=b.any3d?this.options.zoomSnap:1,a=i.x/t.x,i=i.y/t.y,t=e?Math.max(a,i):Math.min(a,i),n=this.getScaleZoom(t,n);return r&&(n=Math.round(n/(r/100))*(r/100),n=e?Math.ceil(n/r)*r:Math.floor(n/r)*r),Math.max(o,Math.min(s,n))},getSize:function(){return this._size&&!this._sizeChanged||(this._size=new p(this._container.clientWidth||0,this._container.clientHeight||0),this._sizeChanged=!1),this._size.clone()},getPixelBounds:function(t,e){t=this._getTopLeftPoint(t,e);return new f(t,t.add(this.getSize()))},getPixelOrigin:function(){return this._checkIfLoaded(),this._pixelOrigin},getPixelWorldBounds:function(t){return this.options.crs.getProjectedBounds(void 0===t?this.getZoom():t)},getPane:function(t){return"string"==typeof t?this._panes[t]:t},getPanes:function(){return this._panes},getContainer:function(){return this._container},getZoomScale:function(t,e){var i=this.options.crs;return e=void 0===e?this._zoom:e,i.scale(t)/i.scale(e)},getScaleZoom:function(t,e){var i=this.options.crs,t=(e=void 0===e?this._zoom:e,i.zoom(t*i.scale(e)));return isNaN(t)?1/0:t},project:function(t,e){return e=void 0===e?this._zoom:e,this.options.crs.latLngToPoint(w(t),e)},unproject:function(t,e){return e=void 0===e?this._zoom:e,this.options.crs.pointToLatLng(m(t),e)},layerPointToLatLng:function(t){t=m(t).add(this.getPixelOrigin());return this.unproject(t)},latLngToLayerPoint:function(t){return this.project(w(t))._round()._subtract(this.getPixelOrigin())},wrapLatLng:function(t){return this.options.crs.wrapLatLng(w(t))},wrapLatLngBounds:function(t){return this.options.crs.wrapLatLngBounds(g(t))},distance:function(t,e){return this.options.crs.distance(w(t),w(e))},containerPointToLayerPoint:function(t){return m(t).subtract(this._getMapPanePos())},layerPointToContainerPoint:function(t){return m(t).add(this._getMapPanePos())},containerPointToLatLng:function(t){t=this.containerPointToLayerPoint(m(t));return this.layerPointToLatLng(t)},latLngToContainerPoint:function(t){return this.layerPointToContainerPoint(this.latLngToLayerPoint(w(t)))},mouseEventToContainerPoint:function(t){return De(t,this._container)},mouseEventToLayerPoint:function(t){return this.containerPointToLayerPoint(this.mouseEventToContainerPoint(t))},mouseEventToLatLng:function(t){return this.layerPointToLatLng(this.mouseEventToLayerPoint(t))},_initContainer:function(t){t=this._container=_e(t);if(!t)throw new Error("Map container not found.");if(t._leaflet_id)throw new Error("Map container is already initialized.");S(t,"scroll",this._onScroll,this),this._containerId=h(t)},_initLayout:function(){var t=this._container,e=(this._fadeAnimated=this.options.fadeAnimation&&b.any3d,M(t,"leaflet-container"+(b.touch?" leaflet-touch":"")+(b.retina?" leaflet-retina":"")+(b.ielt9?" leaflet-oldie":"")+(b.safari?" leaflet-safari":"")+(this._fadeAnimated?" leaflet-fade-anim":"")),pe(t,"position"));"absolute"!==e&&"relative"!==e&&"fixed"!==e&&"sticky"!==e&&(t.style.position="relative"),this._initPanes(),this._initControlPos&&this._initControlPos()},_initPanes:function(){var t=this._panes={};this._paneRenderers={},this._mapPane=this.createPane("mapPane",this._container),Z(this._mapPane,new p(0,0)),this.createPane("tilePane"),this.createPane("overlayPane"),this.createPane("shadowPane"),this.createPane("markerPane"),this.createPane("tooltipPane"),this.createPane("popupPane"),this.options.markerZoomAnimation||(M(t.markerPane,"leaflet-zoom-hide"),M(t.shadowPane,"leaflet-zoom-hide"))},_resetView:function(t,e,i){Z(this._mapPane,new p(0,0));var n=!this._loaded,o=(this._loaded=!0,e=this._limitZoom(e),this.fire("viewprereset"),this._zoom!==e);this._moveStart(o,i)._move(t,e)._moveEnd(o),this.fire("viewreset"),n&&this.fire("load")},_moveStart:function(t,e){return t&&this.fire("zoomstart"),e||this.fire("movestart"),this},_move:function(t,e,i,n){void 0===e&&(e=this._zoom);var o=this._zoom!==e;return this._zoom=e,this._lastCenter=t,this._pixelOrigin=this._getNewPixelOrigin(t),n?i&&i.pinch&&this.fire("zoom",i):((o||i&&i.pinch)&&this.fire("zoom",i),this.fire("move",i)),this},_moveEnd:function(t){return t&&this.fire("zoomend"),this.fire("moveend")},_stop:function(){return r(this._flyToFrame),this._panAnim&&this._panAnim.stop(),this},_rawPanBy:function(t){Z(this._mapPane,this._getMapPanePos().subtract(t))},_getZoomSpan:function(){return this.getMaxZoom()-this.getMinZoom()},_panInsideMaxBounds:function(){this._enforcingBounds||this.panInsideBounds(this.options.maxBounds)},_checkIfLoaded:function(){if(!this._loaded)throw new Error("Set map center and zoom first.")},_initEvents:function(t){this._targets={};var e=t?k:S;e((this._targets[h(this._container)]=this)._container,"click dblclick mousedown mouseup mouseover mouseout mousemove contextmenu keypress keydown keyup",this._handleDOMEvent,this),this.options.trackResize&&e(window,"resize",this._onResize,this),b.any3d&&this.options.transform3DLimit&&(t?this.off:this.on).call(this,"moveend",this._onMoveEnd)},_onResize:function(){r(this._resizeRequest),this._resizeRequest=x(function(){this.invalidateSize({debounceMoveend:!0})},this)},_onScroll:function(){this._container.scrollTop=0,this._container.scrollLeft=0},_onMoveEnd:function(){var t=this._getMapPanePos();Math.max(Math.abs(t.x),Math.abs(t.y))>=this.options.transform3DLimit&&this._resetView(this.getCenter(),this.getZoom())},_findEventTargets:function(t,e){for(var i,n=[],o="mouseout"===e||"mouseover"===e,s=t.target||t.srcElement,r=!1;s;){if((i=this._targets[h(s)])&&("click"===e||"preclick"===e)&&this._draggableMoved(i)){r=!0;break}if(i&&i.listens(e,!0)){if(o&&!We(s,t))break;if(n.push(i),o)break}if(s===this._container)break;s=s.parentNode}return n=n.length||r||o||!this.listens(e,!0)?n:[this]},_isClickDisabled:function(t){for(;t&&t!==this._container;){if(t._leaflet_disable_click)return!0;t=t.parentNode}},_handleDOMEvent:function(t){var e,i=t.target||t.srcElement;!this._loaded||i._leaflet_disable_events||"click"===t.type&&this._isClickDisabled(i)||("mousedown"===(e=t.type)&&Me(i),this._fireDOMEvent(t,e))},_mouseEvents:["click","dblclick","mouseover","mouseout","contextmenu"],_fireDOMEvent:function(t,e,i){"click"===t.type&&((a=l({},t)).type="preclick",this._fireDOMEvent(a,a.type,i));var n=this._findEventTargets(t,e);if(i){for(var o=[],s=0;s<i.length;s++)i[s].listens(e,!0)&&o.push(i[s]);n=o.concat(n)}if(n.length){"contextmenu"===e&&O(t);var r,a=n[0],h={originalEvent:t};for("keypress"!==t.type&&"keydown"!==t.type&&"keyup"!==t.type&&(r=a.getLatLng&&(!a._radius||a._radius<=10),h.containerPoint=r?this.latLngToContainerPoint(a.getLatLng()):this.mouseEventToContainerPoint(t),h.layerPoint=this.containerPointToLayerPoint(h.containerPoint),h.latlng=r?a.getLatLng():this.layerPointToLatLng(h.layerPoint)),s=0;s<n.length;s++)if(n[s].fire(e,h,!0),h.originalEvent._stopped||!1===n[s].options.bubblingMouseEvents&&-1!==G(this._mouseEvents,e))return}},_draggableMoved:function(t){return(t=t.dragging&&t.dragging.enabled()?t:this).dragging&&t.dragging.moved()||this.boxZoom&&this.boxZoom.moved()},_clearHandlers:function(){for(var t=0,e=this._handlers.length;t<e;t++)this._handlers[t].disable()},whenReady:function(t,e){return this._loaded?t.call(e||this,{target:this}):this.on("load",t,e),this},_getMapPanePos:function(){return Pe(this._mapPane)||new p(0,0)},_moved:function(){var t=this._getMapPanePos();return t&&!t.equals([0,0])},_getTopLeftPoint:function(t,e){return(t&&void 0!==e?this._getNewPixelOrigin(t,e):this.getPixelOrigin()).subtract(this._getMapPanePos())},_getNewPixelOrigin:function(t,e){var i=this.getSize()._divideBy(2);return this.project(t,e)._subtract(i)._add(this._getMapPanePos())._round()},_latLngToNewLayerPoint:function(t,e,i){i=this._getNewPixelOrigin(i,e);return this.project(t,e)._subtract(i)},_latLngBoundsToNewLayerBounds:function(t,e,i){i=this._getNewPixelOrigin(i,e);return _([this.project(t.getSouthWest(),e)._subtract(i),this.project(t.getNorthWest(),e)._subtract(i),this.project(t.getSouthEast(),e)._subtract(i),this.project(t.getNorthEast(),e)._subtract(i)])},_getCenterLayerPoint:function(){return this.containerPointToLayerPoint(this.getSize()._divideBy(2))},_getCenterOffset:function(t){return this.latLngToLayerPoint(t).subtract(this._getCenterLayerPoint())},_limitCenter:function(t,e,i){var n,o;return!i||(n=this.project(t,e),o=this.getSize().divideBy(2),o=new f(n.subtract(o),n.add(o)),o=this._getBoundsOffset(o,i,e),Math.abs(o.x)<=1&&Math.abs(o.y)<=1)?t:this.unproject(n.add(o),e)},_limitOffset:function(t,e){var i;return e?(i=new f((i=this.getPixelBounds()).min.add(t),i.max.add(t)),t.add(this._getBoundsOffset(i,e))):t},_getBoundsOffset:function(t,e,i){e=_(this.project(e.getNorthEast(),i),this.project(e.getSouthWest(),i)),i=e.min.subtract(t.min),e=e.max.subtract(t.max);return new p(this._rebound(i.x,-e.x),this._rebound(i.y,-e.y))},_rebound:function(t,e){return 0<t+e?Math.round(t-e)/2:Math.max(0,Math.ceil(t))-Math.max(0,Math.floor(e))},_limitZoom:function(t){var e=this.getMinZoom(),i=this.getMaxZoom(),n=b.any3d?this.options.zoomSnap:1;return n&&(t=Math.round(t/n)*n),Math.max(e,Math.min(i,t))},_onPanTransitionStep:function(){this.fire("move")},_onPanTransitionEnd:function(){z(this._mapPane,"leaflet-pan-anim"),this.fire("moveend")},_tryAnimatedPan:function(t,e){t=this._getCenterOffset(t)._trunc();return!(!0!==(e&&e.animate)&&!this.getSize().contains(t))&&(this.panBy(t,e),!0)},_createAnimProxy:function(){var t=this._proxy=P("div","leaflet-proxy leaflet-zoom-animated");this._panes.mapPane.appendChild(t),this.on("zoomanim",function(t){var e=ue,i=this._proxy.style[e];be(this._proxy,this.project(t.center,t.zoom),this.getZoomScale(t.zoom,1)),i===this._proxy.style[e]&&this._animatingZoom&&this._onZoomTransitionEnd()},this),this.on("load moveend",this._animMoveEnd,this),this._on("unload",this._destroyAnimProxy,this)},_destroyAnimProxy:function(){T(this._proxy),this.off("load moveend",this._animMoveEnd,this),delete this._proxy},_animMoveEnd:function(){var t=this.getCenter(),e=this.getZoom();be(this._proxy,this.project(t,e),this.getZoomScale(e,1))},_catchTransitionEnd:function(t){this._animatingZoom&&0<=t.propertyName.indexOf("transform")&&this._onZoomTransitionEnd()},_nothingToAnimate:function(){return!this._container.getElementsByClassName("leaflet-zoom-animated").length},_tryAnimatedZoom:function(t,e,i){if(!this._animatingZoom){if(i=i||{},!this._zoomAnimated||!1===i.animate||this._nothingToAnimate()||Math.abs(e-this._zoom)>this.options.zoomAnimationThreshold)return!1;var n=this.getZoomScale(e),n=this._getCenterOffset(t)._divideBy(1-1/n);if(!0!==i.animate&&!this.getSize().contains(n))return!1;x(function(){this._moveStart(!0,i.noMoveStart||!1)._animateZoom(t,e,!0)},this)}return!0},_animateZoom:function(t,e,i,n){this._mapPane&&(i&&(this._animatingZoom=!0,this._animateToCenter=t,this._animateToZoom=e,M(this._mapPane,"leaflet-zoom-anim")),this.fire("zoomanim",{center:t,zoom:e,noUpdate:n}),this._tempFireZoomEvent||(this._tempFireZoomEvent=this._zoom!==this._animateToZoom),this._move(this._animateToCenter,this._animateToZoom,void 0,!0),setTimeout(a(this._onZoomTransitionEnd,this),250))},_onZoomTransitionEnd:function(){this._animatingZoom&&(this._mapPane&&z(this._mapPane,"leaflet-zoom-anim"),this._animatingZoom=!1,this._move(this._animateToCenter,this._animateToZoom,void 0,!0),this._tempFireZoomEvent&&this.fire("zoom"),delete this._tempFireZoomEvent,this.fire("move"),this._moveEnd(!0))}});function Ue(t){return new B(t)}var B=et.extend({options:{position:"topright"},initialize:function(t){c(this,t)},getPosition:function(){return this.options.position},setPosition:function(t){var e=this._map;return e&&e.removeControl(this),this.options.position=t,e&&e.addControl(this),this},getContainer:function(){return this._container},addTo:function(t){this.remove(),this._map=t;var e=this._container=this.onAdd(t),i=this.getPosition(),t=t._controlCorners[i];return M(e,"leaflet-control"),-1!==i.indexOf("bottom")?t.insertBefore(e,t.firstChild):t.appendChild(e),this._map.on("unload",this.remove,this),this},remove:function(){return this._map&&(T(this._container),this.onRemove&&this.onRemove(this._map),this._map.off("unload",this.remove,this),this._map=null),this},_refocusOnMap:function(t){this._map&&t&&0<t.screenX&&0<t.screenY&&this._map.getContainer().focus()}}),Ve=(A.include({addControl:function(t){return t.addTo(this),this},removeControl:function(t){return t.remove(),this},_initControlPos:function(){var i=this._controlCorners={},n="leaflet-",o=this._controlContainer=P("div",n+"control-container",this._container);function t(t,e){i[t+e]=P("div",n+t+" "+n+e,o)}t("top","left"),t("top","right"),t("bottom","left"),t("bottom","right")},_clearControlPos:function(){for(var t in this._controlCorners)T(this._controlCorners[t]);T(this._controlContainer),delete this._controlCorners,delete this._controlContainer}}),B.extend({options:{collapsed:!0,position:"topright",autoZIndex:!0,hideSingleBase:!1,sortLayers:!1,sortFunction:function(t,e,i,n){return i<n?-1:n<i?1:0}},initialize:function(t,e,i){for(var n in c(this,i),this._layerControlInputs=[],this._layers=[],this._lastZIndex=0,this._handlingClick=!1,this._preventClick=!1,t)this._addLayer(t[n],n);for(n in e)this._addLayer(e[n],n,!0)},onAdd:function(t){this._initLayout(),this._update(),(this._map=t).on("zoomend",this._checkDisabledLayers,this);for(var e=0;e<this._layers.length;e++)this._layers[e].layer.on("add remove",this._onLayerChange,this);return this._container},addTo:function(t){return B.prototype.addTo.call(this,t),this._expandIfNotCollapsed()},onRemove:function(){this._map.off("zoomend",this._checkDisabledLayers,this);for(var t=0;t<this._layers.length;t++)this._layers[t].layer.off("add remove",this._onLayerChange,this)},addBaseLayer:function(t,e){return this._addLayer(t,e),this._map?this._update():this},addOverlay:function(t,e){return this._addLayer(t,e,!0),this._map?this._update():this},removeLayer:function(t){t.off("add remove",this._onLayerChange,this);t=this._getLayer(h(t));return t&&this._layers.splice(this._layers.indexOf(t),1),this._map?this._update():this},expand:function(){M(this._container,"leaflet-control-layers-expanded"),this._section.style.height=null;var t=this._map.getSize().y-(this._container.offsetTop+50);return t<this._section.clientHeight?(M(this._section,"leaflet-control-layers-scrollbar"),this._section.style.height=t+"px"):z(this._section,"leaflet-control-layers-scrollbar"),this._checkDisabledLayers(),this},collapse:function(){return z(this._container,"leaflet-control-layers-expanded"),this},_initLayout:function(){var t="leaflet-control-layers",e=this._container=P("div",t),i=this.options.collapsed,n=(e.setAttribute("aria-haspopup",!0),Ie(e),Be(e),this._section=P("section",t+"-list")),o=(i&&(this._map.on("click",this.collapse,this),S(e,{mouseenter:this._expandSafely,mouseleave:this.collapse},this)),this._layersLink=P("a",t+"-toggle",e));o.href="#",o.title="Layers",o.setAttribute("role","button"),S(o,{keydown:function(t){13===t.keyCode&&this._expandSafely()},click:function(t){O(t),this._expandSafely()}},this),i||this.expand(),this._baseLayersList=P("div",t+"-base",n),this._separator=P("div",t+"-separator",n),this._overlaysList=P("div",t+"-overlays",n),e.appendChild(n)},_getLayer:function(t){for(var e=0;e<this._layers.length;e++)if(this._layers[e]&&h(this._layers[e].layer)===t)return this._layers[e]},_addLayer:function(t,e,i){this._map&&t.on("add remove",this._onLayerChange,this),this._layers.push({layer:t,name:e,overlay:i}),this.options.sortLayers&&this._layers.sort(a(function(t,e){return this.options.sortFunction(t.layer,e.layer,t.name,e.name)},this)),this.options.autoZIndex&&t.setZIndex&&(this._lastZIndex++,t.setZIndex(this._lastZIndex)),this._expandIfNotCollapsed()},_update:function(){if(this._container){me(this._baseLayersList),me(this._overlaysList),this._layerControlInputs=[];for(var t,e,i,n=0,o=0;o<this._layers.length;o++)i=this._layers[o],this._addItem(i),e=e||i.overlay,t=t||!i.overlay,n+=i.overlay?0:1;this.options.hideSingleBase&&(this._baseLayersList.style.display=(t=t&&1<n)?"":"none"),this._separator.style.display=e&&t?"":"none"}return this},_onLayerChange:function(t){this._handlingClick||this._update();var e=this._getLayer(h(t.target)),t=e.overlay?"add"===t.type?"overlayadd":"overlayremove":"add"===t.type?"baselayerchange":null;t&&this._map.fire(t,e)},_createRadioElement:function(t,e){t=\'<input type="radio" class="leaflet-control-layers-selector" name="\'+t+\'"\'+(e?\' checked="checked"\':"")+"/>",e=document.createElement("div");return e.innerHTML=t,e.firstChild},_addItem:function(t){var e,i=document.createElement("label"),n=this._map.hasLayer(t.layer),n=(t.overlay?((e=document.createElement("input")).type="checkbox",e.className="leaflet-control-layers-selector",e.defaultChecked=n):e=this._createRadioElement("leaflet-base-layers_"+h(this),n),this._layerControlInputs.push(e),e.layerId=h(t.layer),S(e,"click",this._onInputClick,this),document.createElement("span")),o=(n.innerHTML=" "+t.name,document.createElement("span"));return i.appendChild(o),o.appendChild(e),o.appendChild(n),(t.overlay?this._overlaysList:this._baseLayersList).appendChild(i),this._checkDisabledLayers(),i},_onInputClick:function(){if(!this._preventClick){var t,e,i=this._layerControlInputs,n=[],o=[];this._handlingClick=!0;for(var s=i.length-1;0<=s;s--)t=i[s],e=this._getLayer(t.layerId).layer,t.checked?n.push(e):t.checked||o.push(e);for(s=0;s<o.length;s++)this._map.hasLayer(o[s])&&this._map.removeLayer(o[s]);for(s=0;s<n.length;s++)this._map.hasLayer(n[s])||this._map.addLayer(n[s]);this._handlingClick=!1,this._refocusOnMap()}},_checkDisabledLayers:function(){for(var t,e,i=this._layerControlInputs,n=this._map.getZoom(),o=i.length-1;0<=o;o--)t=i[o],e=this._getLayer(t.layerId).layer,t.disabled=void 0!==e.options.minZoom&&n<e.options.minZoom||void 0!==e.options.maxZoom&&n>e.options.maxZoom},_expandIfNotCollapsed:function(){return this._map&&!this.options.collapsed&&this.expand(),this},_expandSafely:function(){var t=this._section,e=(this._preventClick=!0,S(t,"click",O),this.expand(),this);setTimeout(function(){k(t,"click",O),e._preventClick=!1})}})),qe=B.extend({options:{position:"topleft",zoomInText:\'<span aria-hidden="true">+</span>\',zoomInTitle:"Zoom in",zoomOutText:\'<span aria-hidden="true">&#x2212;</span>\',zoomOutTitle:"Zoom out"},onAdd:function(t){var e="leaflet-control-zoom",i=P("div",e+" leaflet-bar"),n=this.options;return this._zoomInButton=this._createButton(n.zoomInText,n.zoomInTitle,e+"-in",i,this._zoomIn),this._zoomOutButton=this._createButton(n.zoomOutText,n.zoomOutTitle,e+"-out",i,this._zoomOut),this._updateDisabled(),t.on("zoomend zoomlevelschange",this._updateDisabled,this),i},onRemove:function(t){t.off("zoomend zoomlevelschange",this._updateDisabled,this)},disable:function(){return this._disabled=!0,this._updateDisabled(),this},enable:function(){return this._disabled=!1,this._updateDisabled(),this},_zoomIn:function(t){!this._disabled&&this._map._zoom<this._map.getMaxZoom()&&this._map.zoomIn(this._map.options.zoomDelta*(t.shiftKey?3:1))},_zoomOut:function(t){!this._disabled&&this._map._zoom>this._map.getMinZoom()&&this._map.zoomOut(this._map.options.zoomDelta*(t.shiftKey?3:1))},_createButton:function(t,e,i,n,o){i=P("a",i,n);return i.innerHTML=t,i.href="#",i.title=e,i.setAttribute("role","button"),i.setAttribute("aria-label",e),Ie(i),S(i,"click",Re),S(i,"click",o,this),S(i,"click",this._refocusOnMap,this),i},_updateDisabled:function(){var t=this._map,e="leaflet-disabled";z(this._zoomInButton,e),z(this._zoomOutButton,e),this._zoomInButton.setAttribute("aria-disabled","false"),this._zoomOutButton.setAttribute("aria-disabled","false"),!this._disabled&&t._zoom!==t.getMinZoom()||(M(this._zoomOutButton,e),this._zoomOutButton.setAttribute("aria-disabled","true")),!this._disabled&&t._zoom!==t.getMaxZoom()||(M(this._zoomInButton,e),this._zoomInButton.setAttribute("aria-disabled","true"))}}),Ge=(A.mergeOptions({zoomControl:!0}),A.addInitHook(function(){this.options.zoomControl&&(this.zoomControl=new qe,this.addControl(this.zoomControl))}),B.extend({options:{position:"bottomleft",maxWidth:100,metric:!0,imperial:!0},onAdd:function(t){var e="leaflet-control-scale",i=P("div",e),n=this.options;return this._addScales(n,e+"-line",i),t.on(n.updateWhenIdle?"moveend":"move",this._update,this),t.whenReady(this._update,this),i},onRemove:function(t){t.off(this.options.updateWhenIdle?"moveend":"move",this._update,this)},_addScales:function(t,e,i){t.metric&&(this._mScale=P("div",e,i)),t.imperial&&(this._iScale=P("div",e,i))},_update:function(){var t=this._map,e=t.getSize().y/2,t=t.distance(t.containerPointToLatLng([0,e]),t.containerPointToLatLng([this.options.maxWidth,e]));this._updateScales(t)},_updateScales:function(t){this.options.metric&&t&&this._updateMetric(t),this.options.imperial&&t&&this._updateImperial(t)},_updateMetric:function(t){var e=this._getRoundNum(t);this._updateScale(this._mScale,e<1e3?e+" m":e/1e3+" km",e/t)},_updateImperial:function(t){var e,i,t=3.2808399*t;5280<t?(i=this._getRoundNum(e=t/5280),this._updateScale(this._iScale,i+" mi",i/e)):(i=this._getRoundNum(t),this._updateScale(this._iScale,i+" ft",i/t))},_updateScale:function(t,e,i){t.style.width=Math.round(this.options.maxWidth*i)+"px",t.innerHTML=e},_getRoundNum:function(t){var e=Math.pow(10,(Math.floor(t)+"").length-1),t=t/e;return e*(t=10<=t?10:5<=t?5:3<=t?3:2<=t?2:1)}})),Ke=B.extend({options:{position:"bottomright",prefix:\'<a href="https://leafletjs.com" title="A JavaScript library for interactive maps">\'+(b.inlineSvg?\'<svg aria-hidden="true" xmlns="http://www.w3.org/2000/svg" width="12" height="8" viewBox="0 0 12 8" class="leaflet-attribution-flag"><path fill="#4C7BE1" d="M0 0h12v4H0z"/><path fill="#FFD500" d="M0 4h12v3H0z"/><path fill="#E0BC00" d="M0 7h12v1H0z"/></svg> \':"")+"Leaflet</a>"},initialize:function(t){c(this,t),this._attributions={}},onAdd:function(t){for(var e in(t.attributionControl=this)._container=P("div","leaflet-control-attribution"),Ie(this._container),t._layers)t._layers[e].getAttribution&&this.addAttribution(t._layers[e].getAttribution());return this._update(),t.on("layeradd",this._addAttribution,this),this._container},onRemove:function(t){t.off("layeradd",this._addAttribution,this)},_addAttribution:function(t){t.layer.getAttribution&&(this.addAttribution(t.layer.getAttribution()),t.layer.once("remove",function(){this.removeAttribution(t.layer.getAttribution())},this))},setPrefix:function(t){return this.options.prefix=t,this._update(),this},addAttribution:function(t){return t&&(this._attributions[t]||(this._attributions[t]=0),this._attributions[t]++,this._update()),this},removeAttribution:function(t){return t&&this._attributions[t]&&(this._attributions[t]--,this._update()),this},_update:function(){if(this._map){var t,e=[];for(t in this._attributions)this._attributions[t]&&e.push(t);var i=[];this.options.prefix&&i.push(this.options.prefix),e.length&&i.push(e.join(", ")),this._container.innerHTML=i.join(\' <span aria-hidden="true">|</span> \')}}}),n=(A.mergeOptions({attributionControl:!0}),A.addInitHook(function(){this.options.attributionControl&&(new Ke).addTo(this)}),B.Layers=Ve,B.Zoom=qe,B.Scale=Ge,B.Attribution=Ke,Ue.layers=function(t,e,i){return new Ve(t,e,i)},Ue.zoom=function(t){return new qe(t)},Ue.scale=function(t){return new Ge(t)},Ue.attribution=function(t){return new Ke(t)},et.extend({initialize:function(t){this._map=t},enable:function(){return this._enabled||(this._enabled=!0,this.addHooks()),this},disable:function(){return this._enabled&&(this._enabled=!1,this.removeHooks()),this},enabled:function(){return!!this._enabled}})),ft=(n.addTo=function(t,e){return t.addHandler(e,this),this},{Events:e}),Ye=b.touch?"touchstart mousedown":"mousedown",Xe=it.extend({options:{clickTolerance:3},initialize:function(t,e,i,n){c(this,n),this._element=t,this._dragStartTarget=e||t,this._preventOutline=i},enable:function(){this._enabled||(S(this._dragStartTarget,Ye,this._onDown,this),this._enabled=!0)},disable:function(){this._enabled&&(Xe._dragging===this&&this.finishDrag(!0),k(this._dragStartTarget,Ye,this._onDown,this),this._enabled=!1,this._moved=!1)},_onDown:function(t){var e,i;this._enabled&&(this._moved=!1,ve(this._element,"leaflet-zoom-anim")||(t.touches&&1!==t.touches.length?Xe._dragging===this&&this.finishDrag():Xe._dragging||t.shiftKey||1!==t.which&&1!==t.button&&!t.touches||((Xe._dragging=this)._preventOutline&&Me(this._element),Le(),re(),this._moving||(this.fire("down"),i=t.touches?t.touches[0]:t,e=Ce(this._element),this._startPoint=new p(i.clientX,i.clientY),this._startPos=Pe(this._element),this._parentScale=Ze(e),i="mousedown"===t.type,S(document,i?"mousemove":"touchmove",this._onMove,this),S(document,i?"mouseup":"touchend touchcancel",this._onUp,this)))))},_onMove:function(t){var e;this._enabled&&(t.touches&&1<t.touches.length?this._moved=!0:!(e=new p((e=t.touches&&1===t.touches.length?t.touches[0]:t).clientX,e.clientY)._subtract(this._startPoint)).x&&!e.y||Math.abs(e.x)+Math.abs(e.y)<this.options.clickTolerance||(e.x/=this._parentScale.x,e.y/=this._parentScale.y,O(t),this._moved||(this.fire("dragstart"),this._moved=!0,M(document.body,"leaflet-dragging"),this._lastTarget=t.target||t.srcElement,window.SVGElementInstance&&this._lastTarget instanceof window.SVGElementInstance&&(this._lastTarget=this._lastTarget.correspondingUseElement),M(this._lastTarget,"leaflet-drag-target")),this._newPos=this._startPos.add(e),this._moving=!0,this._lastEvent=t,this._updatePosition()))},_updatePosition:function(){var t={originalEvent:this._lastEvent};this.fire("predrag",t),Z(this._element,this._newPos),this.fire("drag",t)},_onUp:function(){this._enabled&&this.finishDrag()},finishDrag:function(t){z(document.body,"leaflet-dragging"),this._lastTarget&&(z(this._lastTarget,"leaflet-drag-target"),this._lastTarget=null),k(document,"mousemove touchmove",this._onMove,this),k(document,"mouseup touchend touchcancel",this._onUp,this),Te(),ae();var e=this._moved&&this._moving;this._moving=!1,Xe._dragging=!1,e&&this.fire("dragend",{noInertia:t,distance:this._newPos.distanceTo(this._startPos)})}});function Je(t,e,i){for(var n,o,s,r,a,h,l,u=[1,4,2,8],c=0,d=t.length;c<d;c++)t[c]._code=si(t[c],e);for(s=0;s<4;s++){for(h=u[s],n=[],c=0,o=(d=t.length)-1;c<d;o=c++)r=t[c],a=t[o],r._code&h?a._code&h||((l=oi(a,r,h,e,i))._code=si(l,e),n.push(l)):(a._code&h&&((l=oi(a,r,h,e,i))._code=si(l,e),n.push(l)),n.push(r));t=n}return t}function $e(t,e){var i,n,o,s,r,a,h;if(!t||0===t.length)throw new Error("latlngs not passed");I(t)||(console.warn("latlngs are not flat! Only the first ring will be used"),t=t[0]);for(var l=w([0,0]),u=g(t),c=(u.getNorthWest().distanceTo(u.getSouthWest())*u.getNorthEast().distanceTo(u.getNorthWest())<1700&&(l=Qe(t)),t.length),d=[],_=0;_<c;_++){var p=w(t[_]);d.push(e.project(w([p.lat-l.lat,p.lng-l.lng])))}for(_=r=a=h=0,i=c-1;_<c;i=_++)n=d[_],o=d[i],s=n.y*o.x-o.y*n.x,a+=(n.x+o.x)*s,h+=(n.y+o.y)*s,r+=3*s;u=0===r?d[0]:[a/r,h/r],u=e.unproject(m(u));return w([u.lat+l.lat,u.lng+l.lng])}function Qe(t){for(var e=0,i=0,n=0,o=0;o<t.length;o++){var s=w(t[o]);e+=s.lat,i+=s.lng,n++}return w([e/n,i/n])}var ti,gt={__proto__:null,clipPolygon:Je,polygonCenter:$e,centroid:Qe};function ei(t,e){if(e&&t.length){var i=t=function(t,e){for(var i=[t[0]],n=1,o=0,s=t.length;n<s;n++)(function(t,e){var i=e.x-t.x,e=e.y-t.y;return i*i+e*e})(t[n],t[o])>e&&(i.push(t[n]),o=n);o<s-1&&i.push(t[s-1]);return i}(t,e=e*e),n=i.length,o=new(typeof Uint8Array!=void 0+""?Uint8Array:Array)(n);o[0]=o[n-1]=1,function t(e,i,n,o,s){var r,a,h,l=0;for(a=o+1;a<=s-1;a++)h=ri(e[a],e[o],e[s],!0),l<h&&(r=a,l=h);n<l&&(i[r]=1,t(e,i,n,o,r),t(e,i,n,r,s))}(i,o,e,0,n-1);var s,r=[];for(s=0;s<n;s++)o[s]&&r.push(i[s]);return r}return t.slice()}function ii(t,e,i){return Math.sqrt(ri(t,e,i,!0))}function ni(t,e,i,n,o){var s,r,a,h=n?ti:si(t,i),l=si(e,i);for(ti=l;;){if(!(h|l))return[t,e];if(h&l)return!1;a=si(r=oi(t,e,s=h||l,i,o),i),s===h?(t=r,h=a):(e=r,l=a)}}function oi(t,e,i,n,o){var s,r,a=e.x-t.x,e=e.y-t.y,h=n.min,n=n.max;return 8&i?(s=t.x+a*(n.y-t.y)/e,r=n.y):4&i?(s=t.x+a*(h.y-t.y)/e,r=h.y):2&i?(s=n.x,r=t.y+e*(n.x-t.x)/a):1&i&&(s=h.x,r=t.y+e*(h.x-t.x)/a),new p(s,r,o)}function si(t,e){var i=0;return t.x<e.min.x?i|=1:t.x>e.max.x&&(i|=2),t.y<e.min.y?i|=4:t.y>e.max.y&&(i|=8),i}function ri(t,e,i,n){var o=e.x,e=e.y,s=i.x-o,r=i.y-e,a=s*s+r*r;return 0<a&&(1<(a=((t.x-o)*s+(t.y-e)*r)/a)?(o=i.x,e=i.y):0<a&&(o+=s*a,e+=r*a)),s=t.x-o,r=t.y-e,n?s*s+r*r:new p(o,e)}function I(t){return!d(t[0])||"object"!=typeof t[0][0]&&void 0!==t[0][0]}function ai(t){return console.warn("Deprecated use of _flat, please use L.LineUtil.isFlat instead."),I(t)}function hi(t,e){var i,n,o,s,r,a;if(!t||0===t.length)throw new Error("latlngs not passed");I(t)||(console.warn("latlngs are not flat! Only the first ring will be used"),t=t[0]);for(var h=w([0,0]),l=g(t),u=(l.getNorthWest().distanceTo(l.getSouthWest())*l.getNorthEast().distanceTo(l.getNorthWest())<1700&&(h=Qe(t)),t.length),c=[],d=0;d<u;d++){var _=w(t[d]);c.push(e.project(w([_.lat-h.lat,_.lng-h.lng])))}for(i=d=0;d<u-1;d++)i+=c[d].distanceTo(c[d+1])/2;if(0===i)a=c[0];else for(n=d=0;d<u-1;d++)if(o=c[d],s=c[d+1],i<(n+=r=o.distanceTo(s))){a=[s.x-(r=(n-i)/r)*(s.x-o.x),s.y-r*(s.y-o.y)];break}l=e.unproject(m(a));return w([l.lat+h.lat,l.lng+h.lng])}var vt={__proto__:null,simplify:ei,pointToSegmentDistance:ii,closestPointOnSegment:function(t,e,i){return ri(t,e,i)},clipSegment:ni,_getEdgeIntersection:oi,_getBitCode:si,_sqClosestPointOnSegment:ri,isFlat:I,_flat:ai,polylineCenter:hi},yt={project:function(t){return new p(t.lng,t.lat)},unproject:function(t){return new v(t.y,t.x)},bounds:new f([-180,-90],[180,90])},xt={R:6378137,R_MINOR:6356752.314245179,bounds:new f([-20037508.34279,-15496570.73972],[20037508.34279,18764656.23138]),project:function(t){var e=Math.PI/180,i=this.R,n=t.lat*e,o=this.R_MINOR/i,o=Math.sqrt(1-o*o),s=o*Math.sin(n),s=Math.tan(Math.PI/4-n/2)/Math.pow((1-s)/(1+s),o/2),n=-i*Math.log(Math.max(s,1e-10));return new p(t.lng*e*i,n)},unproject:function(t){for(var e,i=180/Math.PI,n=this.R,o=this.R_MINOR/n,s=Math.sqrt(1-o*o),r=Math.exp(-t.y/n),a=Math.PI/2-2*Math.atan(r),h=0,l=.1;h<15&&1e-7<Math.abs(l);h++)e=s*Math.sin(a),e=Math.pow((1-e)/(1+e),s/2),a+=l=Math.PI/2-2*Math.atan(r*e)-a;return new v(a*i,t.x*i/n)}},wt={__proto__:null,LonLat:yt,Mercator:xt,SphericalMercator:rt},Pt=l({},st,{code:"EPSG:3395",projection:xt,transformation:ht(bt=.5/(Math.PI*xt.R),.5,-bt,.5)}),li=l({},st,{code:"EPSG:4326",projection:yt,transformation:ht(1/180,1,-1/180,.5)}),Lt=l({},ot,{projection:yt,transformation:ht(1,0,-1,0),scale:function(t){return Math.pow(2,t)},zoom:function(t){return Math.log(t)/Math.LN2},distance:function(t,e){var i=e.lng-t.lng,e=e.lat-t.lat;return Math.sqrt(i*i+e*e)},infinite:!0}),o=(ot.Earth=st,ot.EPSG3395=Pt,ot.EPSG3857=lt,ot.EPSG900913=ut,ot.EPSG4326=li,ot.Simple=Lt,it.extend({options:{pane:"overlayPane",attribution:null,bubblingMouseEvents:!0},addTo:function(t){return t.addLayer(this),this},remove:function(){return this.removeFrom(this._map||this._mapToAdd)},removeFrom:function(t){return t&&t.removeLayer(this),this},getPane:function(t){return this._map.getPane(t?this.options[t]||t:this.options.pane)},addInteractiveTarget:function(t){return this._map._targets[h(t)]=this},removeInteractiveTarget:function(t){return delete this._map._targets[h(t)],this},getAttribution:function(){return this.options.attribution},_layerAdd:function(t){var e,i=t.target;i.hasLayer(this)&&(this._map=i,this._zoomAnimated=i._zoomAnimated,this.getEvents&&(e=this.getEvents(),i.on(e,this),this.once("remove",function(){i.off(e,this)},this)),this.onAdd(i),this.fire("add"),i.fire("layeradd",{layer:this}))}})),ui=(A.include({addLayer:function(t){var e;if(t._layerAdd)return e=h(t),this._layers[e]||((this._layers[e]=t)._mapToAdd=this,t.beforeAdd&&t.beforeAdd(this),this.whenReady(t._layerAdd,t)),this;throw new Error("The provided object is not a Layer.")},removeLayer:function(t){var e=h(t);return this._layers[e]&&(this._loaded&&t.onRemove(this),delete this._layers[e],this._loaded&&(this.fire("layerremove",{layer:t}),t.fire("remove")),t._map=t._mapToAdd=null),this},hasLayer:function(t){return h(t)in this._layers},eachLayer:function(t,e){for(var i in this._layers)t.call(e,this._layers[i]);return this},_addLayers:function(t){for(var e=0,i=(t=t?d(t)?t:[t]:[]).length;e<i;e++)this.addLayer(t[e])},_addZoomLimit:function(t){isNaN(t.options.maxZoom)&&isNaN(t.options.minZoom)||(this._zoomBoundLayers[h(t)]=t,this._updateZoomLevels())},_removeZoomLimit:function(t){t=h(t);this._zoomBoundLayers[t]&&(delete this._zoomBoundLayers[t],this._updateZoomLevels())},_updateZoomLevels:function(){var t,e=1/0,i=-1/0,n=this._getZoomSpan();for(t in this._zoomBoundLayers)var o=this._zoomBoundLayers[t].options,e=void 0===o.minZoom?e:Math.min(e,o.minZoom),i=void 0===o.maxZoom?i:Math.max(i,o.maxZoom);this._layersMaxZoom=i===-1/0?void 0:i,this._layersMinZoom=e===1/0?void 0:e,n!==this._getZoomSpan()&&this.fire("zoomlevelschange"),void 0===this.options.maxZoom&&this._layersMaxZoom&&this.getZoom()>this._layersMaxZoom&&this.setZoom(this._layersMaxZoom),void 0===this.options.minZoom&&this._layersMinZoom&&this.getZoom()<this._layersMinZoom&&this.setZoom(this._layersMinZoom)}}),o.extend({initialize:function(t,e){var i,n;if(c(this,e),this._layers={},t)for(i=0,n=t.length;i<n;i++)this.addLayer(t[i])},addLayer:function(t){var e=this.getLayerId(t);return this._layers[e]=t,this._map&&this._map.addLayer(t),this},removeLayer:function(t){t=t in this._layers?t:this.getLayerId(t);return this._map&&this._layers[t]&&this._map.removeLayer(this._layers[t]),delete this._layers[t],this},hasLayer:function(t){return("number"==typeof t?t:this.getLayerId(t))in this._layers},clearLayers:function(){return this.eachLayer(this.removeLayer,this)},invoke:function(t){var e,i,n=Array.prototype.slice.call(arguments,1);for(e in this._layers)(i=this._layers[e])[t]&&i[t].apply(i,n);return this},onAdd:function(t){this.eachLayer(t.addLayer,t)},onRemove:function(t){this.eachLayer(t.removeLayer,t)},eachLayer:function(t,e){for(var i in this._layers)t.call(e,this._layers[i]);return this},getLayer:function(t){return this._layers[t]},getLayers:function(){var t=[];return this.eachLayer(t.push,t),t},setZIndex:function(t){return this.invoke("setZIndex",t)},getLayerId:h})),ci=ui.extend({addLayer:function(t){return this.hasLayer(t)?this:(t.addEventParent(this),ui.prototype.addLayer.call(this,t),this.fire("layeradd",{layer:t}))},removeLayer:function(t){return this.hasLayer(t)?((t=t in this._layers?this._layers[t]:t).removeEventParent(this),ui.prototype.removeLayer.call(this,t),this.fire("layerremove",{layer:t})):this},setStyle:function(t){return this.invoke("setStyle",t)},bringToFront:function(){return this.invoke("bringToFront")},bringToBack:function(){return this.invoke("bringToBack")},getBounds:function(){var t,e=new s;for(t in this._layers){var i=this._layers[t];e.extend(i.getBounds?i.getBounds():i.getLatLng())}return e}}),di=et.extend({options:{popupAnchor:[0,0],tooltipAnchor:[0,0],crossOrigin:!1},initialize:function(t){c(this,t)},createIcon:function(t){return this._createIcon("icon",t)},createShadow:function(t){return this._createIcon("shadow",t)},_createIcon:function(t,e){var i=this._getIconUrl(t);if(i)return i=this._createImg(i,e&&"IMG"===e.tagName?e:null),this._setIconStyles(i,t),!this.options.crossOrigin&&""!==this.options.crossOrigin||(i.crossOrigin=!0===this.options.crossOrigin?"":this.options.crossOrigin),i;if("icon"===t)throw new Error("iconUrl not set in Icon options (see the docs).");return null},_setIconStyles:function(t,e){var i=this.options,n=i[e+"Size"],n=m(n="number"==typeof n?[n,n]:n),o=m("shadow"===e&&i.shadowAnchor||i.iconAnchor||n&&n.divideBy(2,!0));t.className="leaflet-marker-"+e+" "+(i.className||""),o&&(t.style.marginLeft=-o.x+"px",t.style.marginTop=-o.y+"px"),n&&(t.style.width=n.x+"px",t.style.height=n.y+"px")},_createImg:function(t,e){return(e=e||document.createElement("img")).src=t,e},_getIconUrl:function(t){return b.retina&&this.options[t+"RetinaUrl"]||this.options[t+"Url"]}});var _i=di.extend({options:{iconUrl:"marker-icon.png",iconRetinaUrl:"marker-icon-2x.png",shadowUrl:"marker-shadow.png",iconSize:[25,41],iconAnchor:[12,41],popupAnchor:[1,-34],tooltipAnchor:[16,-28],shadowSize:[41,41]},_getIconUrl:function(t){return"string"!=typeof _i.imagePath&&(_i.imagePath=this._detectIconPath()),(this.options.imagePath||_i.imagePath)+di.prototype._getIconUrl.call(this,t)},_stripUrl:function(t){function e(t,e,i){return(e=e.exec(t))&&e[i]}return(t=e(t,/^url\\(([\'"])?(.+)\\1\\)$/,2))&&e(t,/^(.*)marker-icon\\.png$/,1)},_detectIconPath:function(){var t=P("div","leaflet-default-icon-path",document.body),e=pe(t,"background-image")||pe(t,"backgroundImage");return document.body.removeChild(t),(e=this._stripUrl(e))?e:(t=document.querySelector(\'link[href$="leaflet.css"]\'))?t.href.substring(0,t.href.length-"leaflet.css".length-1):""}}),pi=n.extend({initialize:function(t){this._marker=t},addHooks:function(){var t=this._marker._icon;this._draggable||(this._draggable=new Xe(t,t,!0)),this._draggable.on({dragstart:this._onDragStart,predrag:this._onPreDrag,drag:this._onDrag,dragend:this._onDragEnd},this).enable(),M(t,"leaflet-marker-draggable")},removeHooks:function(){this._draggable.off({dragstart:this._onDragStart,predrag:this._onPreDrag,drag:this._onDrag,dragend:this._onDragEnd},this).disable(),this._marker._icon&&z(this._marker._icon,"leaflet-marker-draggable")},moved:function(){return this._draggable&&this._draggable._moved},_adjustPan:function(t){var e=this._marker,i=e._map,n=this._marker.options.autoPanSpeed,o=this._marker.options.autoPanPadding,s=Pe(e._icon),r=i.getPixelBounds(),a=i.getPixelOrigin(),a=_(r.min._subtract(a).add(o),r.max._subtract(a).subtract(o));a.contains(s)||(o=m((Math.max(a.max.x,s.x)-a.max.x)/(r.max.x-a.max.x)-(Math.min(a.min.x,s.x)-a.min.x)/(r.min.x-a.min.x),(Math.max(a.max.y,s.y)-a.max.y)/(r.max.y-a.max.y)-(Math.min(a.min.y,s.y)-a.min.y)/(r.min.y-a.min.y)).multiplyBy(n),i.panBy(o,{animate:!1}),this._draggable._newPos._add(o),this._draggable._startPos._add(o),Z(e._icon,this._draggable._newPos),this._onDrag(t),this._panRequest=x(this._adjustPan.bind(this,t)))},_onDragStart:function(){this._oldLatLng=this._marker.getLatLng(),this._marker.closePopup&&this._marker.closePopup(),this._marker.fire("movestart").fire("dragstart")},_onPreDrag:function(t){this._marker.options.autoPan&&(r(this._panRequest),this._panRequest=x(this._adjustPan.bind(this,t)))},_onDrag:function(t){var e=this._marker,i=e._shadow,n=Pe(e._icon),o=e._map.layerPointToLatLng(n);i&&Z(i,n),e._latlng=o,t.latlng=o,t.oldLatLng=this._oldLatLng,e.fire("move",t).fire("drag",t)},_onDragEnd:function(t){r(this._panRequest),delete this._oldLatLng,this._marker.fire("moveend").fire("dragend",t)}}),mi=o.extend({options:{icon:new _i,interactive:!0,keyboard:!0,title:"",alt:"Marker",zIndexOffset:0,opacity:1,riseOnHover:!1,riseOffset:250,pane:"markerPane",shadowPane:"shadowPane",bubblingMouseEvents:!1,autoPanOnFocus:!0,draggable:!1,autoPan:!1,autoPanPadding:[50,50],autoPanSpeed:10},initialize:function(t,e){c(this,e),this._latlng=w(t)},onAdd:function(t){this._zoomAnimated=this._zoomAnimated&&t.options.markerZoomAnimation,this._zoomAnimated&&t.on("zoomanim",this._animateZoom,this),this._initIcon(),this.update()},onRemove:function(t){this.dragging&&this.dragging.enabled()&&(this.options.draggable=!0,this.dragging.removeHooks()),delete this.dragging,this._zoomAnimated&&t.off("zoomanim",this._animateZoom,this),this._removeIcon(),this._removeShadow()},getEvents:function(){return{zoom:this.update,viewreset:this.update}},getLatLng:function(){return this._latlng},setLatLng:function(t){var e=this._latlng;return this._latlng=w(t),this.update(),this.fire("move",{oldLatLng:e,latlng:this._latlng})},setZIndexOffset:function(t){return this.options.zIndexOffset=t,this.update()},getIcon:function(){return this.options.icon},setIcon:function(t){return this.options.icon=t,this._map&&(this._initIcon(),this.update()),this._popup&&this.bindPopup(this._popup,this._popup.options),this},getElement:function(){return this._icon},update:function(){var t;return this._icon&&this._map&&(t=this._map.latLngToLayerPoint(this._latlng).round(),this._setPos(t)),this},_initIcon:function(){var t=this.options,e="leaflet-zoom-"+(this._zoomAnimated?"animated":"hide"),i=t.icon.createIcon(this._icon),n=!1,i=(i!==this._icon&&(this._icon&&this._removeIcon(),n=!0,t.title&&(i.title=t.title),"IMG"===i.tagName&&(i.alt=t.alt||"")),M(i,e),t.keyboard&&(i.tabIndex="0",i.setAttribute("role","button")),this._icon=i,t.riseOnHover&&this.on({mouseover:this._bringToFront,mouseout:this._resetZIndex}),this.options.autoPanOnFocus&&S(i,"focus",this._panOnFocus,this),t.icon.createShadow(this._shadow)),o=!1;i!==this._shadow&&(this._removeShadow(),o=!0),i&&(M(i,e),i.alt=""),this._shadow=i,t.opacity<1&&this._updateOpacity(),n&&this.getPane().appendChild(this._icon),this._initInteraction(),i&&o&&this.getPane(t.shadowPane).appendChild(this._shadow)},_removeIcon:function(){this.options.riseOnHover&&this.off({mouseover:this._bringToFront,mouseout:this._resetZIndex}),this.options.autoPanOnFocus&&k(this._icon,"focus",this._panOnFocus,this),T(this._icon),this.removeInteractiveTarget(this._icon),this._icon=null},_removeShadow:function(){this._shadow&&T(this._shadow),this._shadow=null},_setPos:function(t){this._icon&&Z(this._icon,t),this._shadow&&Z(this._shadow,t),this._zIndex=t.y+this.options.zIndexOffset,this._resetZIndex()},_updateZIndex:function(t){this._icon&&(this._icon.style.zIndex=this._zIndex+t)},_animateZoom:function(t){t=this._map._latLngToNewLayerPoint(this._latlng,t.zoom,t.center).round();this._setPos(t)},_initInteraction:function(){var t;this.options.interactive&&(M(this._icon,"leaflet-interactive"),this.addInteractiveTarget(this._icon),pi&&(t=this.options.draggable,this.dragging&&(t=this.dragging.enabled(),this.dragging.disable()),this.dragging=new pi(this),t&&this.dragging.enable()))},setOpacity:function(t){return this.options.opacity=t,this._map&&this._updateOpacity(),this},_updateOpacity:function(){var t=this.options.opacity;this._icon&&C(this._icon,t),this._shadow&&C(this._shadow,t)},_bringToFront:function(){this._updateZIndex(this.options.riseOffset)},_resetZIndex:function(){this._updateZIndex(0)},_panOnFocus:function(){var t,e,i=this._map;i&&(t=(e=this.options.icon.options).iconSize?m(e.iconSize):m(0,0),e=e.iconAnchor?m(e.iconAnchor):m(0,0),i.panInside(this._latlng,{paddingTopLeft:e,paddingBottomRight:t.subtract(e)}))},_getPopupAnchor:function(){return this.options.icon.options.popupAnchor},_getTooltipAnchor:function(){return this.options.icon.options.tooltipAnchor}});var fi=o.extend({options:{stroke:!0,color:"#3388ff",weight:3,opacity:1,lineCap:"round",lineJoin:"round",dashArray:null,dashOffset:null,fill:!1,fillColor:null,fillOpacity:.2,fillRule:"evenodd",interactive:!0,bubblingMouseEvents:!0},beforeAdd:function(t){this._renderer=t.getRenderer(this)},onAdd:function(){this._renderer._initPath(this),this._reset(),this._renderer._addPath(this)},onRemove:function(){this._renderer._removePath(this)},redraw:function(){return this._map&&this._renderer._updatePath(this),this},setStyle:function(t){return c(this,t),this._renderer&&(this._renderer._updateStyle(this),this.options.stroke&&t&&Object.prototype.hasOwnProperty.call(t,"weight")&&this._updateBounds()),this},bringToFront:function(){return this._renderer&&this._renderer._bringToFront(this),this},bringToBack:function(){return this._renderer&&this._renderer._bringToBack(this),this},getElement:function(){return this._path},_reset:function(){this._project(),this._update()},_clickTolerance:function(){return(this.options.stroke?this.options.weight/2:0)+(this._renderer.options.tolerance||0)}}),gi=fi.extend({options:{fill:!0,radius:10},initialize:function(t,e){c(this,e),this._latlng=w(t),this._radius=this.options.radius},setLatLng:function(t){var e=this._latlng;return this._latlng=w(t),this.redraw(),this.fire("move",{oldLatLng:e,latlng:this._latlng})},getLatLng:function(){return this._latlng},setRadius:function(t){return this.options.radius=this._radius=t,this.redraw()},getRadius:function(){return this._radius},setStyle:function(t){var e=t&&t.radius||this._radius;return fi.prototype.setStyle.call(this,t),this.setRadius(e),this},_project:function(){this._point=this._map.latLngToLayerPoint(this._latlng),this._updateBounds()},_updateBounds:function(){var t=this._radius,e=this._radiusY||t,i=this._clickTolerance(),t=[t+i,e+i];this._pxBounds=new f(this._point.subtract(t),this._point.add(t))},_update:function(){this._map&&this._updatePath()},_updatePath:function(){this._renderer._updateCircle(this)},_empty:function(){return this._radius&&!this._renderer._bounds.intersects(this._pxBounds)},_containsPoint:function(t){return t.distanceTo(this._point)<=this._radius+this._clickTolerance()}});var vi=gi.extend({initialize:function(t,e,i){if(c(this,e="number"==typeof e?l({},i,{radius:e}):e),this._latlng=w(t),isNaN(this.options.radius))throw new Error("Circle radius cannot be NaN");this._mRadius=this.options.radius},setRadius:function(t){return this._mRadius=t,this.redraw()},getRadius:function(){return this._mRadius},getBounds:function(){var t=[this._radius,this._radiusY||this._radius];return new s(this._map.layerPointToLatLng(this._point.subtract(t)),this._map.layerPointToLatLng(this._point.add(t)))},setStyle:fi.prototype.setStyle,_project:function(){var t,e,i,n,o,s=this._latlng.lng,r=this._latlng.lat,a=this._map,h=a.options.crs;h.distance===st.distance?(n=Math.PI/180,o=this._mRadius/st.R/n,t=a.project([r+o,s]),e=a.project([r-o,s]),e=t.add(e).divideBy(2),i=a.unproject(e).lat,n=Math.acos((Math.cos(o*n)-Math.sin(r*n)*Math.sin(i*n))/(Math.cos(r*n)*Math.cos(i*n)))/n,!isNaN(n)&&0!==n||(n=o/Math.cos(Math.PI/180*r)),this._point=e.subtract(a.getPixelOrigin()),this._radius=isNaN(n)?0:e.x-a.project([i,s-n]).x,this._radiusY=e.y-t.y):(o=h.unproject(h.project(this._latlng).subtract([this._mRadius,0])),this._point=a.latLngToLayerPoint(this._latlng),this._radius=this._point.x-a.latLngToLayerPoint(o).x),this._updateBounds()}});var yi=fi.extend({options:{smoothFactor:1,noClip:!1},initialize:function(t,e){c(this,e),this._setLatLngs(t)},getLatLngs:function(){return this._latlngs},setLatLngs:function(t){return this._setLatLngs(t),this.redraw()},isEmpty:function(){return!this._latlngs.length},closestLayerPoint:function(t){for(var e=1/0,i=null,n=ri,o=0,s=this._parts.length;o<s;o++)for(var r=this._parts[o],a=1,h=r.length;a<h;a++){var l,u,c=n(t,l=r[a-1],u=r[a],!0);c<e&&(e=c,i=n(t,l,u))}return i&&(i.distance=Math.sqrt(e)),i},getCenter:function(){if(this._map)return hi(this._defaultShape(),this._map.options.crs);throw new Error("Must add layer to map before using getCenter()")},getBounds:function(){return this._bounds},addLatLng:function(t,e){return e=e||this._defaultShape(),t=w(t),e.push(t),this._bounds.extend(t),this.redraw()},_setLatLngs:function(t){this._bounds=new s,this._latlngs=this._convertLatLngs(t)},_defaultShape:function(){return I(this._latlngs)?this._latlngs:this._latlngs[0]},_convertLatLngs:function(t){for(var e=[],i=I(t),n=0,o=t.length;n<o;n++)i?(e[n]=w(t[n]),this._bounds.extend(e[n])):e[n]=this._convertLatLngs(t[n]);return e},_project:function(){var t=new f;this._rings=[],this._projectLatlngs(this._latlngs,this._rings,t),this._bounds.isValid()&&t.isValid()&&(this._rawPxBounds=t,this._updateBounds())},_updateBounds:function(){var t=this._clickTolerance(),t=new p(t,t);this._rawPxBounds&&(this._pxBounds=new f([this._rawPxBounds.min.subtract(t),this._rawPxBounds.max.add(t)]))},_projectLatlngs:function(t,e,i){var n,o,s=t[0]instanceof v,r=t.length;if(s){for(o=[],n=0;n<r;n++)o[n]=this._map.latLngToLayerPoint(t[n]),i.extend(o[n]);e.push(o)}else for(n=0;n<r;n++)this._projectLatlngs(t[n],e,i)},_clipPoints:function(){var t=this._renderer._bounds;if(this._parts=[],this._pxBounds&&this._pxBounds.intersects(t))if(this.options.noClip)this._parts=this._rings;else for(var e,i,n,o,s=this._parts,r=0,a=0,h=this._rings.length;r<h;r++)for(e=0,i=(o=this._rings[r]).length;e<i-1;e++)(n=ni(o[e],o[e+1],t,e,!0))&&(s[a]=s[a]||[],s[a].push(n[0]),n[1]===o[e+1]&&e!==i-2||(s[a].push(n[1]),a++))},_simplifyPoints:function(){for(var t=this._parts,e=this.options.smoothFactor,i=0,n=t.length;i<n;i++)t[i]=ei(t[i],e)},_update:function(){this._map&&(this._clipPoints(),this._simplifyPoints(),this._updatePath())},_updatePath:function(){this._renderer._updatePoly(this)},_containsPoint:function(t,e){var i,n,o,s,r,a,h=this._clickTolerance();if(this._pxBounds&&this._pxBounds.contains(t))for(i=0,s=this._parts.length;i<s;i++)for(n=0,o=(r=(a=this._parts[i]).length)-1;n<r;o=n++)if((e||0!==n)&&ii(t,a[o],a[n])<=h)return!0;return!1}});yi._flat=ai;var xi=yi.extend({options:{fill:!0},isEmpty:function(){return!this._latlngs.length||!this._latlngs[0].length},getCenter:function(){if(this._map)return $e(this._defaultShape(),this._map.options.crs);throw new Error("Must add layer to map before using getCenter()")},_convertLatLngs:function(t){var t=yi.prototype._convertLatLngs.call(this,t),e=t.length;return 2<=e&&t[0]instanceof v&&t[0].equals(t[e-1])&&t.pop(),t},_setLatLngs:function(t){yi.prototype._setLatLngs.call(this,t),I(this._latlngs)&&(this._latlngs=[this._latlngs])},_defaultShape:function(){return(I(this._latlngs[0])?this._latlngs:this._latlngs[0])[0]},_clipPoints:function(){var t=this._renderer._bounds,e=this.options.weight,e=new p(e,e),t=new f(t.min.subtract(e),t.max.add(e));if(this._parts=[],this._pxBounds&&this._pxBounds.intersects(t))if(this.options.noClip)this._parts=this._rings;else for(var i,n=0,o=this._rings.length;n<o;n++)(i=Je(this._rings[n],t,!0)).length&&this._parts.push(i)},_updatePath:function(){this._renderer._updatePoly(this,!0)},_containsPoint:function(t){var e,i,n,o,s,r,a,h,l=!1;if(!this._pxBounds||!this._pxBounds.contains(t))return!1;for(o=0,a=this._parts.length;o<a;o++)for(s=0,r=(h=(e=this._parts[o]).length)-1;s<h;r=s++)i=e[s],n=e[r],i.y>t.y!=n.y>t.y&&t.x<(n.x-i.x)*(t.y-i.y)/(n.y-i.y)+i.x&&(l=!l);return l||yi.prototype._containsPoint.call(this,t,!0)}});var wi=ci.extend({initialize:function(t,e){c(this,e),this._layers={},t&&this.addData(t)},addData:function(t){var e,i,n,o=d(t)?t:t.features;if(o){for(e=0,i=o.length;e<i;e++)((n=o[e]).geometries||n.geometry||n.features||n.coordinates)&&this.addData(n);return this}var s,r=this.options;return(!r.filter||r.filter(t))&&(s=bi(t,r))?(s.feature=Zi(t),s.defaultOptions=s.options,this.resetStyle(s),r.onEachFeature&&r.onEachFeature(t,s),this.addLayer(s)):this},resetStyle:function(t){return void 0===t?this.eachLayer(this.resetStyle,this):(t.options=l({},t.defaultOptions),this._setLayerStyle(t,this.options.style),this)},setStyle:function(e){return this.eachLayer(function(t){this._setLayerStyle(t,e)},this)},_setLayerStyle:function(t,e){t.setStyle&&("function"==typeof e&&(e=e(t.feature)),t.setStyle(e))}});function bi(t,e){var i,n,o,s,r="Feature"===t.type?t.geometry:t,a=r?r.coordinates:null,h=[],l=e&&e.pointToLayer,u=e&&e.coordsToLatLng||Li;if(!a&&!r)return null;switch(r.type){case"Point":return Pi(l,t,i=u(a),e);case"MultiPoint":for(o=0,s=a.length;o<s;o++)i=u(a[o]),h.push(Pi(l,t,i,e));return new ci(h);case"LineString":case"MultiLineString":return n=Ti(a,"LineString"===r.type?0:1,u),new yi(n,e);case"Polygon":case"MultiPolygon":return n=Ti(a,"Polygon"===r.type?1:2,u),new xi(n,e);case"GeometryCollection":for(o=0,s=r.geometries.length;o<s;o++){var c=bi({geometry:r.geometries[o],type:"Feature",properties:t.properties},e);c&&h.push(c)}return new ci(h);case"FeatureCollection":for(o=0,s=r.features.length;o<s;o++){var d=bi(r.features[o],e);d&&h.push(d)}return new ci(h);default:throw new Error("Invalid GeoJSON object.")}}function Pi(t,e,i,n){return t?t(e,i):new mi(i,n&&n.markersInheritOptions&&n)}function Li(t){return new v(t[1],t[0],t[2])}function Ti(t,e,i){for(var n,o=[],s=0,r=t.length;s<r;s++)n=e?Ti(t[s],e-1,i):(i||Li)(t[s]),o.push(n);return o}function Mi(t,e){return void 0!==(t=w(t)).alt?[i(t.lng,e),i(t.lat,e),i(t.alt,e)]:[i(t.lng,e),i(t.lat,e)]}function zi(t,e,i,n){for(var o=[],s=0,r=t.length;s<r;s++)o.push(e?zi(t[s],I(t[s])?0:e-1,i,n):Mi(t[s],n));return!e&&i&&0<o.length&&o.push(o[0].slice()),o}function Ci(t,e){return t.feature?l({},t.feature,{geometry:e}):Zi(e)}function Zi(t){return"Feature"===t.type||"FeatureCollection"===t.type?t:{type:"Feature",properties:{},geometry:t}}Tt={toGeoJSON:function(t){return Ci(this,{type:"Point",coordinates:Mi(this.getLatLng(),t)})}};function Si(t,e){return new wi(t,e)}mi.include(Tt),vi.include(Tt),gi.include(Tt),yi.include({toGeoJSON:function(t){var e=!I(this._latlngs);return Ci(this,{type:(e?"Multi":"")+"LineString",coordinates:zi(this._latlngs,e?1:0,!1,t)})}}),xi.include({toGeoJSON:function(t){var e=!I(this._latlngs),i=e&&!I(this._latlngs[0]),t=zi(this._latlngs,i?2:e?1:0,!0,t);return Ci(this,{type:(i?"Multi":"")+"Polygon",coordinates:t=e?t:[t]})}}),ui.include({toMultiPoint:function(e){var i=[];return this.eachLayer(function(t){i.push(t.toGeoJSON(e).geometry.coordinates)}),Ci(this,{type:"MultiPoint",coordinates:i})},toGeoJSON:function(e){var i,n,t=this.feature&&this.feature.geometry&&this.feature.geometry.type;return"MultiPoint"===t?this.toMultiPoint(e):(i="GeometryCollection"===t,n=[],this.eachLayer(function(t){t.toGeoJSON&&(t=t.toGeoJSON(e),i?n.push(t.geometry):"FeatureCollection"===(t=Zi(t)).type?n.push.apply(n,t.features):n.push(t))}),i?Ci(this,{geometries:n,type:"GeometryCollection"}):{type:"FeatureCollection",features:n})}});var Mt=Si,Ei=o.extend({options:{opacity:1,alt:"",interactive:!1,crossOrigin:!1,errorOverlayUrl:"",zIndex:1,className:""},initialize:function(t,e,i){this._url=t,this._bounds=g(e),c(this,i)},onAdd:function(){this._image||(this._initImage(),this.options.opacity<1&&this._updateOpacity()),this.options.interactive&&(M(this._image,"leaflet-interactive"),this.addInteractiveTarget(this._image)),this.getPane().appendChild(this._image),this._reset()},onRemove:function(){T(this._image),this.options.interactive&&this.removeInteractiveTarget(this._image)},setOpacity:function(t){return this.options.opacity=t,this._image&&this._updateOpacity(),this},setStyle:function(t){return t.opacity&&this.setOpacity(t.opacity),this},bringToFront:function(){return this._map&&fe(this._image),this},bringToBack:function(){return this._map&&ge(this._image),this},setUrl:function(t){return this._url=t,this._image&&(this._image.src=t),this},setBounds:function(t){return this._bounds=g(t),this._map&&this._reset(),this},getEvents:function(){var t={zoom:this._reset,viewreset:this._reset};return this._zoomAnimated&&(t.zoomanim=this._animateZoom),t},setZIndex:function(t){return this.options.zIndex=t,this._updateZIndex(),this},getBounds:function(){return this._bounds},getElement:function(){return this._image},_initImage:function(){var t="IMG"===this._url.tagName,e=this._image=t?this._url:P("img");M(e,"leaflet-image-layer"),this._zoomAnimated&&M(e,"leaflet-zoom-animated"),this.options.className&&M(e,this.options.className),e.onselectstart=u,e.onmousemove=u,e.onload=a(this.fire,this,"load"),e.onerror=a(this._overlayOnError,this,"error"),!this.options.crossOrigin&&""!==this.options.crossOrigin||(e.crossOrigin=!0===this.options.crossOrigin?"":this.options.crossOrigin),this.options.zIndex&&this._updateZIndex(),t?this._url=e.src:(e.src=this._url,e.alt=this.options.alt)},_animateZoom:function(t){var e=this._map.getZoomScale(t.zoom),t=this._map._latLngBoundsToNewLayerBounds(this._bounds,t.zoom,t.center).min;be(this._image,t,e)},_reset:function(){var t=this._image,e=new f(this._map.latLngToLayerPoint(this._bounds.getNorthWest()),this._map.latLngToLayerPoint(this._bounds.getSouthEast())),i=e.getSize();Z(t,e.min),t.style.width=i.x+"px",t.style.height=i.y+"px"},_updateOpacity:function(){C(this._image,this.options.opacity)},_updateZIndex:function(){this._image&&void 0!==this.options.zIndex&&null!==this.options.zIndex&&(this._image.style.zIndex=this.options.zIndex)},_overlayOnError:function(){this.fire("error");var t=this.options.errorOverlayUrl;t&&this._url!==t&&(this._url=t,this._image.src=t)},getCenter:function(){return this._bounds.getCenter()}}),ki=Ei.extend({options:{autoplay:!0,loop:!0,keepAspectRatio:!0,muted:!1,playsInline:!0},_initImage:function(){var t="VIDEO"===this._url.tagName,e=this._image=t?this._url:P("video");if(M(e,"leaflet-image-layer"),this._zoomAnimated&&M(e,"leaflet-zoom-animated"),this.options.className&&M(e,this.options.className),e.onselectstart=u,e.onmousemove=u,e.onloadeddata=a(this.fire,this,"load"),t){for(var i=e.getElementsByTagName("source"),n=[],o=0;o<i.length;o++)n.push(i[o].src);this._url=0<i.length?n:[e.src]}else{d(this._url)||(this._url=[this._url]),!this.options.keepAspectRatio&&Object.prototype.hasOwnProperty.call(e.style,"objectFit")&&(e.style.objectFit="fill"),e.autoplay=!!this.options.autoplay,e.loop=!!this.options.loop,e.muted=!!this.options.muted,e.playsInline=!!this.options.playsInline;for(var s=0;s<this._url.length;s++){var r=P("source");r.src=this._url[s],e.appendChild(r)}}}});var Oi=Ei.extend({_initImage:function(){var t=this._image=this._url;M(t,"leaflet-image-layer"),this._zoomAnimated&&M(t,"leaflet-zoom-animated"),this.options.className&&M(t,this.options.className),t.onselectstart=u,t.onmousemove=u}});var Ai=o.extend({options:{interactive:!1,offset:[0,0],className:"",pane:void 0,content:""},initialize:function(t,e){t&&(t instanceof v||d(t))?(this._latlng=w(t),c(this,e)):(c(this,t),this._source=e),this.options.content&&(this._content=this.options.content)},openOn:function(t){return(t=arguments.length?t:this._source._map).hasLayer(this)||t.addLayer(this),this},close:function(){return this._map&&this._map.removeLayer(this),this},toggle:function(t){return this._map?this.close():(arguments.length?this._source=t:t=this._source,this._prepareOpen(),this.openOn(t._map)),this},onAdd:function(t){this._zoomAnimated=t._zoomAnimated,this._container||this._initLayout(),t._fadeAnimated&&C(this._container,0),clearTimeout(this._removeTimeout),this.getPane().appendChild(this._container),this.update(),t._fadeAnimated&&C(this._container,1),this.bringToFront(),this.options.interactive&&(M(this._container,"leaflet-interactive"),this.addInteractiveTarget(this._container))},onRemove:function(t){t._fadeAnimated?(C(this._container,0),this._removeTimeout=setTimeout(a(T,void 0,this._container),200)):T(this._container),this.options.interactive&&(z(this._container,"leaflet-interactive"),this.removeInteractiveTarget(this._container))},getLatLng:function(){return this._latlng},setLatLng:function(t){return this._latlng=w(t),this._map&&(this._updatePosition(),this._adjustPan()),this},getContent:function(){return this._content},setContent:function(t){return this._content=t,this.update(),this},getElement:function(){return this._container},update:function(){this._map&&(this._container.style.visibility="hidden",this._updateContent(),this._updateLayout(),this._updatePosition(),this._container.style.visibility="",this._adjustPan())},getEvents:function(){var t={zoom:this._updatePosition,viewreset:this._updatePosition};return this._zoomAnimated&&(t.zoomanim=this._animateZoom),t},isOpen:function(){return!!this._map&&this._map.hasLayer(this)},bringToFront:function(){return this._map&&fe(this._container),this},bringToBack:function(){return this._map&&ge(this._container),this},_prepareOpen:function(t){if(!(i=this._source)._map)return!1;if(i instanceof ci){var e,i=null,n=this._source._layers;for(e in n)if(n[e]._map){i=n[e];break}if(!i)return!1;this._source=i}if(!t)if(i.getCenter)t=i.getCenter();else if(i.getLatLng)t=i.getLatLng();else{if(!i.getBounds)throw new Error("Unable to get source layer LatLng.");t=i.getBounds().getCenter()}return this.setLatLng(t),this._map&&this.update(),!0},_updateContent:function(){if(this._content){var t=this._contentNode,e="function"==typeof this._content?this._content(this._source||this):this._content;if("string"==typeof e)t.innerHTML=e;else{for(;t.hasChildNodes();)t.removeChild(t.firstChild);t.appendChild(e)}this.fire("contentupdate")}},_updatePosition:function(){var t,e,i;this._map&&(e=this._map.latLngToLayerPoint(this._latlng),t=m(this.options.offset),i=this._getAnchor(),this._zoomAnimated?Z(this._container,e.add(i)):t=t.add(e).add(i),e=this._containerBottom=-t.y,i=this._containerLeft=-Math.round(this._containerWidth/2)+t.x,this._container.style.bottom=e+"px",this._container.style.left=i+"px")},_getAnchor:function(){return[0,0]}}),Bi=(A.include({_initOverlay:function(t,e,i,n){var o=e;return o instanceof t||(o=new t(n).setContent(e)),i&&o.setLatLng(i),o}}),o.include({_initOverlay:function(t,e,i,n){var o=i;return o instanceof t?(c(o,n),o._source=this):(o=e&&!n?e:new t(n,this)).setContent(i),o}}),Ai.extend({options:{pane:"popupPane",offset:[0,7],maxWidth:300,minWidth:50,maxHeight:null,autoPan:!0,autoPanPaddingTopLeft:null,autoPanPaddingBottomRight:null,autoPanPadding:[5,5],keepInView:!1,closeButton:!0,autoClose:!0,closeOnEscapeKey:!0,className:""},openOn:function(t){return!(t=arguments.length?t:this._source._map).hasLayer(this)&&t._popup&&t._popup.options.autoClose&&t.removeLayer(t._popup),t._popup=this,Ai.prototype.openOn.call(this,t)},onAdd:function(t){Ai.prototype.onAdd.call(this,t),t.fire("popupopen",{popup:this}),this._source&&(this._source.fire("popupopen",{popup:this},!0),this._source instanceof fi||this._source.on("preclick",Ae))},onRemove:function(t){Ai.prototype.onRemove.call(this,t),t.fire("popupclose",{popup:this}),this._source&&(this._source.fire("popupclose",{popup:this},!0),this._source instanceof fi||this._source.off("preclick",Ae))},getEvents:function(){var t=Ai.prototype.getEvents.call(this);return(void 0!==this.options.closeOnClick?this.options.closeOnClick:this._map.options.closePopupOnClick)&&(t.preclick=this.close),this.options.keepInView&&(t.moveend=this._adjustPan),t},_initLayout:function(){var t="leaflet-popup",e=this._container=P("div",t+" "+(this.options.className||"")+" leaflet-zoom-animated"),i=this._wrapper=P("div",t+"-content-wrapper",e);this._contentNode=P("div",t+"-content",i),Ie(e),Be(this._contentNode),S(e,"contextmenu",Ae),this._tipContainer=P("div",t+"-tip-container",e),this._tip=P("div",t+"-tip",this._tipContainer),this.options.closeButton&&((i=this._closeButton=P("a",t+"-close-button",e)).setAttribute("role","button"),i.setAttribute("aria-label","Close popup"),i.href="#close",i.innerHTML=\'<span aria-hidden="true">&#215;</span>\',S(i,"click",function(t){O(t),this.close()},this))},_updateLayout:function(){var t=this._contentNode,e=t.style,i=(e.width="",e.whiteSpace="nowrap",t.offsetWidth),i=Math.min(i,this.options.maxWidth),i=(i=Math.max(i,this.options.minWidth),e.width=i+1+"px",e.whiteSpace="",e.height="",t.offsetHeight),n=this.options.maxHeight,o="leaflet-popup-scrolled";(n&&n<i?(e.height=n+"px",M):z)(t,o),this._containerWidth=this._container.offsetWidth},_animateZoom:function(t){var t=this._map._latLngToNewLayerPoint(this._latlng,t.zoom,t.center),e=this._getAnchor();Z(this._container,t.add(e))},_adjustPan:function(){var t,e,i,n,o,s,r,a;this.options.autoPan&&(this._map._panAnim&&this._map._panAnim.stop(),this._autopanning?this._autopanning=!1:(t=this._map,e=parseInt(pe(this._container,"marginBottom"),10)||0,e=this._container.offsetHeight+e,a=this._containerWidth,(i=new p(this._containerLeft,-e-this._containerBottom))._add(Pe(this._container)),i=t.layerPointToContainerPoint(i),o=m(this.options.autoPanPadding),n=m(this.options.autoPanPaddingTopLeft||o),o=m(this.options.autoPanPaddingBottomRight||o),s=t.getSize(),r=0,i.x+a+o.x>s.x&&(r=i.x+a-s.x+o.x),i.x-r-n.x<(a=0)&&(r=i.x-n.x),i.y+e+o.y>s.y&&(a=i.y+e-s.y+o.y),i.y-a-n.y<0&&(a=i.y-n.y),(r||a)&&(this.options.keepInView&&(this._autopanning=!0),t.fire("autopanstart").panBy([r,a]))))},_getAnchor:function(){return m(this._source&&this._source._getPopupAnchor?this._source._getPopupAnchor():[0,0])}})),Ii=(A.mergeOptions({closePopupOnClick:!0}),A.include({openPopup:function(t,e,i){return this._initOverlay(Bi,t,e,i).openOn(this),this},closePopup:function(t){return(t=arguments.length?t:this._popup)&&t.close(),this}}),o.include({bindPopup:function(t,e){return this._popup=this._initOverlay(Bi,this._popup,t,e),this._popupHandlersAdded||(this.on({click:this._openPopup,keypress:this._onKeyPress,remove:this.closePopup,move:this._movePopup}),this._popupHandlersAdded=!0),this},unbindPopup:function(){return this._popup&&(this.off({click:this._openPopup,keypress:this._onKeyPress,remove:this.closePopup,move:this._movePopup}),this._popupHandlersAdded=!1,this._popup=null),this},openPopup:function(t){return this._popup&&(this instanceof ci||(this._popup._source=this),this._popup._prepareOpen(t||this._latlng)&&this._popup.openOn(this._map)),this},closePopup:function(){return this._popup&&this._popup.close(),this},togglePopup:function(){return this._popup&&this._popup.toggle(this),this},isPopupOpen:function(){return!!this._popup&&this._popup.isOpen()},setPopupContent:function(t){return this._popup&&this._popup.setContent(t),this},getPopup:function(){return this._popup},_openPopup:function(t){var e;this._popup&&this._map&&(Re(t),e=t.layer||t.target,this._popup._source!==e||e instanceof fi?(this._popup._source=e,this.openPopup(t.latlng)):this._map.hasLayer(this._popup)?this.closePopup():this.openPopup(t.latlng))},_movePopup:function(t){this._popup.setLatLng(t.latlng)},_onKeyPress:function(t){13===t.originalEvent.keyCode&&this._openPopup(t)}}),Ai.extend({options:{pane:"tooltipPane",offset:[0,0],direction:"auto",permanent:!1,sticky:!1,opacity:.9},onAdd:function(t){Ai.prototype.onAdd.call(this,t),this.setOpacity(this.options.opacity),t.fire("tooltipopen",{tooltip:this}),this._source&&(this.addEventParent(this._source),this._source.fire("tooltipopen",{tooltip:this},!0))},onRemove:function(t){Ai.prototype.onRemove.call(this,t),t.fire("tooltipclose",{tooltip:this}),this._source&&(this.removeEventParent(this._source),this._source.fire("tooltipclose",{tooltip:this},!0))},getEvents:function(){var t=Ai.prototype.getEvents.call(this);return this.options.permanent||(t.preclick=this.close),t},_initLayout:function(){var t="leaflet-tooltip "+(this.options.className||"")+" leaflet-zoom-"+(this._zoomAnimated?"animated":"hide");this._contentNode=this._container=P("div",t),this._container.setAttribute("role","tooltip"),this._container.setAttribute("id","leaflet-tooltip-"+h(this))},_updateLayout:function(){},_adjustPan:function(){},_setPosition:function(t){var e,i=this._map,n=this._container,o=i.latLngToContainerPoint(i.getCenter()),i=i.layerPointToContainerPoint(t),s=this.options.direction,r=n.offsetWidth,a=n.offsetHeight,h=m(this.options.offset),l=this._getAnchor(),i="top"===s?(e=r/2,a):"bottom"===s?(e=r/2,0):(e="center"===s?r/2:"right"===s?0:"left"===s?r:i.x<o.x?(s="right",0):(s="left",r+2*(h.x+l.x)),a/2);t=t.subtract(m(e,i,!0)).add(h).add(l),z(n,"leaflet-tooltip-right"),z(n,"leaflet-tooltip-left"),z(n,"leaflet-tooltip-top"),z(n,"leaflet-tooltip-bottom"),M(n,"leaflet-tooltip-"+s),Z(n,t)},_updatePosition:function(){var t=this._map.latLngToLayerPoint(this._latlng);this._setPosition(t)},setOpacity:function(t){this.options.opacity=t,this._container&&C(this._container,t)},_animateZoom:function(t){t=this._map._latLngToNewLayerPoint(this._latlng,t.zoom,t.center);this._setPosition(t)},_getAnchor:function(){return m(this._source&&this._source._getTooltipAnchor&&!this.options.sticky?this._source._getTooltipAnchor():[0,0])}})),Ri=(A.include({openTooltip:function(t,e,i){return this._initOverlay(Ii,t,e,i).openOn(this),this},closeTooltip:function(t){return t.close(),this}}),o.include({bindTooltip:function(t,e){return this._tooltip&&this.isTooltipOpen()&&this.unbindTooltip(),this._tooltip=this._initOverlay(Ii,this._tooltip,t,e),this._initTooltipInteractions(),this._tooltip.options.permanent&&this._map&&this._map.hasLayer(this)&&this.openTooltip(),this},unbindTooltip:function(){return this._tooltip&&(this._initTooltipInteractions(!0),this.closeTooltip(),this._tooltip=null),this},_initTooltipInteractions:function(t){var e,i;!t&&this._tooltipHandlersAdded||(e=t?"off":"on",i={remove:this.closeTooltip,move:this._moveTooltip},this._tooltip.options.permanent?i.add=this._openTooltip:(i.mouseover=this._openTooltip,i.mouseout=this.closeTooltip,i.click=this._openTooltip,this._map?this._addFocusListeners():i.add=this._addFocusListeners),this._tooltip.options.sticky&&(i.mousemove=this._moveTooltip),this[e](i),this._tooltipHandlersAdded=!t)},openTooltip:function(t){return this._tooltip&&(this instanceof ci||(this._tooltip._source=this),this._tooltip._prepareOpen(t)&&(this._tooltip.openOn(this._map),this.getElement?this._setAriaDescribedByOnLayer(this):this.eachLayer&&this.eachLayer(this._setAriaDescribedByOnLayer,this))),this},closeTooltip:function(){if(this._tooltip)return this._tooltip.close()},toggleTooltip:function(){return this._tooltip&&this._tooltip.toggle(this),this},isTooltipOpen:function(){return this._tooltip.isOpen()},setTooltipContent:function(t){return this._tooltip&&this._tooltip.setContent(t),this},getTooltip:function(){return this._tooltip},_addFocusListeners:function(){this.getElement?this._addFocusListenersOnLayer(this):this.eachLayer&&this.eachLayer(this._addFocusListenersOnLayer,this)},_addFocusListenersOnLayer:function(t){var e="function"==typeof t.getElement&&t.getElement();e&&(S(e,"focus",function(){this._tooltip._source=t,this.openTooltip()},this),S(e,"blur",this.closeTooltip,this))},_setAriaDescribedByOnLayer:function(t){t="function"==typeof t.getElement&&t.getElement();t&&t.setAttribute("aria-describedby",this._tooltip._container.id)},_openTooltip:function(t){var e;this._tooltip&&this._map&&(this._map.dragging&&this._map.dragging.moving()&&!this._openOnceFlag?(this._openOnceFlag=!0,(e=this)._map.once("moveend",function(){e._openOnceFlag=!1,e._openTooltip(t)})):(this._tooltip._source=t.layer||t.target,this.openTooltip(this._tooltip.options.sticky?t.latlng:void 0)))},_moveTooltip:function(t){var e=t.latlng;this._tooltip.options.sticky&&t.originalEvent&&(t=this._map.mouseEventToContainerPoint(t.originalEvent),t=this._map.containerPointToLayerPoint(t),e=this._map.layerPointToLatLng(t)),this._tooltip.setLatLng(e)}}),di.extend({options:{iconSize:[12,12],html:!1,bgPos:null,className:"leaflet-div-icon"},createIcon:function(t){var t=t&&"DIV"===t.tagName?t:document.createElement("div"),e=this.options;return e.html instanceof Element?(me(t),t.appendChild(e.html)):t.innerHTML=!1!==e.html?e.html:"",e.bgPos&&(e=m(e.bgPos),t.style.backgroundPosition=-e.x+"px "+-e.y+"px"),this._setIconStyles(t,"icon"),t},createShadow:function(){return null}}));di.Default=_i;var Ni=o.extend({options:{tileSize:256,opacity:1,updateWhenIdle:b.mobile,updateWhenZooming:!0,updateInterval:200,zIndex:1,bounds:null,minZoom:0,maxZoom:void 0,maxNativeZoom:void 0,minNativeZoom:void 0,noWrap:!1,pane:"tilePane",className:"",keepBuffer:2},initialize:function(t){c(this,t)},onAdd:function(){this._initContainer(),this._levels={},this._tiles={},this._resetView()},beforeAdd:function(t){t._addZoomLimit(this)},onRemove:function(t){this._removeAllTiles(),T(this._container),t._removeZoomLimit(this),this._container=null,this._tileZoom=void 0},bringToFront:function(){return this._map&&(fe(this._container),this._setAutoZIndex(Math.max)),this},bringToBack:function(){return this._map&&(ge(this._container),this._setAutoZIndex(Math.min)),this},getContainer:function(){return this._container},setOpacity:function(t){return this.options.opacity=t,this._updateOpacity(),this},setZIndex:function(t){return this.options.zIndex=t,this._updateZIndex(),this},isLoading:function(){return this._loading},redraw:function(){var t;return this._map&&(this._removeAllTiles(),(t=this._clampZoom(this._map.getZoom()))!==this._tileZoom&&(this._tileZoom=t,this._updateLevels()),this._update()),this},getEvents:function(){var t={viewprereset:this._invalidateAll,viewreset:this._resetView,zoom:this._resetView,moveend:this._onMoveEnd};return this.options.updateWhenIdle||(this._onMove||(this._onMove=j(this._onMoveEnd,this.options.updateInterval,this)),t.move=this._onMove),this._zoomAnimated&&(t.zoomanim=this._animateZoom),t},createTile:function(){return document.createElement("div")},getTileSize:function(){var t=this.options.tileSize;return t instanceof p?t:new p(t,t)},_updateZIndex:function(){this._container&&void 0!==this.options.zIndex&&null!==this.options.zIndex&&(this._container.style.zIndex=this.options.zIndex)},_setAutoZIndex:function(t){for(var e,i=this.getPane().children,n=-t(-1/0,1/0),o=0,s=i.length;o<s;o++)e=i[o].style.zIndex,i[o]!==this._container&&e&&(n=t(n,+e));isFinite(n)&&(this.options.zIndex=n+t(-1,1),this._updateZIndex())},_updateOpacity:function(){if(this._map&&!b.ielt9){C(this._container,this.options.opacity);var t,e=+new Date,i=!1,n=!1;for(t in this._tiles){var o,s=this._tiles[t];s.current&&s.loaded&&(o=Math.min(1,(e-s.loaded)/200),C(s.el,o),o<1?i=!0:(s.active?n=!0:this._onOpaqueTile(s),s.active=!0))}n&&!this._noPrune&&this._pruneTiles(),i&&(r(this._fadeFrame),this._fadeFrame=x(this._updateOpacity,this))}},_onOpaqueTile:u,_initContainer:function(){this._container||(this._container=P("div","leaflet-layer "+(this.options.className||"")),this._updateZIndex(),this.options.opacity<1&&this._updateOpacity(),this.getPane().appendChild(this._container))},_updateLevels:function(){var t=this._tileZoom,e=this.options.maxZoom;if(void 0!==t){for(var i in this._levels)i=Number(i),this._levels[i].el.children.length||i===t?(this._levels[i].el.style.zIndex=e-Math.abs(t-i),this._onUpdateLevel(i)):(T(this._levels[i].el),this._removeTilesAtZoom(i),this._onRemoveLevel(i),delete this._levels[i]);var n=this._levels[t],o=this._map;return n||((n=this._levels[t]={}).el=P("div","leaflet-tile-container leaflet-zoom-animated",this._container),n.el.style.zIndex=e,n.origin=o.project(o.unproject(o.getPixelOrigin()),t).round(),n.zoom=t,this._setZoomTransform(n,o.getCenter(),o.getZoom()),u(n.el.offsetWidth),this._onCreateLevel(n)),this._level=n}},_onUpdateLevel:u,_onRemoveLevel:u,_onCreateLevel:u,_pruneTiles:function(){if(this._map){var t,e,i,n=this._map.getZoom();if(n>this.options.maxZoom||n<this.options.minZoom)this._removeAllTiles();else{for(t in this._tiles)(i=this._tiles[t]).retain=i.current;for(t in this._tiles)(i=this._tiles[t]).current&&!i.active&&(e=i.coords,this._retainParent(e.x,e.y,e.z,e.z-5)||this._retainChildren(e.x,e.y,e.z,e.z+2));for(t in this._tiles)this._tiles[t].retain||this._removeTile(t)}}},_removeTilesAtZoom:function(t){for(var e in this._tiles)this._tiles[e].coords.z===t&&this._removeTile(e)},_removeAllTiles:function(){for(var t in this._tiles)this._removeTile(t)},_invalidateAll:function(){for(var t in this._levels)T(this._levels[t].el),this._onRemoveLevel(Number(t)),delete this._levels[t];this._removeAllTiles(),this._tileZoom=void 0},_retainParent:function(t,e,i,n){var t=Math.floor(t/2),e=Math.floor(e/2),i=i-1,o=new p(+t,+e),o=(o.z=i,this._tileCoordsToKey(o)),o=this._tiles[o];return o&&o.active?o.retain=!0:(o&&o.loaded&&(o.retain=!0),n<i&&this._retainParent(t,e,i,n))},_retainChildren:function(t,e,i,n){for(var o=2*t;o<2*t+2;o++)for(var s=2*e;s<2*e+2;s++){var r=new p(o,s),r=(r.z=i+1,this._tileCoordsToKey(r)),r=this._tiles[r];r&&r.active?r.retain=!0:(r&&r.loaded&&(r.retain=!0),i+1<n&&this._retainChildren(o,s,i+1,n))}},_resetView:function(t){t=t&&(t.pinch||t.flyTo);this._setView(this._map.getCenter(),this._map.getZoom(),t,t)},_animateZoom:function(t){this._setView(t.center,t.zoom,!0,t.noUpdate)},_clampZoom:function(t){var e=this.options;return void 0!==e.minNativeZoom&&t<e.minNativeZoom?e.minNativeZoom:void 0!==e.maxNativeZoom&&e.maxNativeZoom<t?e.maxNativeZoom:t},_setView:function(t,e,i,n){var o=Math.round(e),o=void 0!==this.options.maxZoom&&o>this.options.maxZoom||void 0!==this.options.minZoom&&o<this.options.minZoom?void 0:this._clampZoom(o),s=this.options.updateWhenZooming&&o!==this._tileZoom;n&&!s||(this._tileZoom=o,this._abortLoading&&this._abortLoading(),this._updateLevels(),this._resetGrid(),void 0!==o&&this._update(t),i||this._pruneTiles(),this._noPrune=!!i),this._setZoomTransforms(t,e)},_setZoomTransforms:function(t,e){for(var i in this._levels)this._setZoomTransform(this._levels[i],t,e)},_setZoomTransform:function(t,e,i){var n=this._map.getZoomScale(i,t.zoom),e=t.origin.multiplyBy(n).subtract(this._map._getNewPixelOrigin(e,i)).round();b.any3d?be(t.el,e,n):Z(t.el,e)},_resetGrid:function(){var t=this._map,e=t.options.crs,i=this._tileSize=this.getTileSize(),n=this._tileZoom,o=this._map.getPixelWorldBounds(this._tileZoom);o&&(this._globalTileRange=this._pxBoundsToTileRange(o)),this._wrapX=e.wrapLng&&!this.options.noWrap&&[Math.floor(t.project([0,e.wrapLng[0]],n).x/i.x),Math.ceil(t.project([0,e.wrapLng[1]],n).x/i.y)],this._wrapY=e.wrapLat&&!this.options.noWrap&&[Math.floor(t.project([e.wrapLat[0],0],n).y/i.x),Math.ceil(t.project([e.wrapLat[1],0],n).y/i.y)]},_onMoveEnd:function(){this._map&&!this._map._animatingZoom&&this._update()},_getTiledPixelBounds:function(t){var e=this._map,i=e._animatingZoom?Math.max(e._animateToZoom,e.getZoom()):e.getZoom(),i=e.getZoomScale(i,this._tileZoom),t=e.project(t,this._tileZoom).floor(),e=e.getSize().divideBy(2*i);return new f(t.subtract(e),t.add(e))},_update:function(t){var e=this._map;if(e){var i=this._clampZoom(e.getZoom());if(void 0===t&&(t=e.getCenter()),void 0!==this._tileZoom){var n,e=this._getTiledPixelBounds(t),o=this._pxBoundsToTileRange(e),s=o.getCenter(),r=[],e=this.options.keepBuffer,a=new f(o.getBottomLeft().subtract([e,-e]),o.getTopRight().add([e,-e]));if(!(isFinite(o.min.x)&&isFinite(o.min.y)&&isFinite(o.max.x)&&isFinite(o.max.y)))throw new Error("Attempted to load an infinite number of tiles");for(n in this._tiles){var h=this._tiles[n].coords;h.z===this._tileZoom&&a.contains(new p(h.x,h.y))||(this._tiles[n].current=!1)}if(1<Math.abs(i-this._tileZoom))this._setView(t,i);else{for(var l=o.min.y;l<=o.max.y;l++)for(var u=o.min.x;u<=o.max.x;u++){var c,d=new p(u,l);d.z=this._tileZoom,this._isValidTile(d)&&((c=this._tiles[this._tileCoordsToKey(d)])?c.current=!0:r.push(d))}if(r.sort(function(t,e){return t.distanceTo(s)-e.distanceTo(s)}),0!==r.length){this._loading||(this._loading=!0,this.fire("loading"));for(var _=document.createDocumentFragment(),u=0;u<r.length;u++)this._addTile(r[u],_);this._level.el.appendChild(_)}}}}},_isValidTile:function(t){var e=this._map.options.crs;if(!e.infinite){var i=this._globalTileRange;if(!e.wrapLng&&(t.x<i.min.x||t.x>i.max.x)||!e.wrapLat&&(t.y<i.min.y||t.y>i.max.y))return!1}return!this.options.bounds||(e=this._tileCoordsToBounds(t),g(this.options.bounds).overlaps(e))},_keyToBounds:function(t){return this._tileCoordsToBounds(this._keyToTileCoords(t))},_tileCoordsToNwSe:function(t){var e=this._map,i=this.getTileSize(),n=t.scaleBy(i),i=n.add(i);return[e.unproject(n,t.z),e.unproject(i,t.z)]},_tileCoordsToBounds:function(t){t=this._tileCoordsToNwSe(t),t=new s(t[0],t[1]);return t=this.options.noWrap?t:this._map.wrapLatLngBounds(t)},_tileCoordsToKey:function(t){return t.x+":"+t.y+":"+t.z},_keyToTileCoords:function(t){var t=t.split(":"),e=new p(+t[0],+t[1]);return e.z=+t[2],e},_removeTile:function(t){var e=this._tiles[t];e&&(T(e.el),delete this._tiles[t],this.fire("tileunload",{tile:e.el,coords:this._keyToTileCoords(t)}))},_initTile:function(t){M(t,"leaflet-tile");var e=this.getTileSize();t.style.width=e.x+"px",t.style.height=e.y+"px",t.onselectstart=u,t.onmousemove=u,b.ielt9&&this.options.opacity<1&&C(t,this.options.opacity)},_addTile:function(t,e){var i=this._getTilePos(t),n=this._tileCoordsToKey(t),o=this.createTile(this._wrapCoords(t),a(this._tileReady,this,t));this._initTile(o),this.createTile.length<2&&x(a(this._tileReady,this,t,null,o)),Z(o,i),this._tiles[n]={el:o,coords:t,current:!0},e.appendChild(o),this.fire("tileloadstart",{tile:o,coords:t})},_tileReady:function(t,e,i){e&&this.fire("tileerror",{error:e,tile:i,coords:t});var n=this._tileCoordsToKey(t);(i=this._tiles[n])&&(i.loaded=+new Date,this._map._fadeAnimated?(C(i.el,0),r(this._fadeFrame),this._fadeFrame=x(this._updateOpacity,this)):(i.active=!0,this._pruneTiles()),e||(M(i.el,"leaflet-tile-loaded"),this.fire("tileload",{tile:i.el,coords:t})),this._noTilesToLoad()&&(this._loading=!1,this.fire("load"),b.ielt9||!this._map._fadeAnimated?x(this._pruneTiles,this):setTimeout(a(this._pruneTiles,this),250)))},_getTilePos:function(t){return t.scaleBy(this.getTileSize()).subtract(this._level.origin)},_wrapCoords:function(t){var e=new p(this._wrapX?H(t.x,this._wrapX):t.x,this._wrapY?H(t.y,this._wrapY):t.y);return e.z=t.z,e},_pxBoundsToTileRange:function(t){var e=this.getTileSize();return new f(t.min.unscaleBy(e).floor(),t.max.unscaleBy(e).ceil().subtract([1,1]))},_noTilesToLoad:function(){for(var t in this._tiles)if(!this._tiles[t].loaded)return!1;return!0}});var Di=Ni.extend({options:{minZoom:0,maxZoom:18,subdomains:"abc",errorTileUrl:"",zoomOffset:0,tms:!1,zoomReverse:!1,detectRetina:!1,crossOrigin:!1,referrerPolicy:!1},initialize:function(t,e){this._url=t,(e=c(this,e)).detectRetina&&b.retina&&0<e.maxZoom?(e.tileSize=Math.floor(e.tileSize/2),e.zoomReverse?(e.zoomOffset--,e.minZoom=Math.min(e.maxZoom,e.minZoom+1)):(e.zoomOffset++,e.maxZoom=Math.max(e.minZoom,e.maxZoom-1)),e.minZoom=Math.max(0,e.minZoom)):e.zoomReverse?e.minZoom=Math.min(e.maxZoom,e.minZoom):e.maxZoom=Math.max(e.minZoom,e.maxZoom),"string"==typeof e.subdomains&&(e.subdomains=e.subdomains.split("")),this.on("tileunload",this._onTileRemove)},setUrl:function(t,e){return this._url===t&&void 0===e&&(e=!0),this._url=t,e||this.redraw(),this},createTile:function(t,e){var i=document.createElement("img");return S(i,"load",a(this._tileOnLoad,this,e,i)),S(i,"error",a(this._tileOnError,this,e,i)),!this.options.crossOrigin&&""!==this.options.crossOrigin||(i.crossOrigin=!0===this.options.crossOrigin?"":this.options.crossOrigin),"string"==typeof this.options.referrerPolicy&&(i.referrerPolicy=this.options.referrerPolicy),i.alt="",i.src=this.getTileUrl(t),i},getTileUrl:function(t){var e={r:b.retina?"@2x":"",s:this._getSubdomain(t),x:t.x,y:t.y,z:this._getZoomForUrl()};return this._map&&!this._map.options.crs.infinite&&(t=this._globalTileRange.max.y-t.y,this.options.tms&&(e.y=t),e["-y"]=t),q(this._url,l(e,this.options))},_tileOnLoad:function(t,e){b.ielt9?setTimeout(a(t,this,null,e),0):t(null,e)},_tileOnError:function(t,e,i){var n=this.options.errorTileUrl;n&&e.getAttribute("src")!==n&&(e.src=n),t(i,e)},_onTileRemove:function(t){t.tile.onload=null},_getZoomForUrl:function(){var t=this._tileZoom,e=this.options.maxZoom;return(t=this.options.zoomReverse?e-t:t)+this.options.zoomOffset},_getSubdomain:function(t){t=Math.abs(t.x+t.y)%this.options.subdomains.length;return this.options.subdomains[t]},_abortLoading:function(){var t,e,i;for(t in this._tiles)this._tiles[t].coords.z!==this._tileZoom&&((i=this._tiles[t].el).onload=u,i.onerror=u,i.complete||(i.src=K,e=this._tiles[t].coords,T(i),delete this._tiles[t],this.fire("tileabort",{tile:i,coords:e})))},_removeTile:function(t){var e=this._tiles[t];if(e)return e.el.setAttribute("src",K),Ni.prototype._removeTile.call(this,t)},_tileReady:function(t,e,i){if(this._map&&(!i||i.getAttribute("src")!==K))return Ni.prototype._tileReady.call(this,t,e,i)}});function ji(t,e){return new Di(t,e)}var Hi=Di.extend({defaultWmsParams:{service:"WMS",request:"GetMap",layers:"",styles:"",format:"image/jpeg",transparent:!1,version:"1.1.1"},options:{crs:null,uppercase:!1},initialize:function(t,e){this._url=t;var i,n=l({},this.defaultWmsParams);for(i in e)i in this.options||(n[i]=e[i]);var t=(e=c(this,e)).detectRetina&&b.retina?2:1,o=this.getTileSize();n.width=o.x*t,n.height=o.y*t,this.wmsParams=n},onAdd:function(t){this._crs=this.options.crs||t.options.crs,this._wmsVersion=parseFloat(this.wmsParams.version);var e=1.3<=this._wmsVersion?"crs":"srs";this.wmsParams[e]=this._crs.code,Di.prototype.onAdd.call(this,t)},getTileUrl:function(t){var e=this._tileCoordsToNwSe(t),i=this._crs,i=_(i.project(e[0]),i.project(e[1])),e=i.min,i=i.max,e=(1.3<=this._wmsVersion&&this._crs===li?[e.y,e.x,i.y,i.x]:[e.x,e.y,i.x,i.y]).join(","),i=Di.prototype.getTileUrl.call(this,t);return i+U(this.wmsParams,i,this.options.uppercase)+(this.options.uppercase?"&BBOX=":"&bbox=")+e},setParams:function(t,e){return l(this.wmsParams,t),e||this.redraw(),this}});Di.WMS=Hi,ji.wms=function(t,e){return new Hi(t,e)};var Wi=o.extend({options:{padding:.1},initialize:function(t){c(this,t),h(this),this._layers=this._layers||{}},onAdd:function(){this._container||(this._initContainer(),M(this._container,"leaflet-zoom-animated")),this.getPane().appendChild(this._container),this._update(),this.on("update",this._updatePaths,this)},onRemove:function(){this.off("update",this._updatePaths,this),this._destroyContainer()},getEvents:function(){var t={viewreset:this._reset,zoom:this._onZoom,moveend:this._update,zoomend:this._onZoomEnd};return this._zoomAnimated&&(t.zoomanim=this._onAnimZoom),t},_onAnimZoom:function(t){this._updateTransform(t.center,t.zoom)},_onZoom:function(){this._updateTransform(this._map.getCenter(),this._map.getZoom())},_updateTransform:function(t,e){var i=this._map.getZoomScale(e,this._zoom),n=this._map.getSize().multiplyBy(.5+this.options.padding),o=this._map.project(this._center,e),n=n.multiplyBy(-i).add(o).subtract(this._map._getNewPixelOrigin(t,e));b.any3d?be(this._container,n,i):Z(this._container,n)},_reset:function(){for(var t in this._update(),this._updateTransform(this._center,this._zoom),this._layers)this._layers[t]._reset()},_onZoomEnd:function(){for(var t in this._layers)this._layers[t]._project()},_updatePaths:function(){for(var t in this._layers)this._layers[t]._update()},_update:function(){var t=this.options.padding,e=this._map.getSize(),i=this._map.containerPointToLayerPoint(e.multiplyBy(-t)).round();this._bounds=new f(i,i.add(e.multiplyBy(1+2*t)).round()),this._center=this._map.getCenter(),this._zoom=this._map.getZoom()}}),Fi=Wi.extend({options:{tolerance:0},getEvents:function(){var t=Wi.prototype.getEvents.call(this);return t.viewprereset=this._onViewPreReset,t},_onViewPreReset:function(){this._postponeUpdatePaths=!0},onAdd:function(){Wi.prototype.onAdd.call(this),this._draw()},_initContainer:function(){var t=this._container=document.createElement("canvas");S(t,"mousemove",this._onMouseMove,this),S(t,"click dblclick mousedown mouseup contextmenu",this._onClick,this),S(t,"mouseout",this._handleMouseOut,this),t._leaflet_disable_events=!0,this._ctx=t.getContext("2d")},_destroyContainer:function(){r(this._redrawRequest),delete this._ctx,T(this._container),k(this._container),delete this._container},_updatePaths:function(){if(!this._postponeUpdatePaths){for(var t in this._redrawBounds=null,this._layers)this._layers[t]._update();this._redraw()}},_update:function(){var t,e,i,n;this._map._animatingZoom&&this._bounds||(Wi.prototype._update.call(this),t=this._bounds,e=this._container,i=t.getSize(),n=b.retina?2:1,Z(e,t.min),e.width=n*i.x,e.height=n*i.y,e.style.width=i.x+"px",e.style.height=i.y+"px",b.retina&&this._ctx.scale(2,2),this._ctx.translate(-t.min.x,-t.min.y),this.fire("update"))},_reset:function(){Wi.prototype._reset.call(this),this._postponeUpdatePaths&&(this._postponeUpdatePaths=!1,this._updatePaths())},_initPath:function(t){this._updateDashArray(t);t=(this._layers[h(t)]=t)._order={layer:t,prev:this._drawLast,next:null};this._drawLast&&(this._drawLast.next=t),this._drawLast=t,this._drawFirst=this._drawFirst||this._drawLast},_addPath:function(t){this._requestRedraw(t)},_removePath:function(t){var e=t._order,i=e.next,e=e.prev;i?i.prev=e:this._drawLast=e,e?e.next=i:this._drawFirst=i,delete t._order,delete this._layers[h(t)],this._requestRedraw(t)},_updatePath:function(t){this._extendRedrawBounds(t),t._project(),t._update(),this._requestRedraw(t)},_updateStyle:function(t){this._updateDashArray(t),this._requestRedraw(t)},_updateDashArray:function(t){if("string"==typeof t.options.dashArray){for(var e,i=t.options.dashArray.split(/[, ]+/),n=[],o=0;o<i.length;o++){if(e=Number(i[o]),isNaN(e))return;n.push(e)}t.options._dashArray=n}else t.options._dashArray=t.options.dashArray},_requestRedraw:function(t){this._map&&(this._extendRedrawBounds(t),this._redrawRequest=this._redrawRequest||x(this._redraw,this))},_extendRedrawBounds:function(t){var e;t._pxBounds&&(e=(t.options.weight||0)+1,this._redrawBounds=this._redrawBounds||new f,this._redrawBounds.extend(t._pxBounds.min.subtract([e,e])),this._redrawBounds.extend(t._pxBounds.max.add([e,e])))},_redraw:function(){this._redrawRequest=null,this._redrawBounds&&(this._redrawBounds.min._floor(),this._redrawBounds.max._ceil()),this._clear(),this._draw(),this._redrawBounds=null},_clear:function(){var t,e=this._redrawBounds;e?(t=e.getSize(),this._ctx.clearRect(e.min.x,e.min.y,t.x,t.y)):(this._ctx.save(),this._ctx.setTransform(1,0,0,1,0,0),this._ctx.clearRect(0,0,this._container.width,this._container.height),this._ctx.restore())},_draw:function(){var t,e,i=this._redrawBounds;this._ctx.save(),i&&(e=i.getSize(),this._ctx.beginPath(),this._ctx.rect(i.min.x,i.min.y,e.x,e.y),this._ctx.clip()),this._drawing=!0;for(var n=this._drawFirst;n;n=n.next)t=n.layer,(!i||t._pxBounds&&t._pxBounds.intersects(i))&&t._updatePath();this._drawing=!1,this._ctx.restore()},_updatePoly:function(t,e){if(this._drawing){var i,n,o,s,r=t._parts,a=r.length,h=this._ctx;if(a){for(h.beginPath(),i=0;i<a;i++){for(n=0,o=r[i].length;n<o;n++)s=r[i][n],h[n?"lineTo":"moveTo"](s.x,s.y);e&&h.closePath()}this._fillStroke(h,t)}}},_updateCircle:function(t){var e,i,n,o;this._drawing&&!t._empty()&&(e=t._point,i=this._ctx,n=Math.max(Math.round(t._radius),1),1!=(o=(Math.max(Math.round(t._radiusY),1)||n)/n)&&(i.save(),i.scale(1,o)),i.beginPath(),i.arc(e.x,e.y/o,n,0,2*Math.PI,!1),1!=o&&i.restore(),this._fillStroke(i,t))},_fillStroke:function(t,e){var i=e.options;i.fill&&(t.globalAlpha=i.fillOpacity,t.fillStyle=i.fillColor||i.color,t.fill(i.fillRule||"evenodd")),i.stroke&&0!==i.weight&&(t.setLineDash&&t.setLineDash(e.options&&e.options._dashArray||[]),t.globalAlpha=i.opacity,t.lineWidth=i.weight,t.strokeStyle=i.color,t.lineCap=i.lineCap,t.lineJoin=i.lineJoin,t.stroke())},_onClick:function(t){for(var e,i,n=this._map.mouseEventToLayerPoint(t),o=this._drawFirst;o;o=o.next)(e=o.layer).options.interactive&&e._containsPoint(n)&&(("click"===t.type||"preclick"===t.type)&&this._map._draggableMoved(e)||(i=e));this._fireEvent(!!i&&[i],t)},_onMouseMove:function(t){var e;!this._map||this._map.dragging.moving()||this._map._animatingZoom||(e=this._map.mouseEventToLayerPoint(t),this._handleMouseHover(t,e))},_handleMouseOut:function(t){var e=this._hoveredLayer;e&&(z(this._container,"leaflet-interactive"),this._fireEvent([e],t,"mouseout"),this._hoveredLayer=null,this._mouseHoverThrottled=!1)},_handleMouseHover:function(t,e){if(!this._mouseHoverThrottled){for(var i,n,o=this._drawFirst;o;o=o.next)(i=o.layer).options.interactive&&i._containsPoint(e)&&(n=i);n!==this._hoveredLayer&&(this._handleMouseOut(t),n&&(M(this._container,"leaflet-interactive"),this._fireEvent([n],t,"mouseover"),this._hoveredLayer=n)),this._fireEvent(!!this._hoveredLayer&&[this._hoveredLayer],t),this._mouseHoverThrottled=!0,setTimeout(a(function(){this._mouseHoverThrottled=!1},this),32)}},_fireEvent:function(t,e,i){this._map._fireDOMEvent(e,i||e.type,t)},_bringToFront:function(t){var e,i,n=t._order;n&&(e=n.next,i=n.prev,e&&((e.prev=i)?i.next=e:e&&(this._drawFirst=e),n.prev=this._drawLast,(this._drawLast.next=n).next=null,this._drawLast=n,this._requestRedraw(t)))},_bringToBack:function(t){var e,i,n=t._order;n&&(e=n.next,(i=n.prev)&&((i.next=e)?e.prev=i:i&&(this._drawLast=i),n.prev=null,n.next=this._drawFirst,this._drawFirst.prev=n,this._drawFirst=n,this._requestRedraw(t)))}});function Ui(t){return b.canvas?new Fi(t):null}var Vi=function(){try{return document.namespaces.add("lvml","urn:schemas-microsoft-com:vml"),function(t){return document.createElement("<lvml:"+t+\' class="lvml">\')}}catch(t){}return function(t){return document.createElement("<"+t+\' xmlns="urn:schemas-microsoft.com:vml" class="lvml">\')}}(),zt={_initContainer:function(){this._container=P("div","leaflet-vml-container")},_update:function(){this._map._animatingZoom||(Wi.prototype._update.call(this),this.fire("update"))},_initPath:function(t){var e=t._container=Vi("shape");M(e,"leaflet-vml-shape "+(this.options.className||"")),e.coordsize="1 1",t._path=Vi("path"),e.appendChild(t._path),this._updateStyle(t),this._layers[h(t)]=t},_addPath:function(t){var e=t._container;this._container.appendChild(e),t.options.interactive&&t.addInteractiveTarget(e)},_removePath:function(t){var e=t._container;T(e),t.removeInteractiveTarget(e),delete this._layers[h(t)]},_updateStyle:function(t){var e=t._stroke,i=t._fill,n=t.options,o=t._container;o.stroked=!!n.stroke,o.filled=!!n.fill,n.stroke?(e=e||(t._stroke=Vi("stroke")),o.appendChild(e),e.weight=n.weight+"px",e.color=n.color,e.opacity=n.opacity,n.dashArray?e.dashStyle=d(n.dashArray)?n.dashArray.join(" "):n.dashArray.replace(/( *, *)/g," "):e.dashStyle="",e.endcap=n.lineCap.replace("butt","flat"),e.joinstyle=n.lineJoin):e&&(o.removeChild(e),t._stroke=null),n.fill?(i=i||(t._fill=Vi("fill")),o.appendChild(i),i.color=n.fillColor||n.color,i.opacity=n.fillOpacity):i&&(o.removeChild(i),t._fill=null)},_updateCircle:function(t){var e=t._point.round(),i=Math.round(t._radius),n=Math.round(t._radiusY||i);this._setPath(t,t._empty()?"M0 0":"AL "+e.x+","+e.y+" "+i+","+n+" 0,23592600")},_setPath:function(t,e){t._path.v=e},_bringToFront:function(t){fe(t._container)},_bringToBack:function(t){ge(t._container)}},qi=b.vml?Vi:ct,Gi=Wi.extend({_initContainer:function(){this._container=qi("svg"),this._container.setAttribute("pointer-events","none"),this._rootGroup=qi("g"),this._container.appendChild(this._rootGroup)},_destroyContainer:function(){T(this._container),k(this._container),delete this._container,delete this._rootGroup,delete this._svgSize},_update:function(){var t,e,i;this._map._animatingZoom&&this._bounds||(Wi.prototype._update.call(this),e=(t=this._bounds).getSize(),i=this._container,this._svgSize&&this._svgSize.equals(e)||(this._svgSize=e,i.setAttribute("width",e.x),i.setAttribute("height",e.y)),Z(i,t.min),i.setAttribute("viewBox",[t.min.x,t.min.y,e.x,e.y].join(" ")),this.fire("update"))},_initPath:function(t){var e=t._path=qi("path");t.options.className&&M(e,t.options.className),t.options.interactive&&M(e,"leaflet-interactive"),this._updateStyle(t),this._layers[h(t)]=t},_addPath:function(t){this._rootGroup||this._initContainer(),this._rootGroup.appendChild(t._path),t.addInteractiveTarget(t._path)},_removePath:function(t){T(t._path),t.removeInteractiveTarget(t._path),delete this._layers[h(t)]},_updatePath:function(t){t._project(),t._update()},_updateStyle:function(t){var e=t._path,t=t.options;e&&(t.stroke?(e.setAttribute("stroke",t.color),e.setAttribute("stroke-opacity",t.opacity),e.setAttribute("stroke-width",t.weight),e.setAttribute("stroke-linecap",t.lineCap),e.setAttribute("stroke-linejoin",t.lineJoin),t.dashArray?e.setAttribute("stroke-dasharray",t.dashArray):e.removeAttribute("stroke-dasharray"),t.dashOffset?e.setAttribute("stroke-dashoffset",t.dashOffset):e.removeAttribute("stroke-dashoffset")):e.setAttribute("stroke","none"),t.fill?(e.setAttribute("fill",t.fillColor||t.color),e.setAttribute("fill-opacity",t.fillOpacity),e.setAttribute("fill-rule",t.fillRule||"evenodd")):e.setAttribute("fill","none"))},_updatePoly:function(t,e){this._setPath(t,dt(t._parts,e))},_updateCircle:function(t){var e=t._point,i=Math.max(Math.round(t._radius),1),n="a"+i+","+(Math.max(Math.round(t._radiusY),1)||i)+" 0 1,0 ",e=t._empty()?"M0 0":"M"+(e.x-i)+","+e.y+n+2*i+",0 "+n+2*-i+",0 ";this._setPath(t,e)},_setPath:function(t,e){t._path.setAttribute("d",e)},_bringToFront:function(t){fe(t._path)},_bringToBack:function(t){ge(t._path)}});function Ki(t){return b.svg||b.vml?new Gi(t):null}b.vml&&Gi.include(zt),A.include({getRenderer:function(t){t=(t=t.options.renderer||this._getPaneRenderer(t.options.pane)||this.options.renderer||this._renderer)||(this._renderer=this._createRenderer());return this.hasLayer(t)||this.addLayer(t),t},_getPaneRenderer:function(t){var e;return"overlayPane"!==t&&void 0!==t&&(void 0===(e=this._paneRenderers[t])&&(e=this._createRenderer({pane:t}),this._paneRenderers[t]=e),e)},_createRenderer:function(t){return this.options.preferCanvas&&Ui(t)||Ki(t)}});var Yi=xi.extend({initialize:function(t,e){xi.prototype.initialize.call(this,this._boundsToLatLngs(t),e)},setBounds:function(t){return this.setLatLngs(this._boundsToLatLngs(t))},_boundsToLatLngs:function(t){return[(t=g(t)).getSouthWest(),t.getNorthWest(),t.getNorthEast(),t.getSouthEast()]}});Gi.create=qi,Gi.pointsToPath=dt,wi.geometryToLayer=bi,wi.coordsToLatLng=Li,wi.coordsToLatLngs=Ti,wi.latLngToCoords=Mi,wi.latLngsToCoords=zi,wi.getFeature=Ci,wi.asFeature=Zi,A.mergeOptions({boxZoom:!0});var _t=n.extend({initialize:function(t){this._map=t,this._container=t._container,this._pane=t._panes.overlayPane,this._resetStateTimeout=0,t.on("unload",this._destroy,this)},addHooks:function(){S(this._container,"mousedown",this._onMouseDown,this)},removeHooks:function(){k(this._container,"mousedown",this._onMouseDown,this)},moved:function(){return this._moved},_destroy:function(){T(this._pane),delete this._pane},_resetState:function(){this._resetStateTimeout=0,this._moved=!1},_clearDeferredResetState:function(){0!==this._resetStateTimeout&&(clearTimeout(this._resetStateTimeout),this._resetStateTimeout=0)},_onMouseDown:function(t){if(!t.shiftKey||1!==t.which&&1!==t.button)return!1;this._clearDeferredResetState(),this._resetState(),re(),Le(),this._startPoint=this._map.mouseEventToContainerPoint(t),S(document,{contextmenu:Re,mousemove:this._onMouseMove,mouseup:this._onMouseUp,keydown:this._onKeyDown},this)},_onMouseMove:function(t){this._moved||(this._moved=!0,this._box=P("div","leaflet-zoom-box",this._container),M(this._container,"leaflet-crosshair"),this._map.fire("boxzoomstart")),this._point=this._map.mouseEventToContainerPoint(t);var t=new f(this._point,this._startPoint),e=t.getSize();Z(this._box,t.min),this._box.style.width=e.x+"px",this._box.style.height=e.y+"px"},_finish:function(){this._moved&&(T(this._box),z(this._container,"leaflet-crosshair")),ae(),Te(),k(document,{contextmenu:Re,mousemove:this._onMouseMove,mouseup:this._onMouseUp,keydown:this._onKeyDown},this)},_onMouseUp:function(t){1!==t.which&&1!==t.button||(this._finish(),this._moved&&(this._clearDeferredResetState(),this._resetStateTimeout=setTimeout(a(this._resetState,this),0),t=new s(this._map.containerPointToLatLng(this._startPoint),this._map.containerPointToLatLng(this._point)),this._map.fitBounds(t).fire("boxzoomend",{boxZoomBounds:t})))},_onKeyDown:function(t){27===t.keyCode&&(this._finish(),this._clearDeferredResetState(),this._resetState())}}),Ct=(A.addInitHook("addHandler","boxZoom",_t),A.mergeOptions({doubleClickZoom:!0}),n.extend({addHooks:function(){this._map.on("dblclick",this._onDoubleClick,this)},removeHooks:function(){this._map.off("dblclick",this._onDoubleClick,this)},_onDoubleClick:function(t){var e=this._map,i=e.getZoom(),n=e.options.zoomDelta,i=t.originalEvent.shiftKey?i-n:i+n;"center"===e.options.doubleClickZoom?e.setZoom(i):e.setZoomAround(t.containerPoint,i)}})),Zt=(A.addInitHook("addHandler","doubleClickZoom",Ct),A.mergeOptions({dragging:!0,inertia:!0,inertiaDeceleration:3400,inertiaMaxSpeed:1/0,easeLinearity:.2,worldCopyJump:!1,maxBoundsViscosity:0}),n.extend({addHooks:function(){var t;this._draggable||(t=this._map,this._draggable=new Xe(t._mapPane,t._container),this._draggable.on({dragstart:this._onDragStart,drag:this._onDrag,dragend:this._onDragEnd},this),this._draggable.on("predrag",this._onPreDragLimit,this),t.options.worldCopyJump&&(this._draggable.on("predrag",this._onPreDragWrap,this),t.on("zoomend",this._onZoomEnd,this),t.whenReady(this._onZoomEnd,this))),M(this._map._container,"leaflet-grab leaflet-touch-drag"),this._draggable.enable(),this._positions=[],this._times=[]},removeHooks:function(){z(this._map._container,"leaflet-grab"),z(this._map._container,"leaflet-touch-drag"),this._draggable.disable()},moved:function(){return this._draggable&&this._draggable._moved},moving:function(){return this._draggable&&this._draggable._moving},_onDragStart:function(){var t,e=this._map;e._stop(),this._map.options.maxBounds&&this._map.options.maxBoundsViscosity?(t=g(this._map.options.maxBounds),this._offsetLimit=_(this._map.latLngToContainerPoint(t.getNorthWest()).multiplyBy(-1),this._map.latLngToContainerPoint(t.getSouthEast()).multiplyBy(-1).add(this._map.getSize())),this._viscosity=Math.min(1,Math.max(0,this._map.options.maxBoundsViscosity))):this._offsetLimit=null,e.fire("movestart").fire("dragstart"),e.options.inertia&&(this._positions=[],this._times=[])},_onDrag:function(t){var e,i;this._map.options.inertia&&(e=this._lastTime=+new Date,i=this._lastPos=this._draggable._absPos||this._draggable._newPos,this._positions.push(i),this._times.push(e),this._prunePositions(e)),this._map.fire("move",t).fire("drag",t)},_prunePositions:function(t){for(;1<this._positions.length&&50<t-this._times[0];)this._positions.shift(),this._times.shift()},_onZoomEnd:function(){var t=this._map.getSize().divideBy(2),e=this._map.latLngToLayerPoint([0,0]);this._initialWorldOffset=e.subtract(t).x,this._worldWidth=this._map.getPixelWorldBounds().getSize().x},_viscousLimit:function(t,e){return t-(t-e)*this._viscosity},_onPreDragLimit:function(){var t,e;this._viscosity&&this._offsetLimit&&(t=this._draggable._newPos.subtract(this._draggable._startPos),e=this._offsetLimit,t.x<e.min.x&&(t.x=this._viscousLimit(t.x,e.min.x)),t.y<e.min.y&&(t.y=this._viscousLimit(t.y,e.min.y)),t.x>e.max.x&&(t.x=this._viscousLimit(t.x,e.max.x)),t.y>e.max.y&&(t.y=this._viscousLimit(t.y,e.max.y)),this._draggable._newPos=this._draggable._startPos.add(t))},_onPreDragWrap:function(){var t=this._worldWidth,e=Math.round(t/2),i=this._initialWorldOffset,n=this._draggable._newPos.x,o=(n-e+i)%t+e-i,n=(n+e+i)%t-e-i,t=Math.abs(o+i)<Math.abs(n+i)?o:n;this._draggable._absPos=this._draggable._newPos.clone(),this._draggable._newPos.x=t},_onDragEnd:function(t){var e,i,n,o,s=this._map,r=s.options,a=!r.inertia||t.noInertia||this._times.length<2;s.fire("dragend",t),!a&&(this._prunePositions(+new Date),t=this._lastPos.subtract(this._positions[0]),a=(this._lastTime-this._times[0])/1e3,e=r.easeLinearity,a=(t=t.multiplyBy(e/a)).distanceTo([0,0]),i=Math.min(r.inertiaMaxSpeed,a),t=t.multiplyBy(i/a),n=i/(r.inertiaDeceleration*e),(o=t.multiplyBy(-n/2).round()).x||o.y)?(o=s._limitOffset(o,s.options.maxBounds),x(function(){s.panBy(o,{duration:n,easeLinearity:e,noMoveStart:!0,animate:!0})})):s.fire("moveend")}})),St=(A.addInitHook("addHandler","dragging",Zt),A.mergeOptions({keyboard:!0,keyboardPanDelta:80}),n.extend({keyCodes:{left:[37],right:[39],down:[40],up:[38],zoomIn:[187,107,61,171],zoomOut:[189,109,54,173]},initialize:function(t){this._map=t,this._setPanDelta(t.options.keyboardPanDelta),this._setZoomDelta(t.options.zoomDelta)},addHooks:function(){var t=this._map._container;t.tabIndex<=0&&(t.tabIndex="0"),S(t,{focus:this._onFocus,blur:this._onBlur,mousedown:this._onMouseDown},this),this._map.on({focus:this._addHooks,blur:this._removeHooks},this)},removeHooks:function(){this._removeHooks(),k(this._map._container,{focus:this._onFocus,blur:this._onBlur,mousedown:this._onMouseDown},this),this._map.off({focus:this._addHooks,blur:this._removeHooks},this)},_onMouseDown:function(){var t,e,i;this._focused||(i=document.body,t=document.documentElement,e=i.scrollTop||t.scrollTop,i=i.scrollLeft||t.scrollLeft,this._map._container.focus(),window.scrollTo(i,e))},_onFocus:function(){this._focused=!0,this._map.fire("focus")},_onBlur:function(){this._focused=!1,this._map.fire("blur")},_setPanDelta:function(t){for(var e=this._panKeys={},i=this.keyCodes,n=0,o=i.left.length;n<o;n++)e[i.left[n]]=[-1*t,0];for(n=0,o=i.right.length;n<o;n++)e[i.right[n]]=[t,0];for(n=0,o=i.down.length;n<o;n++)e[i.down[n]]=[0,t];for(n=0,o=i.up.length;n<o;n++)e[i.up[n]]=[0,-1*t]},_setZoomDelta:function(t){for(var e=this._zoomKeys={},i=this.keyCodes,n=0,o=i.zoomIn.length;n<o;n++)e[i.zoomIn[n]]=t;for(n=0,o=i.zoomOut.length;n<o;n++)e[i.zoomOut[n]]=-t},_addHooks:function(){S(document,"keydown",this._onKeyDown,this)},_removeHooks:function(){k(document,"keydown",this._onKeyDown,this)},_onKeyDown:function(t){if(!(t.altKey||t.ctrlKey||t.metaKey)){var e,i,n=t.keyCode,o=this._map;if(n in this._panKeys)o._panAnim&&o._panAnim._inProgress||(i=this._panKeys[n],t.shiftKey&&(i=m(i).multiplyBy(3)),o.options.maxBounds&&(i=o._limitOffset(m(i),o.options.maxBounds)),o.options.worldCopyJump?(e=o.wrapLatLng(o.unproject(o.project(o.getCenter()).add(i))),o.panTo(e)):o.panBy(i));else if(n in this._zoomKeys)o.setZoom(o.getZoom()+(t.shiftKey?3:1)*this._zoomKeys[n]);else{if(27!==n||!o._popup||!o._popup.options.closeOnEscapeKey)return;o.closePopup()}Re(t)}}})),Et=(A.addInitHook("addHandler","keyboard",St),A.mergeOptions({scrollWheelZoom:!0,wheelDebounceTime:40,wheelPxPerZoomLevel:60}),n.extend({addHooks:function(){S(this._map._container,"wheel",this._onWheelScroll,this),this._delta=0},removeHooks:function(){k(this._map._container,"wheel",this._onWheelScroll,this)},_onWheelScroll:function(t){var e=He(t),i=this._map.options.wheelDebounceTime,e=(this._delta+=e,this._lastMousePos=this._map.mouseEventToContainerPoint(t),this._startTime||(this._startTime=+new Date),Math.max(i-(+new Date-this._startTime),0));clearTimeout(this._timer),this._timer=setTimeout(a(this._performZoom,this),e),Re(t)},_performZoom:function(){var t=this._map,e=t.getZoom(),i=this._map.options.zoomSnap||0,n=(t._stop(),this._delta/(4*this._map.options.wheelPxPerZoomLevel)),n=4*Math.log(2/(1+Math.exp(-Math.abs(n))))/Math.LN2,i=i?Math.ceil(n/i)*i:n,n=t._limitZoom(e+(0<this._delta?i:-i))-e;this._delta=0,this._startTime=null,n&&("center"===t.options.scrollWheelZoom?t.setZoom(e+n):t.setZoomAround(this._lastMousePos,e+n))}})),kt=(A.addInitHook("addHandler","scrollWheelZoom",Et),A.mergeOptions({tapHold:b.touchNative&&b.safari&&b.mobile,tapTolerance:15}),n.extend({addHooks:function(){S(this._map._container,"touchstart",this._onDown,this)},removeHooks:function(){k(this._map._container,"touchstart",this._onDown,this)},_onDown:function(t){var e;clearTimeout(this._holdTimeout),1===t.touches.length&&(e=t.touches[0],this._startPos=this._newPos=new p(e.clientX,e.clientY),this._holdTimeout=setTimeout(a(function(){this._cancel(),this._isTapValid()&&(S(document,"touchend",O),S(document,"touchend touchcancel",this._cancelClickPrevent),this._simulateEvent("contextmenu",e))},this),600),S(document,"touchend touchcancel contextmenu",this._cancel,this),S(document,"touchmove",this._onMove,this))},_cancelClickPrevent:function t(){k(document,"touchend",O),k(document,"touchend touchcancel",t)},_cancel:function(){clearTimeout(this._holdTimeout),k(document,"touchend touchcancel contextmenu",this._cancel,this),k(document,"touchmove",this._onMove,this)},_onMove:function(t){t=t.touches[0];this._newPos=new p(t.clientX,t.clientY)},_isTapValid:function(){return this._newPos.distanceTo(this._startPos)<=this._map.options.tapTolerance},_simulateEvent:function(t,e){t=new MouseEvent(t,{bubbles:!0,cancelable:!0,view:window,screenX:e.screenX,screenY:e.screenY,clientX:e.clientX,clientY:e.clientY});t._simulated=!0,e.target.dispatchEvent(t)}})),Ot=(A.addInitHook("addHandler","tapHold",kt),A.mergeOptions({touchZoom:b.touch,bounceAtZoomLimits:!0}),n.extend({addHooks:function(){M(this._map._container,"leaflet-touch-zoom"),S(this._map._container,"touchstart",this._onTouchStart,this)},removeHooks:function(){z(this._map._container,"leaflet-touch-zoom"),k(this._map._container,"touchstart",this._onTouchStart,this)},_onTouchStart:function(t){var e,i,n=this._map;!t.touches||2!==t.touches.length||n._animatingZoom||this._zooming||(e=n.mouseEventToContainerPoint(t.touches[0]),i=n.mouseEventToContainerPoint(t.touches[1]),this._centerPoint=n.getSize()._divideBy(2),this._startLatLng=n.containerPointToLatLng(this._centerPoint),"center"!==n.options.touchZoom&&(this._pinchStartLatLng=n.containerPointToLatLng(e.add(i)._divideBy(2))),this._startDist=e.distanceTo(i),this._startZoom=n.getZoom(),this._moved=!1,this._zooming=!0,n._stop(),S(document,"touchmove",this._onTouchMove,this),S(document,"touchend touchcancel",this._onTouchEnd,this),O(t))},_onTouchMove:function(t){if(t.touches&&2===t.touches.length&&this._zooming){var e=this._map,i=e.mouseEventToContainerPoint(t.touches[0]),n=e.mouseEventToContainerPoint(t.touches[1]),o=i.distanceTo(n)/this._startDist;if(this._zoom=e.getScaleZoom(o,this._startZoom),!e.options.bounceAtZoomLimits&&(this._zoom<e.getMinZoom()&&o<1||this._zoom>e.getMaxZoom()&&1<o)&&(this._zoom=e._limitZoom(this._zoom)),"center"===e.options.touchZoom){if(this._center=this._startLatLng,1==o)return}else{i=i._add(n)._divideBy(2)._subtract(this._centerPoint);if(1==o&&0===i.x&&0===i.y)return;this._center=e.unproject(e.project(this._pinchStartLatLng,this._zoom).subtract(i),this._zoom)}this._moved||(e._moveStart(!0,!1),this._moved=!0),r(this._animRequest);n=a(e._move,e,this._center,this._zoom,{pinch:!0,round:!1},void 0);this._animRequest=x(n,this,!0),O(t)}},_onTouchEnd:function(){this._moved&&this._zooming?(this._zooming=!1,r(this._animRequest),k(document,"touchmove",this._onTouchMove,this),k(document,"touchend touchcancel",this._onTouchEnd,this),this._map.options.zoomAnimation?this._map._animateZoom(this._center,this._map._limitZoom(this._zoom),!0,this._map.options.zoomSnap):this._map._resetView(this._center,this._map._limitZoom(this._zoom))):this._zooming=!1}})),Xi=(A.addInitHook("addHandler","touchZoom",Ot),A.BoxZoom=_t,A.DoubleClickZoom=Ct,A.Drag=Zt,A.Keyboard=St,A.ScrollWheelZoom=Et,A.TapHold=kt,A.TouchZoom=Ot,t.Bounds=f,t.Browser=b,t.CRS=ot,t.Canvas=Fi,t.Circle=vi,t.CircleMarker=gi,t.Class=et,t.Control=B,t.DivIcon=Ri,t.DivOverlay=Ai,t.DomEvent=mt,t.DomUtil=pt,t.Draggable=Xe,t.Evented=it,t.FeatureGroup=ci,t.GeoJSON=wi,t.GridLayer=Ni,t.Handler=n,t.Icon=di,t.ImageOverlay=Ei,t.LatLng=v,t.LatLngBounds=s,t.Layer=o,t.LayerGroup=ui,t.LineUtil=vt,t.Map=A,t.Marker=mi,t.Mixin=ft,t.Path=fi,t.Point=p,t.PolyUtil=gt,t.Polygon=xi,t.Polyline=yi,t.Popup=Bi,t.PosAnimation=Fe,t.Projection=wt,t.Rectangle=Yi,t.Renderer=Wi,t.SVG=Gi,t.SVGOverlay=Oi,t.TileLayer=Di,t.Tooltip=Ii,t.Transformation=at,t.Util=tt,t.VideoOverlay=ki,t.bind=a,t.bounds=_,t.canvas=Ui,t.circle=function(t,e,i){return new vi(t,e,i)},t.circleMarker=function(t,e){return new gi(t,e)},t.control=Ue,t.divIcon=function(t){return new Ri(t)},t.extend=l,t.featureGroup=function(t,e){return new ci(t,e)},t.geoJSON=Si,t.geoJson=Mt,t.gridLayer=function(t){return new Ni(t)},t.icon=function(t){return new di(t)},t.imageOverlay=function(t,e,i){return new Ei(t,e,i)},t.latLng=w,t.latLngBounds=g,t.layerGroup=function(t,e){return new ui(t,e)},t.map=function(t,e){return new A(t,e)},t.marker=function(t,e){return new mi(t,e)},t.point=m,t.polygon=function(t,e){return new xi(t,e)},t.polyline=function(t,e){return new yi(t,e)},t.popup=function(t,e){return new Bi(t,e)},t.rectangle=function(t,e){return new Yi(t,e)},t.setOptions=c,t.stamp=h,t.svg=Ki,t.svgOverlay=function(t,e,i){return new Oi(t,e,i)},t.tileLayer=ji,t.tooltip=function(t,e){return new Ii(t,e)},t.transformation=ht,t.version="1.9.4",t.videoOverlay=function(t,e,i){return new ki(t,e,i)},window.L);t.noConflict=function(){return window.L=Xi,this},window.L=t});\n\n</script>\n<script>\n/* Mapas OD: Leaflet se carga una sola vez desde el documento del visor. */\nfunction createODMaps({geo, container, onSelect, formatNumber, formatPercent}) {\n  if (!container || !window.L) throw new Error(\'No se pudo iniciar el componente de mapas.\');\n  const L = window.L;\n  const number = formatNumber || (v => Number(v).toLocaleString(\'es-CO\', {maximumFractionDigits: 0}));\n  const percent = formatPercent || (v => Number(v).toLocaleString(\'es-CO\', {minimumFractionDigits: 1, maximumFractionDigits: 1}) + \' %\');\n  const escape = v => String(v == null ? \'\' : v).replace(/[&<>"\']/g, c => ({\'&\':\'&amp;\', \'<\':\'&lt;\', \'>\':\'&gt;\', \'"\':\'&quot;\', "\'":\'&#39;\'}[c]));\n  const features = key => (geo[key] && Array.isArray(geo[key].features)) ? geo[key].features : [];\n  const zones = features(\'zonas\');\n  const localities = features(\'localidades\');\n  const municipalities = features(\'municipios\');\n  const utams = features(\'utams\');\n  const zoneIDs = feature => {\n    const p = feature.properties || {};\n    return Array.isArray(p.zone_ids) ? p.zone_ids.map(String) : (p.key != null ? [String(p.key)] : []);\n  };\n  const code = value => value == null ? \'\' : String(value);\n  const fc = items => ({type: \'FeatureCollection\', features: items});\n  const findFeatures = view => view === \'regional\' ? municipalities : view === \'localidades\' ? localities : zones;\n  const keyOf = feature => code((feature.properties || {}).key);\n  const bogotaZone = feature => code((feature.properties || {}).mun_code) === \'11001\';\n  const bogotaLocalKeys = new Set(localities.map(keyOf));\n  const bogotaZones = zones.filter(bogotaZone);\n  const bogotaMunicipalities = municipalities.filter(f => keyOf(f) === \'11001\');\n  let state = {view: null, result: null, filters: {origen: null, destino: null}, destroyed: false};\n  const sides = [];\n  const element = (tag, cls, text) => {\n    const e = document.createElement(tag);\n    if (cls) e.className = cls;\n    if (text != null) e.textContent = text;\n    return e;\n  };\n  const wrapper = element(\'div\', \'od-maps\');\n  const basemapTools = element(\'div\', \'od-basemap-tools\');\n  const basemapLabel = element(\'label\', \'\', \'Fondo del mapa \');\n  const basemapSelect = element(\'select\', \'od-basemap-select\');\n  basemapSelect.setAttribute(\'aria-label\', \'Fondo del mapa\');\n  [[\'osm\', \'OpenStreetMap\'], [\'none\', \'Sin fondo\']].forEach(([value, label]) => {\n    const option = element(\'option\', \'\', label);\n    option.value = value;\n    basemapSelect.append(option);\n  });\n  basemapLabel.append(basemapSelect);\n  const basemapStatus = element(\'span\', \'od-basemap-status\', \'Si el fondo aparece bloqueado, selecciona Sin fondo.\');\n  basemapStatus.setAttribute(\'role\', \'status\');\n  basemapTools.append(basemapLabel, basemapStatus);\n  const grid = element(\'div\', \'od-map-grid\');\n  wrapper.append(basemapTools, grid);\n  container.replaceChildren(wrapper);\n  let tileErrors = 0;\n  function setBasemap(value, failed = false) {\n    if (state.destroyed) return;\n    basemapSelect.value = value;\n    tileErrors = 0;\n    sides.forEach(side => {\n      if (value === \'osm\') {\n        if (!side.map.hasLayer(side.baseLayer)) side.baseLayer.addTo(side.map);\n      } else if (side.map.hasLayer(side.baseLayer)) side.map.removeLayer(side.baseLayer);\n    });\n    basemapStatus.textContent = failed\n      ? \'El fondo no está disponible. Las capas y los cálculos siguen activos.\'\n      : value === \'none\' ? \'Las capas y los cálculos siguen activos.\'\n      : \'Si el fondo aparece bloqueado, selecciona Sin fondo.\';\n  }\n  basemapSelect.addEventListener(\'change\', () => setBasemap(basemapSelect.value));\n\n  function blue(value, maximum) {\n    if (!(value > 0) || !(maximum > 0)) return \'#f5f7fa\';\n    const stops = [[239,246,255], [191,219,254], [96,165,250], [37,99,235], [8,48,107]];\n    const scaled = Math.max(0, Math.min(1, value / maximum)) * (stops.length - 1);\n    const i = Math.min(stops.length - 2, Math.floor(scaled));\n    const t = scaled - i;\n    return \'rgb(\' + stops[i].map((v, k) => Math.round(v + (stops[i + 1][k] - v) * t)).join(\',\') + \')\';\n  }\n\n  function toggle(feature, side) {\n    const values = zoneIDs(feature);\n    if (!values.length || typeof onSelect !== \'function\') return;\n    const previous = state.filters[side];\n    const next = new Set(Array.isArray(previous) ? previous.map(String) : []);\n    const allSelected = values.every(id => next.has(id));\n    values.forEach(id => allSelected ? next.delete(id) : next.add(id));\n    onSelect(side, [...next]);\n  }\n\n  function boundsOf(items) {\n    try {\n      const bounds = L.geoJSON(fc(items)).getBounds();\n      return bounds.isValid() ? bounds : null;\n    } catch (_) { return null; }\n  }\n\n  function defaultBounds(view) {\n    if (view === \'regional\') {\n      const represented = new Set(zones.map(z => code(z.properties && z.properties.mun_code)).filter(Boolean));\n      const candidates = municipalities.filter(f => represented.has(keyOf(f)) || keyOf(f) === \'11001\');\n      return boundsOf(candidates) || boundsOf(bogotaLocalKeys.size ? localities : bogotaZones);\n    }\n    return boundsOf(localities) || boundsOf(bogotaZones) || boundsOf(zones);\n  }\n\n  function resultBounds(result) {\n    const items = findFeatures(result.view);\n    const keys = new Set([...(result.origins || []), ...(result.destinations || [])].filter(r => Number(r.value) > 0).map(r => String(r.key)));\n    const selected = new Set([...(state.filters.origen || []), ...(state.filters.destino || [])].map(String));\n    const picked = zones.filter(z => zoneIDs(z).some(id => selected.has(id)));\n    let relevant = items.filter(f => keys.has(keyOf(f)));\n    if (result.view === \'regional\') relevant = relevant.concat(bogotaMunicipalities);\n    else if (result.view === \'localidades\') relevant = relevant.concat(localities);\n    else if (!relevant.length) relevant = bogotaZones;\n    return boundsOf(relevant.concat(picked)) || defaultBounds(result.view);\n  }\n\n  function resetExtent(side) {\n    if (!state.result || state.destroyed) return;\n    const bounds = resultBounds(state.result);\n    if (bounds) side.map.fitBounds(bounds, {padding: [20, 20], maxZoom: 13, animate: false});\n    else side.map.setView([4.65, -74.1], 10, {animate: false});\n  }\n\n  function labels(view, side) {\n    if (view === \'regional\') return side === \'origen\'\n      ? {heading: \'Orígenes · Municipios → Bogotá\', sub: \'Viajes desde cada municipio hacia Bogotá\', territory: \'Municipio de origen\'}\n      : {heading: \'Destinos · Bogotá → Municipios\', sub: \'Viajes desde Bogotá hacia cada municipio\', territory: \'Municipio de destino\'};\n    if (view === \'localidades\') return side === \'origen\'\n      ? {heading: \'Orígenes por localidad\', sub: \'Viajes que salen de cada localidad\', territory: \'Localidad de origen\'}\n      : {heading: \'Destinos por localidad\', sub: \'Viajes que llegan a cada localidad\', territory: \'Localidad de destino\'};\n    return side === \'origen\'\n      ? {heading: \'Orígenes por ZAT\', sub: \'Viajes que salen de las zonas consultadas\', territory: \'ZAT de origen\'}\n      : {heading: \'Destinos por ZAT\', sub: \'Viajes que llegan a las zonas consultadas\', territory: \'ZAT de destino\'};\n  }\n\n  [\'origen\', \'destino\'].forEach(name => {\n    const card = element(\'section\', \'od-map-column\');\n    const header = element(\'header\', \'od-map-heading\');\n    const title = element(\'h3\', \'od-map-title\');\n    const subtitle = element(\'p\', \'od-map-subtitle\');\n    const count = element(\'div\', \'od-map-count\');\n    header.append(title, subtitle, count);\n    const tools = element(\'div\', \'od-map-tools\');\n    const year = element(\'span\', \'od-map-year\');\n    const reset = element(\'button\', \'od-reset-extent\', \'Restablecer vista\');\n    reset.type = \'button\';\n    tools.append(year, reset);\n    const frame = element(\'div\', \'od-map-frame\');\n    const canvas = element(\'div\', \'od-map-canvas\');\n    canvas.setAttribute(\'aria-label\', name === \'origen\' ? \'Mapa de orígenes\' : \'Mapa de destinos\');\n    const empty = element(\'div\', \'od-map-empty\');\n    empty.setAttribute(\'role\', \'status\');\n    empty.hidden = true;\n    frame.append(canvas, empty);\n    const legend = element(\'div\', \'od-map-legend\');\n    const minLabel = element(\'span\', \'od-legend-min\', \'0\');\n    const gradient = element(\'span\', \'od-legend-gradient\');\n    const maxLabel = element(\'span\', \'od-legend-max\');\n    const legendUnit = element(\'span\', \'od-legend-unit\', \'Viajes\');\n    legend.append(legendUnit, minLabel, gradient, maxLabel);\n    const selectionKey = element(\'p\', \'od-selection-key\', \'O: origen · D: destino (1–4 selecciones por tipo) · Contorno rojo: selección · Gris: localidades\');\n    const chartTitle = element(\'h4\', \'od-section-title\');\n    const chart = element(\'div\', \'od-bar-chart\');\n    const table = element(\'div\', \'od-compact-table\');\n    card.append(header, tools, frame, legend, selectionKey, chartTitle, chart, table);\n    grid.append(card);\n    const map = L.map(canvas, {preferCanvas: true, zoomControl: true, scrollWheelZoom: false, attributionControl: true});\n    map.setView([4.65, -74.1], 10);\n    const baseLayer = L.tileLayer(\'https://tile.openstreetmap.org/{z}/{x}/{y}.png\', {\n      attribution: \'&copy; <a href="https://www.openstreetmap.org/copyright" target="_blank" rel="noopener">OpenStreetMap</a>\',\n      maxZoom: 19,\n      referrerPolicy: \'strict-origin-when-cross-origin\'\n    });\n    baseLayer.on(\'tileerror\', () => {\n      if (!state.destroyed && basemapSelect.value === \'osm\' && ++tileErrors >= 3) setBasemap(\'none\', true);\n    });\n    const shapeLayer = L.layerGroup().addTo(map);\n    const localityLayer = L.layerGroup().addTo(map);\n    const selectionLayer = L.layerGroup().addTo(map);\n    const letterLayer = L.layerGroup().addTo(map);\n    const side = {name, card, title, subtitle, count, year, empty, legend, maxLabel, chartTitle, chart, table, map, baseLayer, shapeLayer, localityLayer, selectionLayer, letterLayer, markers: [], letterCounts: {O: 0, D: 0}};\n    sides.push(side);\n    if (basemapSelect.value === \'osm\') baseLayer.addTo(map);\n    reset.addEventListener(\'click\', () => resetExtent(side));\n    map.on(\'zoomend moveend\', () => renderLetters(side));\n  });\n\n  function selectedContexts() {\n    const grouped = new Map();\n    const involvedLocalities = new Set();\n    const fullLocalities = new Set();\n    [[\'origen\', \'O\'], [\'destino\', \'D\']].forEach(([side, role]) => {\n      const selected = new Set((state.filters[side] || []).map(String));\n      if (!selected.size) return;\n      const covered = new Set();\n      const add = (feature, level) => {\n        const key = level + \':\' + keyOf(feature);\n        const previous = grouped.get(key);\n        if (previous) previous.role = previous.role === role ? role : \'O/D\';\n        else grouped.set(key, {feature, role, level});\n        zoneIDs(feature).forEach(id => covered.add(id));\n      };\n      zones.forEach(feature => {\n        if (zoneIDs(feature).some(id => selected.has(id)) && feature.properties?.loc_code) {\n          involvedLocalities.add(String(feature.properties.loc_code));\n        }\n      });\n      // Se representan primero los conjuntos completos de mayor nivel.\n      municipalities.forEach(feature => {\n        const ids = zoneIDs(feature);\n        if (keyOf(feature) !== \'11001\' && ids.length && ids.every(id => selected.has(id))) add(feature, \'municipio\');\n      });\n      localities.forEach(feature => {\n        const ids = zoneIDs(feature);\n        if (ids.length && ids.every(id => selected.has(id))) {\n          add(feature, \'localidad\');\n          fullLocalities.add(keyOf(feature));\n        }\n      });\n      utams.forEach(feature => {\n        const value = String(feature.properties?.utam || \'\').trim();\n        const ids = zoneIDs(feature);\n        if (!value || /^(?:N\\/?A|NAN|NONE|NULL|SIN\\b)$/i.test(value)) return;\n        if (ids.length && ids.every(id => selected.has(id) && !covered.has(id))) add(feature, \'utam\');\n      });\n      zones.forEach(feature => {\n        if (zoneIDs(feature).some(id => selected.has(id) && !covered.has(id))) add(feature, \'zat\');\n      });\n    });\n    const selections = [...grouped.values()];\n    // Contar antes del recorte por vista y de evitar colisiones; O y D son independientes.\n    const letterCounts = {O: 0, D: 0};\n    selections.forEach(item => item.role.split(\'/\').forEach(role => letterCounts[role] += 1));\n    return {selections, letterCounts, involvedLocalities, fullLocalities};\n  }\n\n  function centerOf(feature) {\n    const p = feature.properties || {};\n    if (Array.isArray(p.center) && p.center.length === 2 && p.center.every(Number.isFinite)) return L.latLng(p.center[0], p.center[1]);\n    const bounds = boundsOf([feature]);\n    return bounds ? bounds.getCenter() : null;\n  }\n\n  function renderLetters(side) {\n    if (state.destroyed) return;\n    side.letterLayer.clearLayers();\n    const occupied = [];\n    let count = 0;\n    const viewport = side.map.getBounds();\n    // Se evitan letras superpuestas; al acercar el mapa aparecen más etiquetas.\n    for (const item of side.markers) {\n      const visibleRole = item.role.split(\'/\').filter(role => side.letterCounts[role] > 0 && side.letterCounts[role] < 5).join(\'/\');\n      if (!visibleRole) continue;\n      const center = centerOf(item.feature);\n      if (!center || !viewport.contains(center)) continue;\n      const point = side.map.latLngToContainerPoint(center);\n      if (occupied.some(q => Math.abs(q.x - point.x) < 38 && Math.abs(q.y - point.y) < 30)) continue;\n      if (count >= 60) break;\n      occupied.push(point);\n      count += 1;\n      const marker = L.marker(center, {\n        interactive: false,\n        keyboard: false,\n        icon: L.divIcon({className: \'od-selection-letter\', html: \'<span>\' + visibleRole + \'</span>\', iconSize: [32, 24], iconAnchor: [16, 12]})\n      });\n      marker.addTo(side.letterLayer);\n    }\n  }\n\n  function renderContext(side, context) {\n    side.localityLayer.clearLayers();\n    side.selectionLayer.clearLayers();\n    L.geoJSON(fc(localities), {\n      interactive: false,\n      style: feature => ({\n        color: context.involvedLocalities.has(keyOf(feature)) ? \'#727985\' : \'#b5bbc3\',\n        weight: context.fullLocalities.has(keyOf(feature)) ? 2.6 : context.involvedLocalities.has(keyOf(feature)) ? 2.2 : 1.15,\n        opacity: 0.95,\n        fillColor: \'#858c95\',\n        fillOpacity: context.fullLocalities.has(keyOf(feature)) ? 0.2 : context.involvedLocalities.has(keyOf(feature)) ? 0.14 : 0\n      })\n    }).addTo(side.localityLayer);\n    L.geoJSON(fc(context.selections.filter(item => item.level !== \'localidad\').map(item => item.feature)), {\n      interactive: false,\n      style: {color: \'#d52b36\', weight: 2.7, opacity: 1, fill: false}\n    }).addTo(side.selectionLayer);\n    side.markers = context.selections;\n    side.letterCounts = context.letterCounts;\n    renderLetters(side);\n  }\n\n  function drawBars(side, rows, maximum, featureMap) {\n    side.chart.replaceChildren();\n    if (!rows.length) return;\n    const fragment = document.createDocumentFragment();\n    rows.forEach(row => {\n      const feature = featureMap.get(String(row.key));\n      const line = element(feature && zoneIDs(feature).length ? \'button\' : \'div\', \'od-bar-row\');\n      if (line.tagName === \'BUTTON\') {\n        line.type = \'button\';\n        line.addEventListener(\'click\', () => toggle(feature, side.name));\n      }\n      const label = element(\'span\', \'od-bar-label\', row.label);\n      const track = element(\'span\', \'od-bar-track\');\n      const fill = element(\'span\', \'od-bar-fill\');\n      const width = maximum > 0 ? Math.max(0, Number(row.value)) / maximum * 100 : 0;\n      fill.style.width = Math.min(100, width) + \'%\';\n      fill.style.backgroundColor = blue(Number(row.value), maximum);\n      track.append(fill);\n      const amount = element(\'span\', \'od-bar-number\', number(row.value));\n      line.title = row.label + \': \' + number(row.value) + \' viajes (\' + percent(row.pct) + \')\';\n      line.append(label, track, amount);\n      fragment.append(line);\n    });\n    side.chart.append(fragment);\n  }\n\n  function drawTable(side, rows, territory) {\n    side.table.replaceChildren();\n    if (!rows.length) return;\n    const table = element(\'table\', \'od-values\');\n    const head = element(\'thead\');\n    const header = element(\'tr\');\n    [territory, \'Viajes\', \'%\'].forEach(text => { const th = element(\'th\', null, text); th.scope = \'col\'; header.append(th); });\n    head.append(header);\n    const body = element(\'tbody\');\n    rows.forEach(row => {\n      const tr = element(\'tr\');\n      tr.append(element(\'td\', null, row.label), element(\'td\', null, number(row.value)), element(\'td\', null, percent(row.pct)));\n      body.append(tr);\n    });\n    const foot = element(\'tfoot\');\n    const tr = element(\'tr\');\n    const total = rows.reduce((sum, row) => sum + Number(row.value || 0), 0);\n    tr.append(element(\'td\', null, \'Total\'), element(\'td\', null, number(total)), element(\'td\', null, total > 0 ? percent(100) : percent(0)));\n    foot.append(tr);\n    table.append(head, body, foot);\n    side.table.append(table);\n  }\n\n  function render(result, filters) {\n    if (state.destroyed) return;\n    if (!result || ![\'regional\', \'localidades\', \'zat\'].includes(result.view)) throw new Error(\'Vista OD no reconocida.\');\n    const changedView = state.view !== result.view;\n    const firstData = !state.result?.available && result.available;\n    state.view = result.view;\n    state.result = result;\n    state.filters = filters || {origen: null, destino: null};\n    const context = selectedContexts();\n    const isIncomplete = result.view === \'zat\' && [\'origen\', \'destino\'].some(key => Array.isArray(state.filters[key]) && state.filters[key].length === 0);\n    const available = result.available !== false && !isIncomplete;\n    const source = findFeatures(result.view);\n    const featureMap = new Map(source.map(feature => [keyOf(feature), feature]));\n    sides.forEach(side => {\n      const text = labels(result.view, side.name);\n      const raw = side.name === \'origen\' ? result.origins : result.destinations;\n      const rows = available ? (raw || []).map(row => ({...row, key: String(row.key), label: String(row.label == null ? row.key : row.label), value: Number(row.value) || 0, pct: Number(row.pct) || 0})).sort((a, b) => b.value - a.value || a.label.localeCompare(b.label, \'es\', {numeric: true})) : [];\n      const maximum = Math.max(0,...rows.map(row=>row.value));\n      const rowMap = new Map(rows.map(row => [row.key, row]));\n      const total = rows.reduce((sum, row) => sum + row.value, 0);\n      side.title.textContent = text.heading;\n      side.subtitle.textContent = text.sub;\n      side.year.textContent = String(result.year || \'\');\n      side.count.textContent = available ? number(total) + \' viajes\' : \'—\';\n      side.chartTitle.textContent = \'Viajes por \' + (result.view === \'regional\' ? \'municipio\' : result.view === \'localidades\' ? \'localidad\' : \'ZAT\');\n      side.maxLabel.textContent = number(maximum);\n      side.legend.hidden = !available || maximum <= 0;\n      side.shapeLayer.clearLayers();\n      L.geoJSON(fc(source), {\n        style: feature => {\n          const row = rowMap.get(keyOf(feature));\n          const isBogotaContext = result.view === \'regional\' && keyOf(feature) === \'11001\';\n          return {\n            color: result.view === \'zat\' ? \'#96a3b2\' : \'#8997a8\',\n            weight: result.view === \'zat\' ? 0.65 : 1,\n            opacity: 0.9,\n            fillColor: isBogotaContext ? \'#d4d8dd\' : blue(row ? row.value : 0, maximum),\n            fillOpacity: isBogotaContext ? 0.55 : row && row.value > 0 ? 0.83 : 0.3\n          };\n        },\n        onEachFeature: (feature, layer) => {\n          const p = feature.properties || {};\n          const row = rowMap.get(keyOf(feature));\n          let tooltip = \'<strong>\' + escape(p.label || keyOf(feature)) + \'</strong>\';\n          tooltip += \'<br>\' + (row ? escape(number(row.value)) + \' viajes · \' + escape(percent(row.pct)) : available ? \'Sin viajes en esta consulta\' : \'Sin información para mostrar\');\n          if (zoneIDs(feature).length) {\n            tooltip += \'<br><span class="od-tooltip-hint">Clic: seleccionar como \' + side.name + \'</span>\';\n            layer.on(\'click\', () => toggle(feature, side.name));\n          }\n          layer.bindTooltip(tooltip, {sticky: true, direction: \'top\', className: \'od-map-tooltip\'});\n        }\n      }).addTo(side.shapeLayer);\n      renderContext(side, context);\n      const hasTrips = total > 0;\n      side.empty.hidden = available && hasTrips;\n      side.empty.textContent = isIncomplete ? \'Selecciona las zonas de origen y destino, o la opción Todas.\' : (!available && result.message) ? result.message : result.view === \'regional\' ? \'No hay viajes regionales para los filtros seleccionados.\' : \'No hay viajes para los filtros seleccionados.\';\n      const displayedRows = hasTrips ? rows : [];\n      drawBars(side, displayedRows, maximum, featureMap);\n      drawTable(side, displayedRows, text.territory);\n      if (!displayedRows.length) side.chart.append(element(\'p\', \'od-empty-chart\', isIncomplete ? \'Selecciona las zonas para consultar sus viajes.\' : available ? \'Sin viajes para esta selección.\' : result.message || \'Información no disponible.\'));\n      if (changedView || firstData) resetExtent(side);\n    });\n    resize();\n  }\n\n  function resize() {\n    if (state.destroyed) return;\n    requestAnimationFrame(() => {\n      if (state.destroyed) return;\n      sides.forEach(side => { side.map.invalidateSize({pan: false}); renderLetters(side); });\n    });\n  }\n\n  const observer = typeof ResizeObserver === \'function\' ? new ResizeObserver(resize) : null;\n  if (observer) observer.observe(container);\n  window.addEventListener(\'resize\', resize);\n  function destroy() {\n    state.destroyed = true;\n    if (observer) observer.disconnect();\n    window.removeEventListener(\'resize\', resize);\n    sides.forEach(side => side.map.remove());\n    container.replaceChildren();\n  }\n  return {render, destroy, resize};\n}\n\n(async()=>{\'use strict\';\nconst root=document.getElementById(\'visor-telefonia\'), $=s=>root.querySelector(s), $$=s=>[...root.querySelectorAll(s)];\nconst initial=__INITIAL_DATA__, geo=__GEODATA__;\nconst clone=x=>JSON.parse(JSON.stringify(x));\nconst defaults={dia:\'laboral\',hora:[0,23],modo:null,motivo:null,residencia:null,origen:null,destino:null};\nlet filters=clone(initial.filters||defaults), applied=clone(filters), payload=initial, normalPayload=initial;\nlet view=\'viajes\', busy=false, dirty=false, exportRows=[], odResult=null, mapYear=2025, maps=null;\nlet normalSelection={origen:null,destino:null}, zatSelection={origen:[],destino:[]};\nlet optionData=initial.options||{}, lastAppliedView=\'viajes\', lastAppliedYear=2025;\nconst years=[2019,2023,2025], colors={2019:\'#a5c900\',2023:\'#6846ff\',2025:\'#f08b00\'};\nconst dayLabels={laboral:\'Laboral\',sabado:\'Sábado\',domingo:\'Domingo\'};\nconst titles={viajes:\'Total de viajes por año\',distancia:\'Viajes por distancia y año\',hora:\'Viajes por hora de inicio de viaje\',modo:\'Viajes por grupo de modo y año\',motivo:\'Viajes por motivo y año\',nse:\'Viajes por estrato y año\',regional:\'Origen–destino regional\',localidades:\'Origen–destino por localidades\',zat:\'Origen–destino por ZAT\'};\nconst isOD=()=>[\'regional\',\'localidades\',\'zat\'].includes(view);\nconst fmt=n=>n===null||n===undefined?\'N/D\':Number(n).toLocaleString(\'es-CO\',{maximumFractionDigits:0});\nconst fmtPct=n=>n===null||n===undefined?\'—\':Number(n).toLocaleString(\'es-CO\',{minimumFractionDigits:1,maximumFractionDigits:1})+\' %\';\nconst esc=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst norm=s=>String(s).normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toLowerCase();\nconst leaves=nodes=>nodes.flatMap(n=>n.children?leaves(n.children):[n]);\nfunction options(key){return key===\'dia\'?Object.keys(dayLabels).map(value=>({value,label:dayLabels[value]})):(optionData[key]||[])}\nfunction flat(key){return leaves(options(key))}\nfunction selected(key,value){return filters[key]===null||filters[key].includes(value)}\nfunction close(){$$(\'.field.open\').forEach(f=>{f.classList.remove(\'open\');f.querySelector(\'.trigger\').setAttribute(\'aria-expanded\',\'false\')})}\nfunction updateTriggers(){\n  $$(\'.field[data-key]\').forEach(f=>{const k=f.dataset.key;let t;\n    if(k===\'dia\')t=dayLabels[filters.dia];else if(filters[k]===null)t=[\'modo\',\'motivo\'].includes(k)?\'Todos\':\'Todas\';\n    else if(!filters[k].length)t=\'Sin selección\';else if(filters[k].length===1)t=flat(k).find(o=>o.value===filters[k][0])?.label||filters[k][0];else t=filters[k].length+\' seleccionadas\';\n    f.querySelector(\'.trigger-text\').textContent=t;f.querySelector(\'.trigger\').title=t;\n  });\n  $$(\'[data-hour]\').forEach(e=>e.value=filters.hora[+e.dataset.hour]);\n  $(\'.rail-active\').style.left=(filters.hora[0]/23*100)+\'%\';$(\'.rail-active\').style.right=((23-filters.hora[1])/23*100)+\'%\';\n  $$(\'[name="od-year"]\').forEach(e=>e.checked=+e.value===mapYear);\n}\nfunction mark(){\n  dirty=JSON.stringify(filters)!==JSON.stringify(applied)||(isOD()&&(view!==lastAppliedView||mapYear!==lastAppliedYear));\n  $(\'.status\').classList.toggle(\'dirty\',dirty);$(\'.status\').textContent=dirty?\'Cambios pendientes de aplicar.\':\'Filtros aplicados.\';updateTriggers();\n}\nfunction setValues(key,values,on){let set=new Set(filters[key]===null?flat(key).map(x=>x.value):filters[key]);values.forEach(v=>on?set.add(v):set.delete(v));filters[key]=[...set].sort();mark()}\nconst expanded=new Set();\nfunction renderMenu(field){\n const key=field.dataset.key,q=norm(field.querySelector(\'.search\').value),box=field.querySelector(\'.choices\'),scroll=box.scrollTop;box.replaceChildren();\n function row(label,values,depth=0,radio=false){\n   const div=document.createElement(\'div\');div.className=\'choice\';div.style.paddingLeft=(10+depth*16)+\'px\';\n   const lab=document.createElement(\'label\');lab.className=\'choice-label\';const inp=document.createElement(\'input\');inp.type=radio?\'radio\':\'checkbox\';inp.setAttribute(\'aria-label\',label);\n   if(radio){inp.name=\'visor-dia\';inp.checked=filters.dia===values[0]}else{const n=values.filter(v=>selected(key,v)).length;inp.checked=!!values.length&&n===values.length;inp.indeterminate=n>0&&n<values.length}\n   const span=document.createElement(\'span\');span.textContent=label;lab.append(inp,span);div.append(lab);\n   inp.addEventListener(\'change\',()=>{if(radio){filters.dia=values[0];mark();close()}else{setValues(key,values,inp.checked);renderMenu(field)}});return div;\n }\n if(key===\'dia\'){options(key).filter(o=>norm(o.label).includes(q)).forEach(o=>box.append(row(o.label,[o.value],0,true)));return}\n const all=flat(key),head=document.createElement(\'div\');head.className=\'choice\';\n const lab=document.createElement(\'label\');lab.className=\'choice-label\';const toggle=document.createElement(\'input\');toggle.type=\'checkbox\';toggle.setAttribute(\'aria-label\',\'Seleccionar todo\');toggle.checked=filters[key]===null;toggle.indeterminate=filters[key]!==null&&filters[key].length>0;\n lab.append(toggle,document.createTextNode(\'Seleccionar todo\'));head.append(lab);box.append(head);\n toggle.addEventListener(\'change\',()=>{filters[key]=toggle.checked?null:[];mark();renderMenu(field)});\n function visit(n,depth,path,ancestorMatches=false){\n   const own=norm(n.label+\' \'+(n.value||\'\')).includes(q), allChildren=leaves([n]);\n   if(q&&!ancestorMatches&&!own&&!allChildren.some(c=>norm(c.label+\' \'+c.value).includes(q)))return;\n   const div=row(n.label,allChildren.map(x=>x.value),depth);box.append(div);\n   if(n.children){\n     const arrow=document.createElement(\'button\');arrow.type=\'button\';arrow.className=\'expand\';const open=q||expanded.has(path);arrow.textContent=open?\'⌄\':\'›\';arrow.setAttribute(\'aria-label\',\'Desplegar \'+n.label);arrow.setAttribute(\'aria-expanded\',String(!!open));div.prepend(arrow);\n     arrow.addEventListener(\'click\',()=>{expanded.has(path)?expanded.delete(path):expanded.add(path);renderMenu(field)});\n     if(open)n.children.forEach((child,i)=>visit(child,depth+1,path+\'/\'+i,ancestorMatches||own));\n   }\n }\n options(key).forEach((n,i)=>visit(n,0,key+\'/\'+i));\n if(box.children.length===1){const d=document.createElement(\'p\');d.className=\'note\';d.textContent=\'Sin coincidencias\';box.append(d)}box.scrollTop=scroll;\n}\n$$(\'.field[data-key]\').forEach(f=>{f.querySelector(\'.trigger\').addEventListener(\'click\',()=>{const open=f.classList.contains(\'open\');close();if(!open){f.classList.add(\'open\');f.querySelector(\'.trigger\').setAttribute(\'aria-expanded\',\'true\');renderMenu(f);f.querySelector(\'.search\').focus()}});f.querySelector(\'.search\').addEventListener(\'input\',()=>renderMenu(f))});\ndocument.addEventListener(\'click\',e=>{if(!root.contains(e.target)||!e.target.closest(\'.field\'))close()});document.addEventListener(\'keydown\',e=>{if(e.key===\'Escape\')close()});\n$$(\'[data-hour]\').forEach(e=>e.addEventListener(\'input\',()=>{const k=+e.dataset.hour;filters.hora[k]=Math.max(0,Math.min(23,Math.round(Number(e.value)||0)));if(filters.hora[0]>filters.hora[1])filters.hora[1-k]=filters.hora[k];mark()}));\nfunction setBusy(value){busy=value;root.classList.toggle(\'busy\',value);$$(\'.btn,.trigger,[data-hour],.nav,[name="od-year"]\').forEach(e=>e.disabled=value)}\nasync function invoke(name,args){\n let r;if(window.__VISOR_BRIDGE__)r=await window.__VISOR_BRIDGE__(name,args);\n else if(window.google?.colab?.kernel){const v=await google.colab.kernel.invokeFunction(name,args,{});r=v.data[\'application/json\']}\n else throw new Error(\'Mantén conectado el entorno de Google Colab.\');\n if(typeof r===\'string\')r=JSON.parse(r);if(r.error)throw new Error(r.error);return r;\n}\nlet leafletPromise;\nasync function ensureMaps(){\n if(maps)return;\n if(!leafletPromise)leafletPromise=(async()=>{\n  if(window.L)return;\n  const css=document.createElement(\'link\');css.rel=\'stylesheet\';css.href=\'https://cdn.jsdelivr.net/npm/leaflet@1.9.4/dist/leaflet.css\';document.head.append(css);\n  for(const src of [\'https://cdn.jsdelivr.net/npm/leaflet@1.9.4/dist/leaflet.js\',\'https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\']){\n   try{await new Promise((resolve,reject)=>{const s=document.createElement(\'script\');s.src=src;const t=setTimeout(()=>{s.remove();reject(new Error(\'Tiempo de carga agotado\'))},12000);s.onload=()=>{clearTimeout(t);resolve()};s.onerror=()=>{clearTimeout(t);s.remove();reject(new Error(\'No se pudo cargar el mapa\'))};document.head.append(s)});if(window.L)return}catch(e){}\n  }\n  throw new Error(\'No se pudo cargar la biblioteca de mapas. Comprueba la conexión y pulsa Aplicar filtros.\');\n })().catch(e=>{leafletPromise=null;throw e});\n await leafletPromise;\n maps=createODMaps({geo,container:$(\'#od-maps\'),formatNumber:fmt,formatPercent:fmtPct,onSelect:(side,ids)=>{if(busy)return;filters[side]=ids;mark();apply()}});\n}\nfunction errors(messages){$(\'.errors\').replaceChildren();[...new Set((messages||[]).filter(Boolean))].forEach(m=>{const d=document.createElement(\'div\');d.textContent=m;$(\'.errors\').append(d)})}\nfunction renderPairs(){\n const area=$(\'#od-pairs\');area.hidden=view!==\'zat\';if(area.hidden)return;\n const pairs=odResult?.pairs||[];$(\'#pair-total\').textContent=odResult?.available?\'Total de la consulta: \'+fmt(odResult.total)+\' viajes\':\'\';\n const rows=pairs.slice(0,300);$(\'#pair-table\').innerHTML=rows.length?\'<table><thead><tr><th>Origen</th><th>Destino</th><th>Viajes</th><th>%</th></tr></thead><tbody>\'+rows.map(p=>\'<tr><td>\'+esc(p.origin_label||p.origin)+\'</td><td>\'+esc(p.destination_label||p.destination)+\'</td><td>\'+fmt(p.value)+\'</td><td>\'+fmtPct(p.pct)+\'</td></tr>\').join(\'\')+\'</tbody></table>\':\'<p class="note">Sin relaciones para mostrar.</p>\';\n $(\'#pair-note\').textContent=pairs.length>300?\'Se muestran las primeras 300 relaciones. Descarga la tabla para consultar todas las relaciones.\':\'\';\n}\nasync function renderOD(){\n $(\'.normal-results\').hidden=true;$(\'.od-results\').hidden=false;$(\'#year-selector\').hidden=false;$(\'#od-title\').textContent=titles[view]+\' · \'+mapYear;\n const result=odResult||{view,year:mapYear,available:false,message:\'Selecciona los filtros y pulsa Aplicar filtros.\',origins:[],destinations:[],pairs:[],total:null};\n errors([result.available===false&&result.message&&!result.pending?result.message:\'\']);\n const diagnostics=[...new Set([...(initial.geoDiagnostics||[]),...(result.warnings||[])])];\n const missing=result.unmapped||{};\n if(missing.either>0){diagnostics.push(\'Viajes con algún extremo sin correspondencia cartográfica: \'+fmt(missing.either)+\'. Origen sin correspondencia: \'+fmt(missing.origins)+\'; destino sin correspondencia: \'+fmt(missing.destinations)+\'.\');errors([fmt(missing.either)+\' viajes tienen algún extremo sin geometría. Consulta las observaciones de los datos.\'])}\n if(missing.locality_origins>0||missing.locality_destinations>0)diagnostics.push(\'Viajes de Bogotá sin localidad asignada: \'+fmt(missing.locality_origins)+\' en origen y \'+fmt(missing.locality_destinations)+\' en destino.\');\n $(\'#od-diagnostics\').replaceChildren();diagnostics.forEach(t=>{const p=document.createElement(\'p\');p.textContent=t;$(\'#od-diagnostics\').append(p)});\n $(\'#od-diagnostics\').parentElement.hidden=diagnostics.length===0;\n try{await ensureMaps();maps.render(result,filters);maps.resize()}catch(e){errors([e.message])}\n renderPairs();\n}\nasync function apply(){\n if(busy)return;setBusy(true);close();const sent=clone(filters),v=view,y=mapYear;$(\'.status\').classList.remove(\'dirty\');$(\'.status\').textContent=\'Calculando… La primera carga puede tardar varios minutos.\';\n try{\n  if(isOD()){\n   odResult=await invoke(\'visor_telefonia.consultar_od\',[sent,v,y]);optionData=odResult.options||optionData;\n   applied=clone(sent);lastAppliedView=v;lastAppliedYear=y;await renderOD();\n  }else{\n   payload=await invoke(\'visor_telefonia.consultar\',[sent]);normalPayload=payload;optionData=payload.options||optionData;\n   applied=clone(sent);lastAppliedView=v;renderResults();\n  }\n  mark();\n }catch(e){\n  if(isOD()){odResult={view:v,year:y,available:false,message:e.message,origins:[],destinations:[],pairs:[],total:null};await renderOD()}\n  errors([e.message]);$(\'.status\').textContent=\'No se pudieron aplicar los filtros.\';$(\'.status\').classList.add(\'dirty\')\n }\n finally{setBusy(false);resizeFrame()}\n}\nasync function switchView(next){\n if(busy||view===next)return;\n if(view===\'zat\'){zatSelection={origen:clone(filters.origen),destino:clone(filters.destino)};Object.assign(filters,clone(normalSelection))}\n if(next===\'zat\'){normalSelection={origen:clone(filters.origen),destino:clone(filters.destino)};Object.assign(filters,clone(zatSelection))}\n view=next;close();$$(\'.nav\').forEach(n=>n.classList.toggle(\'active\',n.dataset.view===view));root.classList.toggle(\'od-active\',isOD());\n if(isOD()){odResult=null;await apply()}\n else{\n  $(\'.normal-results\').hidden=false;$(\'.od-results\').hidden=true;$(\'#year-selector\').hidden=true;\n  if(JSON.stringify(filters)===JSON.stringify(normalPayload.filters)){payload=normalPayload;optionData=payload.options;applied=clone(filters);renderResults();mark()}\n  else await apply();\n }\n updateTriggers();resizeFrame();\n}\n$$(\'.nav[data-view]\').forEach(e=>e.addEventListener(\'click\',()=>switchView(e.dataset.view)));\n$$(\'[name="od-year"]\').forEach(e=>e.addEventListener(\'change\',()=>{mapYear=Number(e.value);mark();apply()}));\nfunction download(rows,name){const quote=x=>\'"\'+String(x??\'N/D\').replace(/"/g,\'""\')+\'"\';const csv=\'\\ufeff\'+rows.map(r=>r.map(quote).join(\';\')).join(\'\\r\\n\');const a=document.createElement(\'a\');const u=URL.createObjectURL(new Blob([csv],{type:\'text/csv;charset=utf-8\'}));a.href=u;a.download=name+\'.csv\';a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)}\n$(\'#v-aplicar\').addEventListener(\'click\',apply);\n$(\'#v-limpiar\').addEventListener(\'click\',()=>{filters=clone(defaults);if(view===\'zat\'){filters.origen=[];filters.destino=[];zatSelection={origen:[],destino:[]}}mapYear=2025;mark();apply()});\n$(\'#v-csv\').addEventListener(\'click\',()=>download(exportRows,\'viajes_\'+view+\'_\'+applied.dia));\n$(\'#od-csv\').addEventListener(\'click\',()=>{if(!odResult?.available)return;const rows=[[\'Sentido\',\'Territorio\',\'Viajes\',\'Porcentaje\',\'Año\']];for(const [key,label]of [[\'origins\',\'Origen\'],[\'destinations\',\'Destino\']])for(const p of odResult?.[key]||[])rows.push([label,p.label,Math.round(p.value),Number(p.pct||0).toFixed(1),odResult.year]);download(rows,\'viajes_\'+odResult.view+\'_\'+odResult.year)});\n$(\'#pairs-csv\').addEventListener(\'click\',async()=>{\n if(!odResult?.available)return;\n if(odResult.pairs_truncated){setBusy(true);try{await invoke(\'visor_telefonia.descargar_pares\',[applied,odResult.year])}catch(e){errors([e.message])}finally{setBusy(false)}}\n else download([[\'ZAT origen\',\'ZAT destino\',\'Viajes\',\'Porcentaje\',\'Año\'],...(odResult.pairs||[]).map(p=>[p.origin_label||p.origin,p.destination_label||p.destination,Math.round(p.value),Number(p.pct||0).toFixed(1),odResult.year])],\'relaciones_zat_\'+odResult.year);\n});\nfunction resizeFrame(){try{google.colab.output.setIframeHeight(Math.max(900,document.body.scrollHeight),true)}catch(e){}}\nfunction chartData(){\n  const series=years.map(y=>payload.series.find(s=>s.year===y)||{year:y,available:false});\n  if(view===\'viajes\')return {categories:years.map(String),labels:years.map(String),series};\n  let categories;\n  const seen=new Set(series.flatMap(s=>Object.keys(s[view]||{})));\n  if(view===\'hora\')categories=Array.from({length:24},(_,i)=>String(i));\n  else if(view===\'distancia\')categories=[...Object.keys(payload.distanceLabels),...[...seen].filter(c=>!payload.distanceLabels[c]).sort()];\n  else if(view===\'modo\')categories=[\'no_motorizado\',\'transporte_privado\',\'transporte_publico\',...[...seen].filter(c=>![\'no_motorizado\',\'transporte_privado\',\'transporte_publico\'].includes(c)).sort()];\n  else if(view===\'motivo\')categories=[...Object.keys(payload.purposeLabels),...[...seen].filter(c=>!payload.purposeLabels[c]).sort()];\n  else categories=[...Array.from({length:6},(_,i)=>\'E\'+(i+1)),...[...seen].filter(c=>!/^E[1-6]$/.test(c)).sort()];\n  const labels=categories.map(c=>c===\'__SIN_DATO__\'?\'Sin dato\':view===\'distancia\'?(payload.distanceLabels[c]||c):view===\'modo\'?(payload.modeLabels[c]||c):view===\'motivo\'?(payload.purposeLabels[c]||c):view===\'nse\'&&/^E[1-6]$/.test(c)?\'Estrato \'+c[1]:c);\n  return {categories,labels,series};\n}\nfunction value(s,c){return s.available?(view===\'viajes\'?s.total:(s[view]?.[c]||0)):null}\nfunction wrap(text,max=21){let lines=[\'\'];String(text).split(\' \').forEach(w=>{let i=lines.length-1;if((lines[i]+\' \'+w).trim().length>max&&lines[i])lines.push(w);else lines[i]=(lines[i]+\' \'+w).trim()});return lines}\nfunction tickStep(max){if(max<=0)return 1;let raw=max/5,p=10**Math.floor(Math.log10(raw)),v=raw/p;return Math.max(1,Math.ceil((v<=1?1:v<=2?2:v<=2.5?2.5:v<=5?5:10)*p))}\nfunction drawChart(d){\n  const W=980,H=390,L=82,R=24,T=24,B=view===\'hora\'?55:(view===\'motivo\'?95:77),pw=W-L-R,ph=H-T-B;\n  let values=view===\'viajes\'?d.series.map(s=>value(s)):d.series.flatMap(s=>d.categories.map(c=>value(s,c)));\n  const max=Math.max(0,...values.filter(v=>v!==null)),step=tickStep(max*1.08),ymax=Math.max(step,Math.ceil(max*1.08/step)*step);\n  const unit=\'Viajes\';\n  const y=v=>T+ph-(v/ymax)*ph;\n  let svg=\'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 \'+W+\' \'+H+\'" role="img" aria-label="\'+esc(titles[view])+\'">\';\n  for(let v=0;v<=ymax+step*.01;v+=step){const yy=y(v);svg+=\'<line x1="\'+L+\'" x2="\'+(W-R)+\'" y1="\'+yy+\'" y2="\'+yy+\'" stroke="#e5e8eb" stroke-dasharray="2 4"/><text x="\'+(L-10)+\'" y="\'+(yy+4)+\'" font-size="12" text-anchor="end" fill="#687386">\'+esc(fmt(v))+\'</text>\'}\n  svg+=\'<text transform="translate(18 \'+(T+ph/2)+\') rotate(-90)" text-anchor="middle" font-size="13" fill="#525e70">\'+unit+\'</text>\';\n  if(view===\'hora\'){\n    const x=i=>L+i/23*pw;\n    d.categories.forEach((c,i)=>{svg+=\'<text x="\'+x(i)+\'" y="\'+(H-B+22)+\'" text-anchor="middle" font-size="12" fill="#687386">\'+c+\'</text>\'});\n    d.series.forEach(s=>{if(!s.available)return;let path=d.categories.map((c,i)=>(i?\'L\':\'M\')+x(i).toFixed(2)+\',\'+y(value(s,c)).toFixed(2)).join(\' \');\n      svg+=\'<path d="\'+path+\'" fill="none" stroke="\'+colors[s.year]+\'" stroke-width="2.8" stroke-linejoin="round"/>\';\n      d.categories.forEach((c,i)=>{svg+=\'<circle cx="\'+x(i)+\'" cy="\'+y(value(s,c))+\'" r="3.8" fill="\'+colors[s.year]+\'" fill-opacity="0"><title>\'+s.year+\' · \'+c+\':00 · \'+fmt(value(s,c))+\' viajes</title></circle>\'});\n    });\n    svg+=\'<text x="\'+(L+pw/2)+\'" y="\'+(H-8)+\'" text-anchor="middle" font-size="13" fill="#525e70">Hora de inicio del viaje</text>\';\n  }else{\n    const count=d.categories.length,groupWidth=pw/count,bw=view===\'viajes\'?Math.min(108,groupWidth*.45):Math.min(36,groupWidth*.23);\n    d.categories.forEach((c,i)=>{\n      const cx=L+(i+.5)*groupWidth;\n      const bars=view===\'viajes\'?[d.series[i]]:d.series;\n      bars.forEach((s,j)=>{const v=value(s,c),xx=view===\'viajes\'?cx-bw/2:cx+(j-1.5)*bw;\n        if(v===null){svg+=\'<text x="\'+(xx+bw/2)+\'" y="\'+(y(0)-8)+\'" font-size="10" text-anchor="middle" fill="#9b6d47">N/D</text>\';return}\n        svg+=\'<rect x="\'+xx+\'" y="\'+y(v)+\'" width="\'+(bw-2)+\'" height="\'+Math.max(0,y(0)-y(v))+\'" rx="2" fill="\'+colors[s.year]+\'"><title>\'+s.year+\' · \'+esc(d.labels[i])+\' · \'+fmt(v)+\' viajes</title></rect>\';\n        if(view===\'viajes\')svg+=\'<text x="\'+cx+\'" y="\'+(y(v)-9)+\'" text-anchor="middle" font-size="14" fill="#354153">\'+fmt(v)+\'</text>\';\n      });\n      wrap(d.labels[i],view===\'distancia\'?15:(count>6?15:23)).forEach((line,j)=>{svg+=\'<text x="\'+cx+\'" y="\'+(H-B+21+j*15)+\'" text-anchor="middle" font-size="12" fill="#525e70">\'+esc(line)+\'</text>\'});\n    });\n    const axis={viajes:\'Año\',distancia:\'Distancia del viaje\',modo:\'Grupo de modo\',motivo:\'Motivo del viaje\',nse:\'Estrato socioeconómico\'}[view];\n    svg+=\'<text x="\'+(L+pw/2)+\'" y="\'+(H-7)+\'" text-anchor="middle" font-size="13" fill="#525e70">\'+axis+\'</text>\';\n  }\n  svg+=\'</svg>\';\n  $(\'.chart\').innerHTML=d.series.some(s=>s.available)?svg:\'<div class="empty">No hay datos cargados. Reintente la descarga o cargue los ZIP locales.</div>\';\n}\n\nfunction drawTable(d){\n $(\'.table-wrap\').classList.toggle(\'totals-only\',view===\'viajes\');let head,rows;\n if(view===\'viajes\'){head=[\'Año\',\'Total de viajes\'];rows=d.series.map(s=>[String(s.year),value(s)]);exportRows=[head,...rows.map(r=>[r[0],r[1]===null?\'N/D\':Math.round(r[1])])];\n $(\'.table-wrap\').innerHTML=\'<table><thead><tr>\'+head.map(x=>\'<th>\'+esc(x)+\'</th>\').join(\'\')+\'</tr></thead><tbody>\'+rows.map(r=>\'<tr><td>\'+r[0]+\'</td><td>\'+fmt(r[1])+\'</td></tr>\').join(\'\')+\'</tbody></table>\';return}\n head=[{distancia:\'Distancia\',hora:\'Hora\',modo:\'Modo\',motivo:\'Motivo\',nse:\'Estrato\'}[view],...years.flatMap(y=>[y+\' viajes\',y+\' %\'])];\n rows=d.categories.map((c,i)=>[view===\'hora\'?c.padStart(2,\'0\')+\':00\':d.labels[i],...d.series.flatMap(s=>{const v=value(s,c);return[v,s.available&&s.total>0?v/s.total*100:null]})]);\n rows.push([\'Total\',...d.series.flatMap(s=>[s.available?s.total:null,s.available&&s.total>0?100:null])]);\n exportRows=[head,...rows.map(r=>r.map((v,i)=>i===0?v:v===null?\'N/D\':i%2?Math.round(v):v.toFixed(1)))];\n $(\'.table-wrap\').innerHTML=\'<table><thead><tr><th rowspan="2">\'+esc(head[0])+\'</th>\'+years.map(y=>\'<th colspan="2">\'+y+\'</th>\').join(\'\')+\'</tr><tr>\'+years.map(()=>\'<th>Viajes</th><th>%</th>\').join(\'\')+\'</tr></thead><tbody>\'+rows.map(r=>\'<tr\'+(r[0]===\'Total\'?\' class="total"\':\'\')+\'>\'+r.map((v,i)=>\'<td>\'+(i===0?esc(v):i%2?fmt(v):fmtPct(v))+\'</td>\').join(\'\')+\'</tr>\').join(\'\')+\'</tbody></table>\';\n}\nfunction renderResults(){\n $(\'.normal-results\').hidden=false;$(\'.od-results\').hidden=true;$(\'#year-selector\').hidden=true;$(\'.normal-results h2\').textContent=titles[view];\n $(\'.view-note\').textContent=view===\'modo\'?\'Grupos comunes a los tres años. El filtro permite elegir modos específicos cuando están disponibles.\':\'\';\n errors(payload.series.filter(s=>!s.available).map(s=>s.message||s.year+\': no disponible\'));\n $(\'.legend\').innerHTML=years.map(y=>\'<span><i class="swatch" style="background:\'+colors[y]+\'"></i>\'+y+\'</span>\').join(\'\');const d=chartData();drawChart(d);drawTable(d);\n}\nupdateTriggers();renderResults();mark();resizeFrame();\nwindow.__visorDebug={getFilters:()=>clone(filters),getPayload:()=>isOD()?odResult:payload,getView:()=>view,getYear:()=>mapYear,switchView,apply};\n})();\n\n</script>\n</div>\n'


In [ ]:
#@title 3. Descargar los cinco insumos de Drive y abrir el visor
ACTUALIZAR_INSUMOS = False #@param {type:"boolean"}
import gdown

# Los mismos archivos se reutilizan durante esta sesión de Colab. Active la
# casilla para descargar versiones nuevas conservando los mismos ID de Drive.
INSUMOS_LISTOS = False
carpeta = Path('/content/visor_telefonia') if EN_COLAB else Path.cwd()/'visor_telefonia'
carpeta.mkdir(parents=True,exist_ok=True)
CARPETA_DRIVE = 'https://drive.google.com/drive/folders/1qr8RvinqSzde3zjt7ujsOqrhw-t3XRJv'
FUENTES_DRIVE = {
    'zonificacion': ('01-zonificacion.zip', '1Z3tFeuTDvwDB3p2qh168xm46fWm1nVsj'),
    'localidades': ('02-localidades.zip', '1VUGUPegqYBjiZE5Z0xx18CfPBaDXYhpd'),
    'municipios': ('03-municipios.zip', '1o37USK3OR514U0E0SYbpXoZDbfxiEOxI'),
    'final': ('04-zonificacion_final.zip', '12DNDkIqbel423GGy7mcXCu3JZWmUYk4J'),
    'matrices': ('05-matrices.zip', '1fkyzWgX_QBfue85Z8ILBIGC6p-shgbXY'),
}
_INSUMOS_VALIDADOS = globals().get('_INSUMOS_VALIDADOS', set())

def _firma_local(ruta):
    estado = ruta.stat()
    return (str(ruta.resolve()), estado.st_size, estado.st_mtime_ns)

def comprobar_insumo(ruta, clave, completo=False):
    """Comprueba estructura, las nueve cabeceras y, la primera vez, todo el CRC ZIP."""
    with zipfile.ZipFile(ruta) as archivo:
        entradas = archivo.infolist()
        archivos = {}
        for entrada in entradas:
            nombre = entrada.filename.replace('\\', '/')
            partes = PurePosixPath(nombre).parts
            if PurePosixPath(nombre).is_absolute() or '..' in partes or (partes and ':' in partes[0]) or stat.S_ISLNK(entrada.external_attr >> 16):
                raise ValueError('El ZIP contiene una ruta no permitida: ' + nombre)
            if entrada.is_dir() or '__macosx' in [p.lower() for p in partes]:
                continue
            if nombre.lower() in archivos:
                raise ValueError('El ZIP contiene entradas repetidas: ' + nombre)
            archivos[nombre.lower()] = entrada
        if not archivos:
            raise ValueError('El ZIP no contiene archivos utilizables.')
        if completo:
            defectuoso = archivo.testzip()
            if defectuoso:
                raise ValueError('El ZIP contiene un archivo corrupto: ' + defectuoso)
        if clave != 'matrices':
            shapes = [nombre[:-4] for nombre in archivos if nombre.endswith('.shp')]
            if len(shapes) != 1:
                raise ValueError(f'Se esperaba una sola capa SHP; se encontraron {len(shapes)}.')
            faltantes = [ext for ext in ('.shp', '.shx', '.dbf', '.prj') if shapes[0] + ext not in archivos]
            if faltantes:
                raise ValueError('Faltan archivos de la capa: ' + ', '.join(faltantes))
        else:
            seleccionadas = set()
            for anio in ANIOS:
                for dia in PERIODOS:
                    # Usar exactamente el mismo selector que usará el motor.
                    entrada = VisorTelefonia.miembro(None, ruta, anio, dia)
                    if entrada.filename in seleccionadas:
                        raise ValueError('Una matriz fue asignada a más de un año o tipo de día.')
                    seleccionadas.add(entrada.filename)
                    with archivo.open(entrada.filename) as bruto:
                        contenido = gzip.GzipFile(fileobj=bruto) if entrada.filename.endswith('.gz') else bruto
                        try:
                            cabecera = contenido.readline(65537)
                            if len(cabecera) > 65536:
                                raise ValueError(f'Cabecera inválida en {anio}, {dia}.')
                            campos = next(csv.reader([cabecera.decode('utf-8-sig').rstrip('\r\n')], delimiter='|'), [])
                            if campos != CAMPOS:
                                raise ValueError(f'Columnas incorrectas en {anio}, {dia}.')
                            if not contenido.read(1):
                                raise ValueError(f'La matriz {anio}, {dia} no contiene registros.')
                        finally:
                            if contenido is not bruto:
                                contenido.close()

def descargar_insumo(clave, nombre, identificador):
    # Aislar por ID impide confundir ZIP antiguos con estos cinco insumos.
    destino = carpeta / 'insumos_drive' / identificador / nombre
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.is_file() and not ACTUALIZAR_INSUMOS:
        try:
            firma = _firma_local(destino)
            comprobar_insumo(destino, clave, completo=firma not in _INSUMOS_VALIDADOS)
            _INSUMOS_VALIDADOS.add(firma)
            print('Reutilizando ' + nombre)
            return destino
        except (OSError, ValueError, EOFError, zipfile.BadZipFile, RuntimeError) as exc:
            print(f'La copia local de {nombre} no es válida; se descargará de nuevo. {exc}')
    temporal = destino.with_suffix('.zip.part')
    # No reutilizar una descarga de contenido desconocido al pedir actualización.
    temporal.unlink(missing_ok=True)
    print('Descargando y verificando ' + nombre + '…')
    try:
        descargado = gdown.download(id=identificador, output=str(temporal), quiet=False,
                                   resume=False, retries=3, timeout=60)
        if not descargado or not temporal.is_file():
            raise RuntimeError('Google Drive no entregó el archivo solicitado.')
        comprobar_insumo(temporal, clave, completo=True)
        temporal.replace(destino)
        _INSUMOS_VALIDADOS.add(_firma_local(destino))
        return destino
    except Exception as exc:
        temporal.unlink(missing_ok=True)
        raise RuntimeError(
            f'No se pudo cargar {nombre}. Compruebe que mantiene acceso para cualquier '
            'persona con el enlace. Si Drive indica cuota o demasiadas descargas, '
            'vuelva a ejecutar más tarde. El visor no se abrirá con insumos incompletos. '
            f'Detalle: {str(exc)[:500]}'
        ) from exc

INSUMOS = {clave: descargar_insumo(clave, *fuente) for clave, fuente in FUENTES_DRIVE.items()}
rutas = {clave: str(INSUMOS[clave]) for clave in ('zonificacion', 'localidades', 'municipios', 'final')}
print('Cinco ZIP verificados; nueve matrices identificadas, con cabeceras válidas. Preparando mapas…')

geo=GeografiaVisor(rutas)
if 'visor' in globals():
    try: visor.db.close()
    except Exception: pass
visor=VisorConMapas(carpeta,geo,rutas_locales={anio: str(INSUMOS['matrices']) for anio in ANIOS})
registrados = visor.registrar_zip(INSUMOS['matrices'])
if set(registrados) != set(ANIOS):
    raise ValueError('El ZIP de matrices debe contener 2019, 2023 y 2025.')
INSUMOS_LISTOS = True

def comprobar_carga_lista():
    if not INSUMOS_LISTOS:
        raise RuntimeError('La carga de Drive no se completó. Ejecute de nuevo la celda 3 antes de consultar.')

def consulta_colab(filtros):
    try:
        comprobar_carga_lista()
        return JSON(visor.consultar(filtros))
    except Exception as exc: return JSON({'error':str(exc)[:400]})

def consulta_od_colab(filtros,vista,anio):
    try:
        comprobar_carga_lista()
        return JSON(visor.consultar_od(filtros,vista,anio))
    except Exception as exc: return JSON({'error':str(exc)[:400]})

def descargar_pares_colab(filtros,anio):
    try:
        comprobar_carga_lista()
        ruta=visor.exportar_pares(filtros,anio,carpeta/f'relaciones_zat_{anio}.csv')
        files.download(str(ruta))
        return JSON({'ok':True})
    except Exception as exc: return JSON({'error':str(exc)[:400]})

if EN_COLAB:
    output.register_callback('visor_telefonia.consultar',consulta_colab)
    output.register_callback('visor_telefonia.consultar_od',consulta_od_colab)
    output.register_callback('visor_telefonia.descargar_pares',descargar_pares_colab)

def abrir_visor():
    import html
    comprobar_carga_lista()
    estado=display(HTML('<p style="font:14px Arial;color:#697386">Preparando el visor…</p>'),display_id=True)
    def progreso(texto): estado.update(HTML('<p style="font:14px Arial;color:#697386">'+html.escape(texto)+'</p>'))
    visor.avisar=progreso
    try:
        datos=visor.consultar()
        if datos.get('errors'):
            raise RuntimeError('No se abrió el visor porque falló la preparación inicial: ' + '; '.join(datos['errors']))
        datos['geoDiagnostics']=geo.diagnostico
    finally:
        visor.avisar=lambda texto:None
    inicial=json.dumps(datos,ensure_ascii=False,allow_nan=False).replace('<','\\u003c')
    geometria=json.dumps(geo.geodata,ensure_ascii=False,allow_nan=False,separators=(',',':')).replace('<','\\u003c')
    clear_output(wait=True)
    display(HTML(PLANTILLA.replace('__INITIAL_DATA__',inicial).replace('__GEODATA__',geometria)))

abrir_visor()
